# Step 1 — Reproducible Environment

In [1]:
# ============================================================
# STEP 1 — REPRODUCIBLE EXPERIMENTAL ENVIRONMENT
# Solar + Load Multi-Task Forecasting Study
# ============================================================

import os
import sys
import platform
import random
import time
from datetime import datetime

import numpy as np
import pandas as pd

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

random.seed(SEED)
np.random.seed(SEED)

print("=" * 70)
print("EXPERIMENTAL ENVIRONMENT")
print("=" * 70)

print(f"Timestamp        : {datetime.now()}")
print(f"Python version   : {sys.version.split()[0]}")
print(f"Platform         : {platform.platform()}")
print(f"Processor        : {platform.processor()}")
print(f"CPU cores        : {os.cpu_count()}")

print(f"\nNumPy version    : {np.__version__}")
print(f"Pandas version   : {pd.__version__}")

# ------------------------------------------------------------
# TensorFlow
# ------------------------------------------------------------
try:
    import tensorflow as tf

    tf.random.set_seed(SEED)

    try:
        tf.config.experimental.enable_op_determinism()
        deterministic_status = "Enabled"
    except Exception as e:
        deterministic_status = f"Requested via environment; API status: {e}"

    print(f"TensorFlow       : {tf.__version__}")
    print(f"TF deterministic : {deterministic_status}")

    gpus = tf.config.list_physical_devices("GPU")

    print(f"TF GPU available : {len(gpus) > 0}")

    if gpus:
        for i, gpu in enumerate(gpus):
            print(f"  GPU {i}: {gpu}")

except Exception as e:
    print("TensorFlow error :", e)

# ------------------------------------------------------------
# PyTorch / CUDA information
# ------------------------------------------------------------
try:
    import torch

    torch.manual_seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    print(f"\nPyTorch version  : {torch.__version__}")
    print(f"CUDA available   : {torch.cuda.is_available()}")

    if torch.cuda.is_available():
        print(f"CUDA version     : {torch.version.cuda}")
        print(f"GPU name         : {torch.cuda.get_device_name(0)}")

except Exception as e:
    print("PyTorch error    :", e)

# ------------------------------------------------------------
# RAM
# ------------------------------------------------------------
try:
    import psutil

    ram_gb = psutil.virtual_memory().total / (1024**3)
    print(f"\nSystem RAM       : {ram_gb:.2f} GB")

except Exception as e:
    print("RAM info error   :", e)

print("\n" + "=" * 70)
print(f"GLOBAL RANDOM SEED = {SEED}")
print("=" * 70)

EXPERIMENTAL ENVIRONMENT
Timestamp        : 2026-09-17 12:35:08.490157
Python version   : 3.13.15
Platform         : Linux-6.6.122+-x86_64-with-glibc2.39
Processor        : x86_64
CPU cores        : 12

NumPy version    : 2.1.3
Pandas version   : 2.2.3
TensorFlow       : 2.20.0
TF deterministic : Enabled
TF GPU available : True
  GPU 0: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

PyTorch version  : 2.11.0+cu128
CUDA available   : True
CUDA version     : 12.8
GPU name         : NVIDIA A100-SXM4-40GB

System RAM       : 83.47 GB

GLOBAL RANDOM SEED = 42


Step 2 — Raw OPSD Dataset Audit

In [2]:
# ============================================================
# STEP 2.1 — RAW OPSD DATASET ACQUISITION AND INITIAL AUDIT
# ============================================================

import os
import pandas as pd
import numpy as np

DATA_URL = (
    "https://data.open-power-system-data.org/"
    "time_series/2020-10-06/"
    "time_series_60min_singleindex.csv"
)

DATA_FILE = "time_series_60min_singleindex.csv"

print("=" * 70)
print("STEP 2.1 — OPSD RAW DATASET ACQUISITION")
print("=" * 70)

# ------------------------------------------------------------
# Download once
# ------------------------------------------------------------
if not os.path.exists(DATA_FILE):
    print("\nDownloading OPSD dataset...")
    df_raw = pd.read_csv(DATA_URL)
    df_raw.to_csv(DATA_FILE, index=False)
    print("Dataset downloaded and saved locally.")
else:
    print("\nLocal dataset found. Loading...")
    df_raw = pd.read_csv(DATA_FILE)

print("\n" + "-" * 70)
print("RAW DATASET SHAPE")
print("-" * 70)

print(f"Rows    : {df_raw.shape[0]:,}")
print(f"Columns : {df_raw.shape[1]:,}")

print("\n" + "-" * 70)
print("FIRST 10 COLUMN NAMES")
print("-" * 70)

for col in df_raw.columns[:10]:
    print(col)

print("\n" + "-" * 70)
print("TIMESTAMP-RELATED COLUMNS")
print("-" * 70)

for col in df_raw.columns:
    if "time" in col.lower() or "utc" in col.lower():
        print(col)

print("\n" + "-" * 70)
print("GERMANY-RELATED COLUMNS")
print("-" * 70)

de_cols = [col for col in df_raw.columns if col.startswith("DE")]

print(f"Number of DE columns: {len(de_cols)}")

for col in de_cols:
    print(col)

print("\n" + "-" * 70)
print("SOLAR-RELATED GERMANY COLUMNS")
print("-" * 70)

de_solar_cols = [
    col for col in de_cols
    if "solar" in col.lower()
]

for col in de_solar_cols:
    print(col)

print("\n" + "-" * 70)
print("LOAD-RELATED GERMANY COLUMNS")
print("-" * 70)

de_load_cols = [
    col for col in de_cols
    if "load" in col.lower()
]

for col in de_load_cols:
    print(col)

print("\n" + "=" * 70)
print("STEP 2.1 COMPLETE")
print("=" * 70)

STEP 2.1 — OPSD RAW DATASET ACQUISITION

Dataset downloaded and saved locally.

----------------------------------------------------------------------
RAW DATASET SHAPE
----------------------------------------------------------------------
Rows    : 50,401
Columns : 300

----------------------------------------------------------------------
FIRST 10 COLUMN NAMES
----------------------------------------------------------------------
utc_timestamp
cet_cest_timestamp
AT_load_actual_entsoe_transparency
AT_load_forecast_entsoe_transparency
AT_price_day_ahead
AT_solar_generation_actual
AT_wind_onshore_generation_actual
BE_load_actual_entsoe_transparency
BE_load_forecast_entsoe_transparency
BE_solar_generation_actual

----------------------------------------------------------------------
TIMESTAMP-RELATED COLUMNS
----------------------------------------------------------------------
utc_timestamp
cet_cest_timestamp

----------------------------------------------------------------------
GERMAN

Step 2.2 — Target Coverage & Missingness Audit

In [3]:
# ============================================================
# STEP 2.2 — CANDIDATE TARGET COVERAGE AND MISSINGNESS AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 78)
print("STEP 2.2 — CANDIDATE TARGET COVERAGE AND MISSINGNESS AUDIT")
print("=" * 78)

# ------------------------------------------------------------
# Parse timestamps
# ------------------------------------------------------------
df_audit = df_raw.copy()

df_audit["utc_timestamp"] = pd.to_datetime(
    df_audit["utc_timestamp"],
    errors="coerce",
    utc=True
)

print("\nTimestamp parsing:")
print(f"Invalid UTC timestamps : {df_audit['utc_timestamp'].isna().sum():,}")
print(f"First UTC timestamp    : {df_audit['utc_timestamp'].min()}")
print(f"Last UTC timestamp     : {df_audit['utc_timestamp'].max()}")

# ------------------------------------------------------------
# Candidate target columns
# ------------------------------------------------------------
candidate_cols = [
    "DE_load_actual_entsoe_transparency",
    "DE_solar_generation_actual",
    "DE_LU_load_actual_entsoe_transparency",
    "DE_LU_solar_generation_actual"
]

print("\n" + "-" * 78)
print("CANDIDATE TARGET AUDIT")
print("-" * 78)

audit_rows = []

for col in candidate_cols:

    if col not in df_audit.columns:
        print(f"\nWARNING: {col} not found.")
        continue

    s = pd.to_numeric(df_audit[col], errors="coerce")

    valid_mask = s.notna()

    valid_count = int(valid_mask.sum())
    missing_count = int(s.isna().sum())
    missing_pct = 100 * missing_count / len(s)

    first_valid = (
        df_audit.loc[valid_mask, "utc_timestamp"].min()
        if valid_count > 0 else pd.NaT
    )

    last_valid = (
        df_audit.loc[valid_mask, "utc_timestamp"].max()
        if valid_count > 0 else pd.NaT
    )

    audit_rows.append({
        "column": col,
        "valid_n": valid_count,
        "missing_n": missing_count,
        "missing_pct": missing_pct,
        "first_valid": first_valid,
        "last_valid": last_valid,
        "min": s.min(),
        "max": s.max(),
        "mean": s.mean(),
        "std": s.std()
    })

audit_table = pd.DataFrame(audit_rows)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print(
    audit_table.to_string(
        index=False,
        formatters={
            "missing_pct": lambda x: f"{x:.2f}",
            "min": lambda x: f"{x:.3f}",
            "max": lambda x: f"{x:.3f}",
            "mean": lambda x: f"{x:.3f}",
            "std": lambda x: f"{x:.3f}",
        }
    )
)

# ------------------------------------------------------------
# Joint availability for each candidate pair
# ------------------------------------------------------------
print("\n" + "-" * 78)
print("JOINT LOAD + SOLAR AVAILABILITY")
print("-" * 78)

candidate_pairs = [
    (
        "Legacy DE",
        "DE_load_actual_entsoe_transparency",
        "DE_solar_generation_actual"
    ),
    (
        "DE_LU",
        "DE_LU_load_actual_entsoe_transparency",
        "DE_LU_solar_generation_actual"
    )
]

pair_rows = []

for label, load_col, solar_col in candidate_pairs:

    if load_col not in df_audit.columns or solar_col not in df_audit.columns:
        continue

    load_s = pd.to_numeric(df_audit[load_col], errors="coerce")
    solar_s = pd.to_numeric(df_audit[solar_col], errors="coerce")

    joint_mask = load_s.notna() & solar_s.notna()

    joint_n = int(joint_mask.sum())

    first_joint = (
        df_audit.loc[joint_mask, "utc_timestamp"].min()
        if joint_n > 0 else pd.NaT
    )

    last_joint = (
        df_audit.loc[joint_mask, "utc_timestamp"].max()
        if joint_n > 0 else pd.NaT
    )

    pair_rows.append({
        "pair": label,
        "joint_valid_n": joint_n,
        "joint_missing_n": len(df_audit) - joint_n,
        "joint_coverage_pct": 100 * joint_n / len(df_audit),
        "first_joint": first_joint,
        "last_joint": last_joint
    })

pair_table = pd.DataFrame(pair_rows)

print(
    pair_table.to_string(
        index=False,
        formatters={
            "joint_coverage_pct": lambda x: f"{x:.2f}"
        }
    )
)

# ------------------------------------------------------------
# Duplicate timestamps and temporal spacing
# ------------------------------------------------------------
print("\n" + "-" * 78)
print("TEMPORAL INTEGRITY")
print("-" * 78)

valid_times = df_audit["utc_timestamp"].dropna().sort_values()

duplicate_timestamps = valid_times.duplicated().sum()

time_diffs = valid_times.diff().dropna()
expected_delta = pd.Timedelta(hours=1)

non_hourly_gaps = (time_diffs != expected_delta).sum()

print(f"Duplicate UTC timestamps       : {duplicate_timestamps:,}")
print(f"Non-1-hour timestamp intervals : {non_hourly_gaps:,}")

if non_hourly_gaps > 0:
    gap_counts = time_diffs.value_counts().head(10)

    print("\nMost common timestamp intervals:")
    print(gap_counts)

print("\n" + "=" * 78)
print("STEP 2.2 COMPLETE")
print("=" * 78)

STEP 2.2 — CANDIDATE TARGET COVERAGE AND MISSINGNESS AUDIT

Timestamp parsing:
Invalid UTC timestamps : 0
First UTC timestamp    : 2014-12-31 23:00:00+00:00
Last UTC timestamp     : 2020-09-30 23:00:00+00:00

------------------------------------------------------------------------------
CANDIDATE TARGET AUDIT
------------------------------------------------------------------------------
                               column  valid_n  missing_n missing_pct               first_valid                last_valid       min       max      mean       std
   DE_load_actual_entsoe_transparency    50400          1        0.00 2015-01-01 00:00:00+00:00 2020-09-30 23:00:00+00:00 31307.000 77549.000 55492.469 10015.431
           DE_solar_generation_actual    50297        104        0.21 2015-01-01 07:00:00+00:00 2020-09-30 23:00:00+00:00     0.000 32947.000  4566.043  6940.268
DE_LU_load_actual_entsoe_transparency    17524      32877       65.23 2018-10-01 00:00:00+00:00 2020-09-30 23:00:00+00:00 32

Step 2.3 — Missing-Value Structure Audit

In [4]:
# ============================================================
# STEP 2.3 — MISSING-VALUE STRUCTURE AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 78)
print("STEP 2.3 — MISSING-VALUE STRUCTURE AUDIT")
print("=" * 78)

LOAD_COL = "DE_load_actual_entsoe_transparency"
SOLAR_COL = "DE_solar_generation_actual"

df_targets = df_audit[
    ["utc_timestamp", LOAD_COL, SOLAR_COL]
].copy()

# Restrict audit to meaningful target period
# First load observation through final dataset timestamp
target_start = pd.Timestamp("2015-01-01 00:00:00", tz="UTC")

df_targets = (
    df_targets[df_targets["utc_timestamp"] >= target_start]
    .sort_values("utc_timestamp")
    .reset_index(drop=True)
)

print(f"\nAudit period : {df_targets['utc_timestamp'].min()} "
      f"to {df_targets['utc_timestamp'].max()}")
print(f"Rows         : {len(df_targets):,}")


# ------------------------------------------------------------
# Function for identifying consecutive missing runs
# ------------------------------------------------------------
def missing_run_audit(df, column):

    missing = df[column].isna()

    groups = (missing != missing.shift()).cumsum()

    runs = []

    for _, group in df[missing].groupby(groups[missing]):

        if len(group) == 0:
            continue

        runs.append({
            "start": group["utc_timestamp"].iloc[0],
            "end": group["utc_timestamp"].iloc[-1],
            "length_hours": len(group)
        })

    return pd.DataFrame(runs)


# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------
print("\n" + "-" * 78)
print("LOAD MISSINGNESS")
print("-" * 78)

load_runs = missing_run_audit(df_targets, LOAD_COL)

print(f"Total missing load values : "
      f"{df_targets[LOAD_COL].isna().sum():,}")

if len(load_runs) > 0:

    print(f"Number of missing runs    : {len(load_runs):,}")
    print(f"Longest missing run       : "
          f"{load_runs['length_hours'].max()} hours")

    print("\nMissing run length distribution:")
    print(
        load_runs["length_hours"]
        .value_counts()
        .sort_index()
        .to_string()
    )

    print("\nAll load missing runs:")
    print(load_runs.to_string(index=False))

else:
    print("No missing load values.")


# ------------------------------------------------------------
# SOLAR
# ------------------------------------------------------------
print("\n" + "-" * 78)
print("SOLAR MISSINGNESS")
print("-" * 78)

solar_runs = missing_run_audit(df_targets, SOLAR_COL)

print(f"Total missing solar values : "
      f"{df_targets[SOLAR_COL].isna().sum():,}")

if len(solar_runs) > 0:

    print(f"Number of missing runs     : {len(solar_runs):,}")
    print(f"Longest missing run        : "
          f"{solar_runs['length_hours'].max()} hours")

    print("\nMissing run length distribution:")
    print(
        solar_runs["length_hours"]
        .value_counts()
        .sort_index()
        .to_string()
    )

    print("\nSolar missing runs:")
    print(solar_runs.to_string(index=False))

else:
    print("No missing solar values.")


# ------------------------------------------------------------
# Missingness by year
# ------------------------------------------------------------
print("\n" + "-" * 78)
print("MISSING VALUES BY YEAR")
print("-" * 78)

temp = df_targets.copy()
temp["year"] = temp["utc_timestamp"].dt.year

yearly_missing = temp.groupby("year").agg(
    rows=("utc_timestamp", "size"),
    load_missing=(LOAD_COL, lambda x: x.isna().sum()),
    solar_missing=(SOLAR_COL, lambda x: x.isna().sum())
)

yearly_missing["load_missing_pct"] = (
    100 * yearly_missing["load_missing"] /
    yearly_missing["rows"]
)

yearly_missing["solar_missing_pct"] = (
    100 * yearly_missing["solar_missing"] /
    yearly_missing["rows"]
)

print(yearly_missing.to_string())


# ------------------------------------------------------------
# Missingness by month
# ------------------------------------------------------------
print("\n" + "-" * 78)
print("MONTHS CONTAINING MISSING TARGET VALUES")
print("-" * 78)

temp["year_month"] = (
    temp["utc_timestamp"]
    .dt.to_period("M")
    .astype(str)
)

monthly_missing = temp.groupby("year_month").agg(
    rows=("utc_timestamp", "size"),
    load_missing=(LOAD_COL, lambda x: x.isna().sum()),
    solar_missing=(SOLAR_COL, lambda x: x.isna().sum())
)

monthly_missing = monthly_missing[
    (monthly_missing["load_missing"] > 0) |
    (monthly_missing["solar_missing"] > 0)
]

print(monthly_missing.to_string())


print("\n" + "=" * 78)
print("STEP 2.3 COMPLETE")
print("=" * 78)

STEP 2.3 — MISSING-VALUE STRUCTURE AUDIT

Audit period : 2015-01-01 00:00:00+00:00 to 2020-09-30 23:00:00+00:00
Rows         : 50,400

------------------------------------------------------------------------------
LOAD MISSINGNESS
------------------------------------------------------------------------------
Total missing load values : 0
No missing load values.

------------------------------------------------------------------------------
SOLAR MISSINGNESS
------------------------------------------------------------------------------
Total missing solar values : 103
Number of missing runs     : 5
Longest missing run        : 24 hours

Missing run length distribution:
length_hours
7     1
24    4

Solar missing runs:
                    start                       end  length_hours
2015-01-01 00:00:00+00:00 2015-01-01 06:00:00+00:00             7
2015-02-28 00:00:00+00:00 2015-02-28 23:00:00+00:00            24
2016-05-31 21:00:00+00:00 2016-06-01 20:00:00+00:00            24
2016-10-2

/tmp/ipykernel_3882/1329589647.py:163: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  .dt.to_period("M")


Step 2.4 — Chronological Split Design

In [5]:
# ============================================================
# STEP 2.4 — CHRONOLOGICAL TRAIN / VALIDATION / TEST AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 78)
print("STEP 2.4 — CHRONOLOGICAL SPLIT AUDIT")
print("=" * 78)

LOAD_COL = "DE_load_actual_entsoe_transparency"
SOLAR_COL = "DE_solar_generation_actual"

df_split = df_targets.copy()

# ------------------------------------------------------------
# Frozen candidate split boundaries
# ------------------------------------------------------------
TRAIN_START = pd.Timestamp("2015-01-01 00:00:00", tz="UTC")
TRAIN_END   = pd.Timestamp("2018-12-31 23:00:00", tz="UTC")

VAL_START   = pd.Timestamp("2019-01-01 00:00:00", tz="UTC")
VAL_END     = pd.Timestamp("2019-12-31 23:00:00", tz="UTC")

TEST_START  = pd.Timestamp("2020-01-01 00:00:00", tz="UTC")
TEST_END    = pd.Timestamp("2020-09-30 23:00:00", tz="UTC")

train_raw = df_split[
    (df_split["utc_timestamp"] >= TRAIN_START) &
    (df_split["utc_timestamp"] <= TRAIN_END)
].copy()

val_raw = df_split[
    (df_split["utc_timestamp"] >= VAL_START) &
    (df_split["utc_timestamp"] <= VAL_END)
].copy()

test_raw = df_split[
    (df_split["utc_timestamp"] >= TEST_START) &
    (df_split["utc_timestamp"] <= TEST_END)
].copy()

# ------------------------------------------------------------
# Audit function
# ------------------------------------------------------------
def audit_partition(name, df):

    return {
        "partition": name,
        "start": df["utc_timestamp"].min(),
        "end": df["utc_timestamp"].max(),
        "rows": len(df),

        "load_valid": df[LOAD_COL].notna().sum(),
        "load_missing": df[LOAD_COL].isna().sum(),

        "solar_valid": df[SOLAR_COL].notna().sum(),
        "solar_missing": df[SOLAR_COL].isna().sum(),

        "joint_valid": (
            df[LOAD_COL].notna() &
            df[SOLAR_COL].notna()
        ).sum(),

        "load_mean": df[LOAD_COL].mean(),
        "load_std": df[LOAD_COL].std(),

        "solar_mean": df[SOLAR_COL].mean(),
        "solar_std": df[SOLAR_COL].std()
    }


split_audit = pd.DataFrame([
    audit_partition("Train", train_raw),
    audit_partition("Validation", val_raw),
    audit_partition("Test", test_raw)
])

print("\n" + "-" * 78)
print("PARTITION SUMMARY")
print("-" * 78)

print(
    split_audit.to_string(
        index=False,
        formatters={
            "load_mean": lambda x: f"{x:.3f}",
            "load_std": lambda x: f"{x:.3f}",
            "solar_mean": lambda x: f"{x:.3f}",
            "solar_std": lambda x: f"{x:.3f}"
        }
    )
)

# ------------------------------------------------------------
# Percentages
# ------------------------------------------------------------
total_rows = len(train_raw) + len(val_raw) + len(test_raw)

print("\n" + "-" * 78)
print("PARTITION PROPORTIONS")
print("-" * 78)

for name, df in [
    ("Train", train_raw),
    ("Validation", val_raw),
    ("Test", test_raw)
]:
    pct = 100 * len(df) / total_rows
    print(f"{name:<12}: {len(df):>6,} rows ({pct:6.2f}%)")

# ------------------------------------------------------------
# Boundary integrity
# ------------------------------------------------------------
print("\n" + "-" * 78)
print("BOUNDARY INTEGRITY")
print("-" * 78)

print("Train → Validation gap:",
      VAL_START - TRAIN_END)

print("Validation → Test gap:",
      TEST_START - VAL_END)

print("Train/Val overlap:",
      bool(set(train_raw["utc_timestamp"]) &
           set(val_raw["utc_timestamp"])))

print("Val/Test overlap:",
      bool(set(val_raw["utc_timestamp"]) &
           set(test_raw["utc_timestamp"])))

print("Train/Test overlap:",
      bool(set(train_raw["utc_timestamp"]) &
           set(test_raw["utc_timestamp"])))

# ------------------------------------------------------------
# Preliminary distribution differences
# ------------------------------------------------------------
print("\n" + "-" * 78)
print("PRELIMINARY TEMPORAL DISTRIBUTION COMPARISON")
print("-" * 78)

for target, label in [
    (LOAD_COL, "LOAD"),
    (SOLAR_COL, "SOLAR")
]:

    train_mean = train_raw[target].mean()
    val_mean   = val_raw[target].mean()
    test_mean  = test_raw[target].mean()

    train_std = train_raw[target].std()
    val_std   = val_raw[target].std()
    test_std  = test_raw[target].std()

    print(f"\n{label}")
    print(f"Train mean/std : {train_mean:.3f} / {train_std:.3f}")
    print(f"Val   mean/std : {val_mean:.3f} / {val_std:.3f}")
    print(f"Test  mean/std : {test_mean:.3f} / {test_std:.3f}")

    print(
        f"Test mean change vs Train : "
        f"{100 * (test_mean - train_mean) / train_mean:.2f}%"
    )

    print(
        f"Test std change vs Train  : "
        f"{100 * (test_std - train_std) / train_std:.2f}%"
    )

print("\n" + "=" * 78)
print("STEP 2.4 COMPLETE")
print("=" * 78)

STEP 2.4 — CHRONOLOGICAL SPLIT AUDIT

------------------------------------------------------------------------------
PARTITION SUMMARY
------------------------------------------------------------------------------
 partition                     start                       end  rows  load_valid  load_missing  solar_valid  solar_missing  joint_valid load_mean  load_std solar_mean solar_std
     Train 2015-01-01 00:00:00+00:00 2018-12-31 23:00:00+00:00 35064       35064             0        34961            103        34961 55826.577 10033.521   4184.239  6415.453
Validation 2019-01-01 00:00:00+00:00 2019-12-31 23:00:00+00:00  8760        8760             0         8760              0         8760 55990.259  9892.634   4784.788  7288.198
      Test 2020-01-01 00:00:00+00:00 2020-09-30 23:00:00+00:00  6576        6576             0         6576              0         6576 53047.851  9734.444   6304.488  8659.945

-----------------------------------------------------------------------------

Step 3 — Leakage-Safe Feature Engineering

Step 3.1 — Feature Engineering

In [6]:
# ============================================================
# STEP 3.1 — LEAKAGE-SAFE TEMPORAL FEATURE ENGINEERING
# ============================================================

import numpy as np
import pandas as pd

print("=" * 78)
print("STEP 3.1 — TEMPORAL FEATURE ENGINEERING")
print("=" * 78)

LOAD_COL = "DE_load_actual_entsoe_transparency"
SOLAR_COL = "DE_solar_generation_actual"

df_feat = df_targets.copy()
df_feat = df_feat.sort_values("utc_timestamp").reset_index(drop=True)

# ------------------------------------------------------------
# Basic calendar variables from timestamp only
# These contain no future target information.
# ------------------------------------------------------------
df_feat["hour"] = df_feat["utc_timestamp"].dt.hour
df_feat["dayofweek"] = df_feat["utc_timestamp"].dt.dayofweek
df_feat["dayofyear"] = df_feat["utc_timestamp"].dt.dayofyear

# ------------------------------------------------------------
# Cyclical encodings
# ------------------------------------------------------------
df_feat["hour_sin"] = np.sin(
    2 * np.pi * df_feat["hour"] / 24.0
)
df_feat["hour_cos"] = np.cos(
    2 * np.pi * df_feat["hour"] / 24.0
)

df_feat["dow_sin"] = np.sin(
    2 * np.pi * df_feat["dayofweek"] / 7.0
)
df_feat["dow_cos"] = np.cos(
    2 * np.pi * df_feat["dayofweek"] / 7.0
)

# Use 365.25 to represent annual periodicity
df_feat["doy_sin"] = np.sin(
    2 * np.pi * (df_feat["dayofyear"] - 1) / 365.25
)
df_feat["doy_cos"] = np.cos(
    2 * np.pi * (df_feat["dayofyear"] - 1) / 365.25
)

FEATURE_COLS = [
    LOAD_COL,
    SOLAR_COL,
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "doy_sin",
    "doy_cos"
]

TARGET_COLS = [
    LOAD_COL,
    SOLAR_COL
]

print("\nFeature columns:")
for i, col in enumerate(FEATURE_COLS, 1):
    print(f"{i:>2}. {col}")

print("\nTarget columns:")
for i, col in enumerate(TARGET_COLS, 1):
    print(f"{i:>2}. {col}")

print("\n" + "-" * 78)
print("FEATURE AUDIT")
print("-" * 78)

feature_audit = pd.DataFrame({
    "dtype": df_feat[FEATURE_COLS].dtypes.astype(str),
    "missing": df_feat[FEATURE_COLS].isna().sum(),
    "min": df_feat[FEATURE_COLS].min(),
    "max": df_feat[FEATURE_COLS].max(),
    "mean": df_feat[FEATURE_COLS].mean()
})

print(feature_audit.to_string())

print("\n" + "-" * 78)
print("TEMPORAL FEATURE SANITY CHECK — FIRST 8 ROWS")
print("-" * 78)

display_cols = [
    "utc_timestamp",
    "hour",
    "dayofweek",
    "dayofyear",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "doy_sin",
    "doy_cos"
]

print(
    df_feat[display_cols]
    .head(8)
    .to_string(index=False)
)

print("\n" + "=" * 78)
print("STEP 3.1 COMPLETE")
print("=" * 78)

STEP 3.1 — TEMPORAL FEATURE ENGINEERING

Feature columns:
 1. DE_load_actual_entsoe_transparency
 2. DE_solar_generation_actual
 3. hour_sin
 4. hour_cos
 5. dow_sin
 6. dow_cos
 7. doy_sin
 8. doy_cos

Target columns:
 1. DE_load_actual_entsoe_transparency
 2. DE_solar_generation_actual

------------------------------------------------------------------------------
FEATURE AUDIT
------------------------------------------------------------------------------
                                      dtype  missing           min           max          mean
DE_load_actual_entsoe_transparency  float64        0  31307.000000  77549.000000  5.549247e+04
DE_solar_generation_actual          float64      103      0.000000  32947.000000  4.566043e+03
hour_sin                            float64        0     -1.000000      1.000000 -1.853896e-17
hour_cos                            float64        0     -1.000000      1.000000 -5.554640e-17
dow_sin                             float64        0     -0.974

Step 3.2 — Leakage-Safe Scaling + Sequence Construction

In [7]:
# ============================================================
# STEP 3.2 — TRAIN-ONLY SCALING + SEQUENCE CONSTRUCTION AUDIT
# ============================================================

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

print("=" * 78)
print("STEP 3.2 — LEAKAGE-SAFE SCALING AND SEQUENCE AUDIT")
print("=" * 78)

LOAD_COL = "DE_load_actual_entsoe_transparency"
SOLAR_COL = "DE_solar_generation_actual"

DYNAMIC_COLS = [LOAD_COL, SOLAR_COL]

CALENDAR_COLS = [
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "doy_sin",
    "doy_cos"
]

FEATURE_COLS = DYNAMIC_COLS + CALENDAR_COLS
TARGET_COLS = [LOAD_COL, SOLAR_COL]

# ------------------------------------------------------------
# 1. Partition labels based strictly on FORECAST TARGET TIME
# ------------------------------------------------------------
def get_partition(ts):
    if TRAIN_START <= ts <= TRAIN_END:
        return "Train"
    elif VAL_START <= ts <= VAL_END:
        return "Validation"
    elif TEST_START <= ts <= TEST_END:
        return "Test"
    else:
        return None


# ------------------------------------------------------------
# 2. Fit scalers using TRAINING OBSERVATIONS ONLY
#
# Missing solar values are ignored during fitting.
# No imputation is performed.
# ------------------------------------------------------------
train_mask = (
    (df_feat["utc_timestamp"] >= TRAIN_START) &
    (df_feat["utc_timestamp"] <= TRAIN_END)
)

train_for_scaler = df_feat.loc[
    train_mask,
    DYNAMIC_COLS
].dropna()

feature_scaler = StandardScaler()
target_scaler = StandardScaler()

feature_scaler.fit(train_for_scaler[DYNAMIC_COLS])
target_scaler.fit(train_for_scaler[TARGET_COLS])

print("\n" + "-" * 78)
print("TRAIN-ONLY SCALER PARAMETERS")
print("-" * 78)

for i, col in enumerate(DYNAMIC_COLS):
    print(
        f"{col}\n"
        f"  feature mean  = {feature_scaler.mean_[i]:.6f}\n"
        f"  feature scale = {feature_scaler.scale_[i]:.6f}\n"
        f"  target mean   = {target_scaler.mean_[i]:.6f}\n"
        f"  target scale  = {target_scaler.scale_[i]:.6f}"
    )


# ------------------------------------------------------------
# 3. Create scaled copy
#
# Calendar sin/cos variables are already bounded approximately
# within [-1, 1], so they are not standardized.
# ------------------------------------------------------------
df_scaled = df_feat.copy()

df_scaled[DYNAMIC_COLS] = feature_scaler.transform(
    df_scaled[DYNAMIC_COLS]
)


# ------------------------------------------------------------
# 4. Sequence constructor
#
# For forecast target at index i:
# X = observations [i-seq_len, ..., i-1]
# y = observation at i
#
# Therefore this is strictly one-step-ahead forecasting.
#
# Windows containing missing load/solar observations are rejected.
# ------------------------------------------------------------
def build_sequences(df, seq_len):

    X_dict = {
        "Train": [],
        "Validation": [],
        "Test": []
    }

    y_dict = {
        "Train": [],
        "Validation": [],
        "Test": []
    }

    time_dict = {
        "Train": [],
        "Validation": [],
        "Test": []
    }

    rejected_missing = {
        "Train": 0,
        "Validation": 0,
        "Test": 0
    }

    candidate_count = {
        "Train": 0,
        "Validation": 0,
        "Test": 0
    }

    feature_values = df[FEATURE_COLS].to_numpy(dtype=np.float32)
    target_values = df[TARGET_COLS].to_numpy(dtype=np.float32)
    timestamps = df["utc_timestamp"].tolist()

    for i in range(seq_len, len(df)):

        target_time = timestamps[i]
        partition = get_partition(target_time)

        if partition is None:
            continue

        candidate_count[partition] += 1

        X_window = feature_values[i-seq_len:i]
        y_target = target_values[i]

        # Reject if any input or target is missing/non-finite
        if (
            not np.isfinite(X_window).all()
            or not np.isfinite(y_target).all()
        ):
            rejected_missing[partition] += 1
            continue

        X_dict[partition].append(X_window)
        y_dict[partition].append(y_target)
        time_dict[partition].append(target_time)

    # Convert to arrays
    for partition in X_dict:

        X_dict[partition] = np.asarray(
            X_dict[partition],
            dtype=np.float32
        )

        y_dict[partition] = np.asarray(
            y_dict[partition],
            dtype=np.float32
        )

    return (
        X_dict,
        y_dict,
        time_dict,
        candidate_count,
        rejected_missing
    )


# ------------------------------------------------------------
# 5. Audit candidate context lengths
# ------------------------------------------------------------
SEQUENCE_LENGTHS = [24, 48, 168]

sequence_results = {}
audit_rows = []

for seq_len in SEQUENCE_LENGTHS:

    print("\n" + "=" * 78)
    print(f"CONTEXT WINDOW = {seq_len} HOURS")
    print("=" * 78)

    (
        X_sets,
        y_sets,
        time_sets,
        candidate_count,
        rejected_missing
    ) = build_sequences(df_scaled, seq_len)

    sequence_results[seq_len] = {
        "X": X_sets,
        "y": y_sets,
        "times": time_sets
    }

    for partition in ["Train", "Validation", "Test"]:

        accepted = len(X_sets[partition])

        first_target = (
            time_sets[partition][0]
            if accepted > 0 else None
        )

        last_target = (
            time_sets[partition][-1]
            if accepted > 0 else None
        )

        print(f"\n{partition}")
        print(f"  Candidates       : {candidate_count[partition]:,}")
        print(f"  Rejected missing : {rejected_missing[partition]:,}")
        print(f"  Accepted         : {accepted:,}")

        if accepted > 0:
            print(f"  X shape          : {X_sets[partition].shape}")
            print(f"  y shape          : {y_sets[partition].shape}")
            print(f"  First target     : {first_target}")
            print(f"  Last target      : {last_target}")

        audit_rows.append({
            "context_hours": seq_len,
            "partition": partition,
            "candidates": candidate_count[partition],
            "rejected_missing": rejected_missing[partition],
            "accepted": accepted,
            "first_target": first_target,
            "last_target": last_target
        })


# ------------------------------------------------------------
# 6. Final summary table
# ------------------------------------------------------------
sequence_audit = pd.DataFrame(audit_rows)

print("\n" + "=" * 78)
print("SEQUENCE CONSTRUCTION SUMMARY")
print("=" * 78)

print(sequence_audit.to_string(index=False))


# ------------------------------------------------------------
# 7. Critical boundary check
# ------------------------------------------------------------
print("\n" + "=" * 78)
print("BOUNDARY CHECK")
print("=" * 78)

for seq_len in SEQUENCE_LENGTHS:

    times = sequence_results[seq_len]["times"]

    print(f"\nContext = {seq_len} h")

    for partition in ["Train", "Validation", "Test"]:

        if len(times[partition]) > 0:
            print(
                f"{partition:<10} "
                f"{times[partition][0]} -> "
                f"{times[partition][-1]}"
            )

print("\n" + "=" * 78)
print("STEP 3.2 COMPLETE")
print("=" * 78)

STEP 3.2 — LEAKAGE-SAFE SCALING AND SEQUENCE AUDIT

------------------------------------------------------------------------------
TRAIN-ONLY SCALER PARAMETERS
------------------------------------------------------------------------------
DE_load_actual_entsoe_transparency
  feature mean  = 55821.818140
  feature scale = 10034.618508
  target mean   = 55821.818140
  target scale  = 10034.618508
DE_solar_generation_actual
  feature mean  = 4184.239495
  feature scale = 6415.360951
  target mean   = 4184.239495
  target scale  = 6415.360951

CONTEXT WINDOW = 24 HOURS

Train
  Candidates       : 35,040
  Rejected missing : 199
  Accepted         : 34,841
  X shape          : (34841, 24, 8)
  y shape          : (34841, 2)
  First target     : 2015-01-02 07:00:00+00:00
  Last target      : 2018-12-31 23:00:00+00:00

Validation
  Candidates       : 8,760
  Rejected missing : 0
  Accepted         : 8,760
  X shape          : (8760, 24, 8)
  y shape          : (8760, 2)
  First target     : 20

Step 3.3 — Naïve Baselines

In [8]:
# ============================================================
# STEP 3.3 — PERSISTENCE AND SEASONAL-NAIVE BASELINES
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("=" * 78)
print("STEP 3.3 — NAIVE FORECASTING BASELINES")
print("=" * 78)

LOAD_COL = "DE_load_actual_entsoe_transparency"
SOLAR_COL = "DE_solar_generation_actual"

# ------------------------------------------------------------
# Metric function
# ------------------------------------------------------------
def regression_metrics(y_true, y_pred):

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }


# ------------------------------------------------------------
# Build baseline predictions from ORIGINAL UNSCALED data
#
# Persistence:
#   prediction at t = value at t-1
#
# Seasonal persistence:
#   prediction at t = value at t-24
# ------------------------------------------------------------
df_baseline = df_feat[
    ["utc_timestamp", LOAD_COL, SOLAR_COL]
].copy()

df_baseline["load_persistence"] = (
    df_baseline[LOAD_COL].shift(1)
)

df_baseline["solar_persistence"] = (
    df_baseline[SOLAR_COL].shift(1)
)

df_baseline["load_daily_naive"] = (
    df_baseline[LOAD_COL].shift(24)
)

df_baseline["solar_daily_naive"] = (
    df_baseline[SOLAR_COL].shift(24)
)


# ------------------------------------------------------------
# Evaluate validation and test only
# ------------------------------------------------------------
evaluation_periods = {
    "Validation": (VAL_START, VAL_END),
    "Test": (TEST_START, TEST_END)
}

results = []

for period_name, (start, end) in evaluation_periods.items():

    subset = df_baseline[
        (df_baseline["utc_timestamp"] >= start) &
        (df_baseline["utc_timestamp"] <= end)
    ].copy()

    # --------------------------------------------------------
    # Persistence
    # --------------------------------------------------------
    persistence_valid = subset[
        [
            LOAD_COL,
            SOLAR_COL,
            "load_persistence",
            "solar_persistence"
        ]
    ].dropna()

    load_m = regression_metrics(
        persistence_valid[LOAD_COL],
        persistence_valid["load_persistence"]
    )

    solar_m = regression_metrics(
        persistence_valid[SOLAR_COL],
        persistence_valid["solar_persistence"]
    )

    results.append({
        "Period": period_name,
        "Model": "Persistence (t-1)",
        "Target": "Load",
        "N": len(persistence_valid),
        **load_m
    })

    results.append({
        "Period": period_name,
        "Model": "Persistence (t-1)",
        "Target": "Solar",
        "N": len(persistence_valid),
        **solar_m
    })

    # --------------------------------------------------------
    # Daily seasonal persistence
    # --------------------------------------------------------
    daily_valid = subset[
        [
            LOAD_COL,
            SOLAR_COL,
            "load_daily_naive",
            "solar_daily_naive"
        ]
    ].dropna()

    load_m = regression_metrics(
        daily_valid[LOAD_COL],
        daily_valid["load_daily_naive"]
    )

    solar_m = regression_metrics(
        daily_valid[SOLAR_COL],
        daily_valid["solar_daily_naive"]
    )

    results.append({
        "Period": period_name,
        "Model": "Daily persistence (t-24)",
        "Target": "Load",
        "N": len(daily_valid),
        **load_m
    })

    results.append({
        "Period": period_name,
        "Model": "Daily persistence (t-24)",
        "Target": "Solar",
        "N": len(daily_valid),
        **solar_m
    })


baseline_results = pd.DataFrame(results)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------
print("\n" + "-" * 78)
print("BASELINE RESULTS")
print("-" * 78)

print(
    baseline_results.to_string(
        index=False,
        formatters={
            "MAE": lambda x: f"{x:.3f}",
            "RMSE": lambda x: f"{x:.3f}",
            "R2": lambda x: f"{x:.6f}"
        }
    )
)


# ------------------------------------------------------------
# Best baseline by validation RMSE for each target
# ------------------------------------------------------------
print("\n" + "-" * 78)
print("BEST VALIDATION BASELINE BY RMSE")
print("-" * 78)

validation_results = baseline_results[
    baseline_results["Period"] == "Validation"
]

for target in ["Load", "Solar"]:

    temp = validation_results[
        validation_results["Target"] == target
    ].sort_values("RMSE")

    best = temp.iloc[0]

    print(
        f"{target:<6}: {best['Model']} | "
        f"RMSE={best['RMSE']:.3f} | "
        f"MAE={best['MAE']:.3f} | "
        f"R²={best['R2']:.6f}"
    )


# ------------------------------------------------------------
# Save baseline evidence
# ------------------------------------------------------------
baseline_results.to_csv(
    "baseline_results.csv",
    index=False
)

print("\nSaved: baseline_results.csv")

print("\n" + "=" * 78)
print("STEP 3.3 COMPLETE")
print("=" * 78)

STEP 3.3 — NAIVE FORECASTING BASELINES

------------------------------------------------------------------------------
BASELINE RESULTS
------------------------------------------------------------------------------
    Period                    Model Target    N      MAE     RMSE       R2
Validation        Persistence (t-1)   Load 8760 1898.436 2525.656 0.934811
Validation        Persistence (t-1)  Solar 8760 1293.684 2155.637 0.912510
Validation Daily persistence (t-24)   Load 8760 4557.693 6943.991 0.507230
Validation Daily persistence (t-24)  Solar 8760 1136.623 2477.540 0.884429
      Test        Persistence (t-1)   Load 6576 1848.604 2391.092 0.939656
      Test        Persistence (t-1)  Solar 6576 1623.435 2583.798 0.910967
      Test Daily persistence (t-24)   Load 6576 4049.460 6161.737 0.599272
      Test Daily persistence (t-24)  Solar 6576 1259.569 2633.191 0.907530

------------------------------------------------------------------------------
BEST VALIDATION BASELINE BY RM

Step 4 — Select Transformer Context Length

Step 4.1 — Fixed Multi-Task Transformer

In [9]:
# ============================================================
# STEP 4.1 — VALIDATION-BASED CONTEXT LENGTH SCREENING
# Fixed Multi-Task Transformer
# ============================================================

import os
import gc
import time
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("=" * 78)
print("STEP 4.1 — TRANSFORMER CONTEXT LENGTH SCREENING")
print("=" * 78)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42

def reset_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

reset_seed()

# ------------------------------------------------------------
# Fixed screening hyperparameters
# ------------------------------------------------------------
D_MODEL = 64
NUM_HEADS = 4
FF_DIM = 128
NUM_BLOCKS = 2
DROPOUT = 0.20

LEARNING_RATE = 1e-3
BATCH_SIZE = 64
MAX_EPOCHS = 50
PATIENCE = 7

CONTEXTS = [24, 48, 168]

print("\nFixed architecture:")
print(f"d_model       : {D_MODEL}")
print(f"attention heads: {NUM_HEADS}")
print(f"FF dimension  : {FF_DIM}")
print(f"blocks        : {NUM_BLOCKS}")
print(f"dropout       : {DROPOUT}")
print(f"learning rate : {LEARNING_RATE}")
print(f"batch size    : {BATCH_SIZE}")
print(f"max epochs    : {MAX_EPOCHS}")
print(f"patience      : {PATIENCE}")
print(f"seed          : {SEED}")


# ------------------------------------------------------------
# Positional encoding layer
# ------------------------------------------------------------
class SinusoidalPositionalEncoding(layers.Layer):

    def __init__(self, d_model, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model

    def call(self, inputs):

        seq_len = tf.shape(inputs)[1]

        position = tf.cast(
            tf.range(seq_len)[:, tf.newaxis],
            tf.float32
        )

        div_term = tf.exp(
            tf.cast(
                tf.range(0, self.d_model, 2),
                tf.float32
            ) *
            (-np.log(10000.0) / self.d_model)
        )

        sin_part = tf.sin(position * div_term)
        cos_part = tf.cos(position * div_term)

        pos_encoding = tf.reshape(
            tf.stack([sin_part, cos_part], axis=-1),
            (seq_len, self.d_model)
        )

        return inputs + pos_encoding[tf.newaxis, :, :]


# ------------------------------------------------------------
# Transformer encoder block
# ------------------------------------------------------------
def transformer_block(x):

    attn_output = layers.MultiHeadAttention(
        num_heads=NUM_HEADS,
        key_dim=D_MODEL // NUM_HEADS,
        dropout=DROPOUT
    )(x, x)

    x = layers.Add()([x, attn_output])
    x = layers.LayerNormalization(epsilon=1e-6)(x)

    ff = layers.Dense(
        FF_DIM,
        activation="gelu"
    )(x)

    ff = layers.Dropout(DROPOUT)(ff)

    ff = layers.Dense(D_MODEL)(ff)

    x = layers.Add()([x, ff])
    x = layers.LayerNormalization(epsilon=1e-6)(x)

    return x


# ------------------------------------------------------------
# Multi-task Transformer
# ------------------------------------------------------------
def build_mtl_transformer(seq_len, n_features):

    inputs = keras.Input(
        shape=(seq_len, n_features),
        name="input_sequence"
    )

    x = layers.Dense(
        D_MODEL,
        name="input_projection"
    )(inputs)

    x = SinusoidalPositionalEncoding(
        D_MODEL,
        name="positional_encoding"
    )(x)

    for _ in range(NUM_BLOCKS):
        x = transformer_block(x)

    x = layers.GlobalAveragePooling1D(
        name="global_average_pooling"
    )(x)

    x = layers.Dense(
        64,
        activation="gelu",
        name="shared_dense"
    )(x)

    x = layers.Dropout(DROPOUT)(x)

    # Two task-specific regression heads
    load_output = layers.Dense(
        1,
        name="load_output"
    )(x)

    solar_output = layers.Dense(
        1,
        name="solar_output"
    )(x)

    model = keras.Model(
        inputs=inputs,
        outputs=[load_output, solar_output]
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss={
            "load_output": "mse",
            "solar_output": "mse"
        }
    )

    return model


# ------------------------------------------------------------
# Evaluation helper — inverse transform to original units
# ------------------------------------------------------------
def original_scale_metrics(y_scaled, pred_scaled):

    y_true = target_scaler.inverse_transform(y_scaled)
    y_pred = target_scaler.inverse_transform(pred_scaled)

    rows = []

    for idx, target in enumerate(["Load", "Solar"]):

        mae = mean_absolute_error(
            y_true[:, idx],
            y_pred[:, idx]
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_true[:, idx],
                y_pred[:, idx]
            )
        )

        r2 = r2_score(
            y_true[:, idx],
            y_pred[:, idx]
        )

        rows.append({
            "Target": target,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })

    return rows


# ------------------------------------------------------------
# Train one fixed model per context
# ------------------------------------------------------------
screening_rows = []
history_store = {}

for seq_len in CONTEXTS:

    print("\n" + "=" * 78)
    print(f"TRAINING CONTEXT = {seq_len} HOURS")
    print("=" * 78)

    tf.keras.backend.clear_session()
    gc.collect()
    reset_seed()

    X_train = sequence_results[seq_len]["X"]["Train"]
    y_train = sequence_results[seq_len]["y"]["Train"]

    X_val = sequence_results[seq_len]["X"]["Validation"]
    y_val = sequence_results[seq_len]["y"]["Validation"]

    model = build_mtl_transformer(
        seq_len=seq_len,
        n_features=X_train.shape[-1]
    )

    if seq_len == CONTEXTS[0]:
        model.summary()

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=1
        )
    ]

    start_time = time.perf_counter()

    history = model.fit(
        X_train,
        {
            "load_output": y_train[:, 0],
            "solar_output": y_train[:, 1]
        },
        validation_data=(
            X_val,
            {
                "load_output": y_val[:, 0],
                "solar_output": y_val[:, 1]
            }
        ),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=2,
        shuffle=True
    )

    training_time = time.perf_counter() - start_time

    history_store[seq_len] = history.history

    # --------------------------------------------------------
    # Validation predictions
    # --------------------------------------------------------
    pred = model.predict(
        X_val,
        batch_size=BATCH_SIZE,
        verbose=0
    )

    pred_scaled = np.column_stack([
        pred[0].reshape(-1),
        pred[1].reshape(-1)
    ])

    metric_rows = original_scale_metrics(
        y_val,
        pred_scaled
    )

    best_epoch = int(
        np.argmin(history.history["val_loss"]) + 1
    )

    best_val_loss = float(
        np.min(history.history["val_loss"])
    )

    params = model.count_params()

    print(f"\nContext       : {seq_len} h")
    print(f"Parameters    : {params:,}")
    print(f"Best epoch    : {best_epoch}")
    print(f"Best val loss : {best_val_loss:.6f}")
    print(f"Training time : {training_time:.2f} s")

    for row in metric_rows:

        print(
            f"{row['Target']:<5} | "
            f"MAE={row['MAE']:.3f} | "
            f"RMSE={row['RMSE']:.3f} | "
            f"R²={row['R2']:.6f}"
        )

        screening_rows.append({
            "Context_hours": seq_len,
            "Parameters": params,
            "Best_epoch": best_epoch,
            "Best_val_loss": best_val_loss,
            "Training_time_s": training_time,
            **row
        })

    # Do not retain screening models
    del model
    gc.collect()


# ------------------------------------------------------------
# Results table
# ------------------------------------------------------------
context_screening_results = pd.DataFrame(
    screening_rows
)

print("\n" + "=" * 78)
print("CONTEXT SCREENING RESULTS")
print("=" * 78)

print(
    context_screening_results.to_string(
        index=False,
        formatters={
            "Best_val_loss": lambda x: f"{x:.6f}",
            "Training_time_s": lambda x: f"{x:.2f}",
            "MAE": lambda x: f"{x:.3f}",
            "RMSE": lambda x: f"{x:.3f}",
            "R2": lambda x: f"{x:.6f}"
        }
    )
)


# ------------------------------------------------------------
# Context-level balanced validation criterion
#
# Normalized RMSE uses TRAINING target scale so load and solar
# contribute equally despite different physical magnitudes.
# ------------------------------------------------------------
context_scores = []

for seq_len in CONTEXTS:

    temp = context_screening_results[
        context_screening_results["Context_hours"] == seq_len
    ]

    load_rmse = float(
        temp.loc[temp["Target"] == "Load", "RMSE"].iloc[0]
    )

    solar_rmse = float(
        temp.loc[temp["Target"] == "Solar", "RMSE"].iloc[0]
    )

    load_nrmse = load_rmse / target_scaler.scale_[0]
    solar_nrmse = solar_rmse / target_scaler.scale_[1]

    balanced_nrmse = (
        load_nrmse + solar_nrmse
    ) / 2.0

    context_scores.append({
        "Context_hours": seq_len,
        "Load_NRMSE": load_nrmse,
        "Solar_NRMSE": solar_nrmse,
        "Balanced_NRMSE": balanced_nrmse
    })


context_scores = pd.DataFrame(context_scores)

print("\n" + "-" * 78)
print("BALANCED VALIDATION CONTEXT SCORE")
print("-" * 78)

print(
    context_scores.to_string(
        index=False,
        formatters={
            "Load_NRMSE": lambda x: f"{x:.6f}",
            "Solar_NRMSE": lambda x: f"{x:.6f}",
            "Balanced_NRMSE": lambda x: f"{x:.6f}"
        }
    )
)

best_context = int(
    context_scores
    .sort_values("Balanced_NRMSE")
    .iloc[0]["Context_hours"]
)

print(
    f"\nBEST VALIDATION CONTEXT = "
    f"{best_context} HOURS"
)


# ------------------------------------------------------------
# Save evidence
# ------------------------------------------------------------
context_screening_results.to_csv(
    "context_screening_results.csv",
    index=False
)

context_scores.to_csv(
    "context_balanced_scores.csv",
    index=False
)

print("\nSaved:")
print("  context_screening_results.csv")
print("  context_balanced_scores.csv")

print("\n" + "=" * 78)
print("STEP 4.1 COMPLETE")
print("=" * 78)

STEP 4.1 — TRANSFORMER CONTEXT LENGTH SCREENING

Fixed architecture:
d_model       : 64
attention heads: 4
FF dimension  : 128
blocks        : 2
dropout       : 0.2
learning rate : 0.001
batch size    : 64
max epochs    : 50
patience      : 7
seed          : 42

TRAINING CONTEXT = 24 HOURS


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_sequence      │ (None, 24, 8)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_projection    │ (None, 24, 64)    │        576 │ input_sequence[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_encoding │ (None, 24, 64)    │          0 │ input_projection… │
│ (SinusoidalPositio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 24, 64)    │     16,640 │ positional_encod… │
│ (MultiHeadAttentio… │                   │            │ positional_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 24, 64)    │          0 │ positional_encod… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 24, 64)    │        128 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 24, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 24, 128)   │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 24, 64)    │      8,256 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 24, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 24, 64)    │        128 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 24, 64)    │     16,640 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 24, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 24, 64)    │        128 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 24, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 24, 128)   │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 24, 64)    │      8,256 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 24, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 24, 64)    │        128 │ add_3[0][0]     

 Total params: 71,810 (280.51 KB)

 Trainable params: 71,810 (280.51 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
545/545 - 16s - 30ms/step - load_output_loss: 0.0848 - loss: 0.1670 - solar_output_loss: 0.0821 - val_load_output_loss: 0.0165 - val_loss: 0.0470 - val_solar_output_loss: 0.0305
Epoch 2/50
545/545 - 8s - 14ms/step - load_output_loss: 0.0265 - loss: 0.0510 - solar_output_loss: 0.0245 - val_load_output_loss: 0.0108 - val_loss: 0.0237 - val_solar_output_loss: 0.0129
Epoch 3/50
545/545 - 8s - 14ms/step - load_output_loss: 0.0218 - loss: 0.0418 - solar_output_loss: 0.0200 - val_load_output_loss: 0.0088 - val_loss: 0.0176 - val_solar_output_loss: 0.0088
Epoch 4/50
545/545 - 8s - 14ms/step - load_output_loss: 0.0198 - loss: 0.0383 - solar_output_loss: 0.0185 - val_load_output_loss: 0.0081 - val_loss: 0.0296 - val_solar_output_loss: 0.0215
Epoch 5/50
545/545 - 7s - 14ms/step - load_output_loss: 0.0183 - loss: 0.0364 - solar_output_loss: 0.0180 - val_load_output_loss: 0.0138 - val_loss: 0.0290 - val_solar_output_loss: 0.0153
Epoch 6/50
545/545 - 8s - 14ms/step - load_output_loss: 0.0

Save the Colab

In [10]:
# ============================================================
# CHECKPOINT — SAVE ALL WORK BEFORE STOPPING
# ============================================================

import os
import json
import joblib
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/SolarLoad_MTL_Study"
os.makedirs(SAVE_DIR, exist_ok=True)

# ------------------------------------------------------------
# Save result tables
# ------------------------------------------------------------
baseline_results.to_csv(
    os.path.join(SAVE_DIR, "baseline_results.csv"),
    index=False
)

context_screening_results.to_csv(
    os.path.join(SAVE_DIR, "context_screening_results.csv"),
    index=False
)

context_scores.to_csv(
    os.path.join(SAVE_DIR, "context_balanced_scores.csv"),
    index=False
)

sequence_audit.to_csv(
    os.path.join(SAVE_DIR, "sequence_audit.csv"),
    index=False
)

split_audit.to_csv(
    os.path.join(SAVE_DIR, "split_audit.csv"),
    index=False
)

# ------------------------------------------------------------
# Save fitted TRAIN-ONLY scalers
# ------------------------------------------------------------
joblib.dump(
    feature_scaler,
    os.path.join(SAVE_DIR, "feature_scaler.joblib")
)

joblib.dump(
    target_scaler,
    os.path.join(SAVE_DIR, "target_scaler.joblib")
)

# ------------------------------------------------------------
# Save experiment manifest
# ------------------------------------------------------------
manifest = {
    "study": "Solar-Load Multi-Task Transformer Study",

    "dataset": {
        "source": "Open Power System Data",
        "release": "2020-10-06",
        "resolution": "60 min",
        "raw_rows": 50401,
        "raw_columns": 300,
        "load_target": "DE_load_actual_entsoe_transparency",
        "solar_target": "DE_solar_generation_actual"
    },

    "split": {
        "train": "2015-01-01 00:00 UTC -> 2018-12-31 23:00 UTC",
        "validation": "2019-01-01 00:00 UTC -> 2019-12-31 23:00 UTC",
        "test": "2020-01-01 00:00 UTC -> 2020-09-30 23:00 UTC"
    },

    "missing_data": {
        "load_missing": 0,
        "solar_missing": 103,
        "solar_missing_runs": 5,
        "longest_solar_gap_hours": 24,
        "policy": "No interpolation. Reject supervised windows containing missing target observations."
    },

    "features": {
        "n_features": 8,
        "columns": FEATURE_COLS,
        "forecast_horizon_hours": 1
    },

    "sequence_candidates_hours": [24, 48, 168],
    "selected_context_hours": 168,

    "screening_transformer": {
        "d_model": 64,
        "num_heads": 4,
        "ff_dim": 128,
        "num_blocks": 2,
        "dropout": 0.20,
        "learning_rate": 0.001,
        "batch_size": 64,
        "max_epochs": 50,
        "early_stopping_patience": 7,
        "seed": 42,
        "parameters": 71810
    },

    "selected_context_validation": {
        "context_hours": 168,
        "best_epoch": 10,
        "best_val_loss": 0.011494,
        "load_mae": 579.762,
        "load_rmse": 764.885,
        "load_r2": 0.994021,
        "solar_mae": 296.219,
        "solar_rmse": 483.675,
        "solar_r2": 0.995595,
        "balanced_nrmse": 0.075809
    },

    "environment": {
        "python": "3.13.15",
        "numpy": "2.1.3",
        "pandas": "2.2.3",
        "tensorflow": "2.20.0",
        "pytorch": "2.11.0+cu128",
        "cuda": "12.8",
        "gpu": "NVIDIA A100-SXM4-40GB",
        "ram_gb": 83.47,
        "cpu_cores": 12,
        "seed": 42
    },

    "status": {
        "R1_environment": "FROZEN",
        "D0_dataset": "FROZEN",
        "D1_targets": "FROZEN",
        "D2_missingness": "FROZEN",
        "D3_missing_policy": "FROZEN",
        "D4_chronological_split": "FROZEN",
        "F1_features": "FROZEN",
        "F2_scaling": "FROZEN",
        "F3_sequences": "FROZEN",
        "B1_naive_baselines": "FROZEN",
        "M1_context_selection": "FROZEN"
    },

    "next_step": "Step 4.2 — Controlled model comparison using selected 168-hour context."
}

with open(
    os.path.join(SAVE_DIR, "experiment_manifest.json"),
    "w"
) as f:
    json.dump(manifest, f, indent=4)

print("=" * 70)
print("CHECKPOINT SAVED SUCCESSFULLY")
print("=" * 70)

print(f"\nLocation: {SAVE_DIR}\n")

for file in sorted(os.listdir(SAVE_DIR)):
    print(" -", file)

Mounted at /content/drive
CHECKPOINT SAVED SUCCESSFULLY

Location: /content/drive/MyDrive/SolarLoad_MTL_Study

 - baseline_results.csv
 - context_balanced_scores.csv
 - context_screening_results.csv
 - experiment_manifest.json
 - feature_scaler.joblib
 - sequence_audit.csv
 - split_audit.csv
 - target_scaler.joblib


Step R1 — Mount Drive and restore the saved checkpoint

In [1]:
# ============================================================
# STEP R1 — RESTORE SAVED EXPERIMENT CHECKPOINT
# ============================================================

import os
import json
import joblib
import pandas as pd

from google.colab import drive

print("=" * 78)
print("STEP R1 — RESTORE SAVED EXPERIMENT CHECKPOINT")
print("=" * 78)

# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------
drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/SolarLoad_MTL_Study"

if not os.path.exists(SAVE_DIR):
    raise FileNotFoundError(
        f"Checkpoint directory not found:\n{SAVE_DIR}"
    )

print(f"\nCheckpoint directory found:\n{SAVE_DIR}")


# ------------------------------------------------------------
# 2. Show saved files
# ------------------------------------------------------------
print("\n" + "-" * 78)
print("SAVED FILES")
print("-" * 78)

saved_files = sorted(os.listdir(SAVE_DIR))

for filename in saved_files:
    print(" -", filename)


# ------------------------------------------------------------
# 3. Required checkpoint files
# ------------------------------------------------------------
required_files = [
    "baseline_results.csv",
    "context_screening_results.csv",
    "context_balanced_scores.csv",
    "sequence_audit.csv",
    "split_audit.csv",
    "feature_scaler.joblib",
    "target_scaler.joblib",
    "experiment_manifest.json"
]

print("\n" + "-" * 78)
print("CHECKPOINT FILE VERIFICATION")
print("-" * 78)

missing_files = []

for filename in required_files:

    path = os.path.join(SAVE_DIR, filename)

    if os.path.exists(path):
        print(f"✓ {filename}")
    else:
        print(f"✗ MISSING: {filename}")
        missing_files.append(filename)

if missing_files:
    raise FileNotFoundError(
        "Checkpoint is incomplete. Missing:\n" +
        "\n".join(missing_files)
    )

print("\n✓ All required checkpoint files are present.")


# ------------------------------------------------------------
# 4. Restore saved result tables
# ------------------------------------------------------------
baseline_results = pd.read_csv(
    os.path.join(SAVE_DIR, "baseline_results.csv")
)

context_screening_results = pd.read_csv(
    os.path.join(SAVE_DIR, "context_screening_results.csv")
)

context_scores = pd.read_csv(
    os.path.join(SAVE_DIR, "context_balanced_scores.csv")
)

sequence_audit = pd.read_csv(
    os.path.join(SAVE_DIR, "sequence_audit.csv")
)

split_audit = pd.read_csv(
    os.path.join(SAVE_DIR, "split_audit.csv")
)


# ------------------------------------------------------------
# 5. Restore fitted TRAIN-ONLY scalers
# ------------------------------------------------------------
feature_scaler = joblib.load(
    os.path.join(SAVE_DIR, "feature_scaler.joblib")
)

target_scaler = joblib.load(
    os.path.join(SAVE_DIR, "target_scaler.joblib")
)


# ------------------------------------------------------------
# 6. Restore experiment manifest
# ------------------------------------------------------------
with open(
    os.path.join(SAVE_DIR, "experiment_manifest.json"),
    "r"
) as f:

    manifest = json.load(f)


# ------------------------------------------------------------
# 7. Critical frozen-state verification
# ------------------------------------------------------------
print("\n" + "-" * 78)
print("FROZEN EXPERIMENT STATE")
print("-" * 78)

print(
    "Selected context:",
    manifest["selected_context_hours"],
    "hours"
)

print(
    "Forecast horizon:",
    manifest["features"]["forecast_horizon_hours"],
    "hour"
)

print(
    "Number of features:",
    manifest["features"]["n_features"]
)

print(
    "Feature columns:",
    manifest["features"]["columns"]
)

print("\nScaler parameters:")

print(
    "Feature means :",
    feature_scaler.mean_
)

print(
    "Feature scales:",
    feature_scaler.scale_
)

print(
    "Target means  :",
    target_scaler.mean_
)

print(
    "Target scales :",
    target_scaler.scale_
)


# ------------------------------------------------------------
# 8. Verify selected context from saved screening table
# ------------------------------------------------------------
best_saved_context = int(
    context_scores
    .sort_values("Balanced_NRMSE")
    .iloc[0]["Context_hours"]
)

assert best_saved_context == 168
assert manifest["selected_context_hours"] == 168

print(
    f"\n✓ Saved validation-selected context verified: "
    f"{best_saved_context} hours"
)


# ------------------------------------------------------------
# 9. Show frozen baseline evidence
# ------------------------------------------------------------
print("\n" + "-" * 78)
print("RESTORED BASELINE RESULTS")
print("-" * 78)

print(
    baseline_results.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 10. Show frozen context evidence
# ------------------------------------------------------------
print("\n" + "-" * 78)
print("RESTORED CONTEXT SCORES")
print("-" * 78)

print(
    context_scores.to_string(
        index=False
    )
)


print("\n" + "=" * 78)
print("CHECKPOINT RESTORE SUCCESSFUL")
print("=" * 78)

print(
    "\nNext action: reconstruct runtime dataset objects "
    "without repeating model training."
)

STEP R1 — RESTORE SAVED EXPERIMENT CHECKPOINT
Mounted at /content/drive

Checkpoint directory found:
/content/drive/MyDrive/SolarLoad_MTL_Study

------------------------------------------------------------------------------
SAVED FILES
------------------------------------------------------------------------------
 - baseline_results.csv
 - context_balanced_scores.csv
 - context_screening_results.csv
 - experiment_manifest.json
 - feature_scaler.joblib
 - sequence_audit.csv
 - split_audit.csv
 - target_scaler.joblib

------------------------------------------------------------------------------
CHECKPOINT FILE VERIFICATION
------------------------------------------------------------------------------
✓ baseline_results.csv
✓ context_screening_results.csv
✓ context_balanced_scores.csv
✓ sequence_audit.csv
✓ split_audit.csv
✓ feature_scaler.joblib
✓ target_scaler.joblib
✓ experiment_manifest.json

✓ All required checkpoint files are present.

----------------------------------------------

Step R2 — Reconstruct the Frozen 168-hour Dataset

In [2]:
# ============================================================
# STEP R2 — RECONSTRUCT FROZEN 168-HOUR RUNTIME DATASET
# ============================================================

import os
import numpy as np
import pandas as pd

print("=" * 78)
print("STEP R2 — RECONSTRUCT FROZEN 168-HOUR DATASET")
print("=" * 78)

# ------------------------------------------------------------
# 1. Frozen dataset identity
# ------------------------------------------------------------
DATA_URL = (
    "https://data.open-power-system-data.org/"
    "time_series/2020-10-06/"
    "time_series_60min_singleindex.csv"
)

RAW_FILE = "/content/time_series_60min_singleindex.csv"

LOAD_COL = "DE_load_actual_entsoe_transparency"
SOLAR_COL = "DE_solar_generation_actual"

SEQ_LEN = 168

FEATURE_COLS = [
    LOAD_COL,
    SOLAR_COL,
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "doy_sin",
    "doy_cos"
]

TARGET_COLS = [
    LOAD_COL,
    SOLAR_COL
]

# ------------------------------------------------------------
# 2. Frozen chronological boundaries
# ------------------------------------------------------------
TRAIN_START = pd.Timestamp("2015-01-01 00:00:00", tz="UTC")
TRAIN_END   = pd.Timestamp("2018-12-31 23:00:00", tz="UTC")

VAL_START   = pd.Timestamp("2019-01-01 00:00:00", tz="UTC")
VAL_END     = pd.Timestamp("2019-12-31 23:00:00", tz="UTC")

TEST_START  = pd.Timestamp("2020-01-01 00:00:00", tz="UTC")
TEST_END    = pd.Timestamp("2020-09-30 23:00:00", tz="UTC")


# ------------------------------------------------------------
# 3. Download same frozen OPSD release if necessary
# ------------------------------------------------------------
if not os.path.exists(RAW_FILE):

    print("\nDownloading frozen OPSD release...")

    import urllib.request

    urllib.request.urlretrieve(
        DATA_URL,
        RAW_FILE
    )

    print("✓ Download complete.")

else:
    print("\n✓ Raw OPSD file already exists.")


# ------------------------------------------------------------
# 4. Load raw dataset
# ------------------------------------------------------------
print("\nLoading dataset...")

df_raw = pd.read_csv(RAW_FILE)

print(f"Raw rows   : {len(df_raw):,}")
print(f"Raw columns: {len(df_raw.columns):,}")

assert len(df_raw) == 50401
assert len(df_raw.columns) == 300

print("✓ Raw dataset dimensions verified.")


# ------------------------------------------------------------
# 5. Parse timestamp and retain frozen study period
# ------------------------------------------------------------
df_raw["utc_timestamp"] = pd.to_datetime(
    df_raw["utc_timestamp"],
    utc=True,
    errors="coerce"
)

df_feat = df_raw[
    [
        "utc_timestamp",
        LOAD_COL,
        SOLAR_COL
    ]
].copy()

df_feat = df_feat[
    (df_feat["utc_timestamp"] >= TRAIN_START) &
    (df_feat["utc_timestamp"] <= TEST_END)
].copy()

df_feat = (
    df_feat
    .sort_values("utc_timestamp")
    .reset_index(drop=True)
)

print("\nStudy-period rows:", f"{len(df_feat):,}")

assert len(df_feat) == 50400

print("✓ Study period verified.")


# ------------------------------------------------------------
# 6. Reconstruct frozen calendar features
# ------------------------------------------------------------
df_feat["hour"] = (
    df_feat["utc_timestamp"].dt.hour
)

df_feat["dayofweek"] = (
    df_feat["utc_timestamp"].dt.dayofweek
)

df_feat["dayofyear"] = (
    df_feat["utc_timestamp"].dt.dayofyear
)

df_feat["hour_sin"] = np.sin(
    2 * np.pi * df_feat["hour"] / 24.0
)

df_feat["hour_cos"] = np.cos(
    2 * np.pi * df_feat["hour"] / 24.0
)

df_feat["dow_sin"] = np.sin(
    2 * np.pi * df_feat["dayofweek"] / 7.0
)

df_feat["dow_cos"] = np.cos(
    2 * np.pi * df_feat["dayofweek"] / 7.0
)

df_feat["doy_sin"] = np.sin(
    2 * np.pi *
    (df_feat["dayofyear"] - 1) /
    365.25
)

df_feat["doy_cos"] = np.cos(
    2 * np.pi *
    (df_feat["dayofyear"] - 1) /
    365.25
)

print("\n✓ Frozen temporal features reconstructed.")


# ------------------------------------------------------------
# 7. Verify raw missingness
# ------------------------------------------------------------
load_missing = int(
    df_feat[LOAD_COL].isna().sum()
)

solar_missing = int(
    df_feat[SOLAR_COL].isna().sum()
)

print("\nRaw missingness:")
print("Load :", load_missing)
print("Solar:", solar_missing)

assert load_missing == 0
assert solar_missing == 103

print("✓ Missingness matches frozen audit.")


# ------------------------------------------------------------
# 8. Apply SAVED scaler — DO NOT REFIT
# ------------------------------------------------------------
df_scaled = df_feat.copy()

df_scaled[
    [LOAD_COL, SOLAR_COL]
] = feature_scaler.transform(
    df_scaled[
        [LOAD_COL, SOLAR_COL]
    ]
)

print("\n✓ Saved train-only scaler applied.")
print("✓ No scaler fitting performed.")


# ------------------------------------------------------------
# 9. Partition function based on TARGET timestamp
# ------------------------------------------------------------
def get_partition(ts):

    if TRAIN_START <= ts <= TRAIN_END:
        return "Train"

    elif VAL_START <= ts <= VAL_END:
        return "Validation"

    elif TEST_START <= ts <= TEST_END:
        return "Test"

    return None


# ------------------------------------------------------------
# 10. Reconstruct frozen 168-hour sequences
# ------------------------------------------------------------
def build_frozen_sequences(df, seq_len=168):

    X = {
        "Train": [],
        "Validation": [],
        "Test": []
    }

    y = {
        "Train": [],
        "Validation": [],
        "Test": []
    }

    times = {
        "Train": [],
        "Validation": [],
        "Test": []
    }

    rejected = {
        "Train": 0,
        "Validation": 0,
        "Test": 0
    }

    features = df[
        FEATURE_COLS
    ].to_numpy(dtype=np.float32)

    targets = df[
        TARGET_COLS
    ].to_numpy(dtype=np.float32)

    timestamps = df[
        "utc_timestamp"
    ].tolist()

    for i in range(seq_len, len(df)):

        target_time = timestamps[i]

        partition = get_partition(
            target_time
        )

        if partition is None:
            continue

        X_window = features[
            i-seq_len:i
        ]

        y_target = targets[i]

        # Frozen missing-data policy
        if (
            not np.isfinite(X_window).all()
            or
            not np.isfinite(y_target).all()
        ):

            rejected[partition] += 1
            continue

        X[partition].append(
            X_window
        )

        y[partition].append(
            y_target
        )

        times[partition].append(
            target_time
        )

    for partition in X:

        X[partition] = np.asarray(
            X[partition],
            dtype=np.float32
        )

        y[partition] = np.asarray(
            y[partition],
            dtype=np.float32
        )

    return X, y, times, rejected


X_sets, y_sets, time_sets, rejected = (
    build_frozen_sequences(
        df_scaled,
        SEQ_LEN
    )
)

sequence_results = {
    SEQ_LEN: {
        "X": X_sets,
        "y": y_sets,
        "times": time_sets
    }
}


# ------------------------------------------------------------
# 11. Verify exact frozen sequence counts
# ------------------------------------------------------------
print("\n" + "-" * 78)
print("RECONSTRUCTED SEQUENCE AUDIT")
print("-" * 78)

expected_counts = {
    "Train": 34121,
    "Validation": 8760,
    "Test": 6576
}

expected_rejected = {
    "Train": 775,
    "Validation": 0,
    "Test": 0
}

for partition in [
    "Train",
    "Validation",
    "Test"
]:

    actual = len(
        X_sets[partition]
    )

    print(f"\n{partition}")
    print(
        "  X shape     :",
        X_sets[partition].shape
    )
    print(
        "  y shape     :",
        y_sets[partition].shape
    )
    print(
        "  Rejected    :",
        rejected[partition]
    )
    print(
        "  First target:",
        time_sets[partition][0]
    )
    print(
        "  Last target :",
        time_sets[partition][-1]
    )

    assert (
        actual ==
        expected_counts[partition]
    )

    assert (
        rejected[partition] ==
        expected_rejected[partition]
    )


# ------------------------------------------------------------
# 12. Verify exact frozen boundaries
# ------------------------------------------------------------
assert (
    time_sets["Train"][0] ==
    pd.Timestamp(
        "2015-01-08 07:00:00",
        tz="UTC"
    )
)

assert (
    time_sets["Train"][-1] ==
    TRAIN_END
)

assert (
    time_sets["Validation"][0] ==
    VAL_START
)

assert (
    time_sets["Validation"][-1] ==
    VAL_END
)

assert (
    time_sets["Test"][0] ==
    TEST_START
)

assert (
    time_sets["Test"][-1] ==
    TEST_END
)


# ------------------------------------------------------------
# 13. Final integrity checks
# ------------------------------------------------------------
assert np.isfinite(
    X_sets["Train"]
).all()

assert np.isfinite(
    y_sets["Train"]
).all()

assert np.isfinite(
    X_sets["Validation"]
).all()

assert np.isfinite(
    y_sets["Validation"]
).all()

assert np.isfinite(
    X_sets["Test"]
).all()

assert np.isfinite(
    y_sets["Test"]
).all()


print("\n" + "=" * 78)
print("RUNTIME RESTORATION SUCCESSFUL")
print("=" * 78)

print("\nFrozen context : 168 hours")
print("Input features : 8")
print("Forecast horizon: 1 hour")

print("\nSequence counts:")
print("Train      :", len(X_sets["Train"]))
print("Validation :", len(X_sets["Validation"]))
print("Test       :", len(X_sets["Test"]))

print(
    "\n✓ All counts, missingness and "
    "temporal boundaries match the frozen checkpoint."
)

print(
    "✓ Runtime is ready to continue from Step 4.2."
)

STEP R2 — RECONSTRUCT FROZEN 168-HOUR DATASET

✓ Download complete.

Loading dataset...
Raw rows   : 50,401
Raw columns: 300
✓ Raw dataset dimensions verified.

Study-period rows: 50,400
✓ Study period verified.

✓ Frozen temporal features reconstructed.

Raw missingness:
Load : 0
Solar: 103
✓ Missingness matches frozen audit.

✓ Saved train-only scaler applied.
✓ No scaler fitting performed.

------------------------------------------------------------------------------
RECONSTRUCTED SEQUENCE AUDIT
------------------------------------------------------------------------------

Train
  X shape     : (34121, 168, 8)
  y shape     : (34121, 2)
  Rejected    : 775
  First target: 2015-01-08 07:00:00+00:00
  Last target : 2018-12-31 23:00:00+00:00

Validation
  X shape     : (8760, 168, 8)
  y shape     : (8760, 2)
  Rejected    : 0
  First target: 2019-01-01 00:00:00+00:00
  Last target : 2019-12-31 23:00:00+00:00

Test
  X shape     : (6576, 168, 8)
  y shape     : (6576, 2)
  Rejected  

Step 4.2 — Controlled Multi-Output LSTM Baseline

In [3]:
# ============================================================
# STEP 4.2 — CONTROLLED MULTI-OUTPUT LSTM BASELINE
# Frozen context: 168 hours
# Validation evaluation only
# ============================================================

import os
import gc
import time
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("=" * 78)
print("STEP 4.2 — CONTROLLED MULTI-OUTPUT LSTM BASELINE")
print("=" * 78)

# ------------------------------------------------------------
# 1. Frozen experimental settings
# ------------------------------------------------------------
SEED = 42
SEQ_LEN = 168

BATCH_SIZE = 64
MAX_EPOCHS = 50
PATIENCE = 7
LEARNING_RATE = 1e-3


def reset_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


reset_seed()


# ------------------------------------------------------------
# 2. Retrieve reconstructed frozen datasets
# ------------------------------------------------------------
X_train = sequence_results[SEQ_LEN]["X"]["Train"]
y_train = sequence_results[SEQ_LEN]["y"]["Train"]

X_val = sequence_results[SEQ_LEN]["X"]["Validation"]
y_val = sequence_results[SEQ_LEN]["y"]["Validation"]


print("\nDataset verification:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val  :", X_val.shape)
print("y_val  :", y_val.shape)


assert X_train.shape == (34121, 168, 8)
assert y_train.shape == (34121, 2)

assert X_val.shape == (8760, 168, 8)
assert y_val.shape == (8760, 2)

print("\n✓ Frozen dataset dimensions verified.")


# ------------------------------------------------------------
# 3. Build controlled LSTM
# ------------------------------------------------------------
tf.keras.backend.clear_session()
gc.collect()
reset_seed()


inputs = keras.Input(
    shape=(SEQ_LEN, 8),
    name="input_sequence"
)


x = layers.LSTM(
    64,
    return_sequences=True,
    name="lstm_1"
)(inputs)


x = layers.Dropout(
    0.20,
    name="dropout_1"
)(x)


x = layers.LSTM(
    32,
    return_sequences=False,
    name="lstm_2"
)(x)


x = layers.Dropout(
    0.20,
    name="dropout_2"
)(x)


x = layers.Dense(
    64,
    activation="gelu",
    name="shared_dense"
)(x)


x = layers.Dropout(
    0.20,
    name="shared_dropout"
)(x)


load_output = layers.Dense(
    1,
    name="load_output"
)(x)


solar_output = layers.Dense(
    1,
    name="solar_output"
)(x)


lstm_model = keras.Model(
    inputs=inputs,
    outputs=[
        load_output,
        solar_output
    ],
    name="controlled_multioutput_lstm"
)


lstm_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    ),
    loss={
        "load_output": "mse",
        "solar_output": "mse"
    }
)


# ------------------------------------------------------------
# 4. Architecture audit
# ------------------------------------------------------------
print("\n" + "-" * 78)
print("MODEL ARCHITECTURE")
print("-" * 78)

lstm_model.summary()

LSTM_PARAMS = lstm_model.count_params()

print(
    f"\nTotal trainable model parameters: "
    f"{LSTM_PARAMS:,}"
)


# ------------------------------------------------------------
# 5. Train using TRAIN + VALIDATION only
# ------------------------------------------------------------
callbacks = [

    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=1
    )

]


start_time = time.perf_counter()


history_lstm = lstm_model.fit(

    X_train,

    {
        "load_output": y_train[:, 0],
        "solar_output": y_train[:, 1]
    },

    validation_data=(

        X_val,

        {
            "load_output": y_val[:, 0],
            "solar_output": y_val[:, 1]
        }

    ),

    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,

    callbacks=callbacks,

    shuffle=True,

    verbose=2
)


LSTM_TRAIN_TIME = (
    time.perf_counter() -
    start_time
)


# ------------------------------------------------------------
# 6. Determine best epoch
# ------------------------------------------------------------
best_epoch = int(
    np.argmin(
        history_lstm.history["val_loss"]
    ) + 1
)


best_val_loss = float(
    np.min(
        history_lstm.history["val_loss"]
    )
)


# ------------------------------------------------------------
# 7. Validation prediction ONLY
#
# IMPORTANT:
# Test set remains untouched.
# ------------------------------------------------------------
pred = lstm_model.predict(
    X_val,
    batch_size=BATCH_SIZE,
    verbose=0
)


pred_scaled = np.column_stack([
    pred[0].reshape(-1),
    pred[1].reshape(-1)
])


# ------------------------------------------------------------
# 8. Return predictions to original scale
# ------------------------------------------------------------
y_val_original = (
    target_scaler.inverse_transform(
        y_val
    )
)


pred_original = (
    target_scaler.inverse_transform(
        pred_scaled
    )
)


# ------------------------------------------------------------
# 9. Compute validation metrics
# ------------------------------------------------------------
metric_rows = []


for idx, target in enumerate(
    ["Load", "Solar"]
):

    mae = mean_absolute_error(
        y_val_original[:, idx],
        pred_original[:, idx]
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_val_original[:, idx],
            pred_original[:, idx]
        )
    )

    r2 = r2_score(
        y_val_original[:, idx],
        pred_original[:, idx]
    )


    metric_rows.append({

        "Model":
            "Multi-output LSTM",

        "Context_hours":
            SEQ_LEN,

        "Target":
            target,

        "MAE":
            mae,

        "RMSE":
            rmse,

        "R2":
            r2,

        "Parameters":
            LSTM_PARAMS,

        "Best_epoch":
            best_epoch,

        "Best_val_loss":
            best_val_loss,

        "Training_time_s":
            LSTM_TRAIN_TIME
    })


lstm_validation_results = (
    pd.DataFrame(metric_rows)
)


# ------------------------------------------------------------
# 10. Display results
# ------------------------------------------------------------
print("\n" + "=" * 78)
print("LSTM VALIDATION RESULTS")
print("=" * 78)


print(
    f"Best epoch    : "
    f"{best_epoch}"
)

print(
    f"Best val loss : "
    f"{best_val_loss:.6f}"
)

print(
    f"Training time : "
    f"{LSTM_TRAIN_TIME:.2f} s"
)

print(
    f"Parameters    : "
    f"{LSTM_PARAMS:,}"
)


print(
    "\n" +
    lstm_validation_results[
        [
            "Target",
            "MAE",
            "RMSE",
            "R2"
        ]
    ].to_string(

        index=False,

        formatters={

            "MAE":
                lambda x: f"{x:.3f}",

            "RMSE":
                lambda x: f"{x:.3f}",

            "R2":
                lambda x: f"{x:.6f}"
        }
    )
)


# ------------------------------------------------------------
# 11. Compare with frozen persistence baseline
# ------------------------------------------------------------
print("\n" + "-" * 78)
print("IMPROVEMENT OVER PERSISTENCE — VALIDATION")
print("-" * 78)


for target in [
    "Load",
    "Solar"
]:

    lstm_rmse = float(

        lstm_validation_results.loc[

            lstm_validation_results[
                "Target"
            ] == target,

            "RMSE"

        ].iloc[0]
    )


    persistence_rmse = float(

        baseline_results.loc[

            (
                baseline_results["Period"]
                == "Validation"
            )

            &

            (
                baseline_results["Model"]
                == "Persistence (t-1)"
            )

            &

            (
                baseline_results["Target"]
                == target
            ),

            "RMSE"

        ].iloc[0]
    )


    improvement = (

        (
            persistence_rmse -
            lstm_rmse
        )

        /

        persistence_rmse

        * 100
    )


    print(

        f"{target:<6}: "

        f"Persistence RMSE="
        f"{persistence_rmse:.3f} | "

        f"LSTM RMSE="
        f"{lstm_rmse:.3f} | "

        f"Improvement="
        f"{improvement:.2f}%"
    )


# ------------------------------------------------------------
# 12. Save results to checkpoint
# ------------------------------------------------------------
SAVE_DIR = (
    "/content/drive/MyDrive/"
    "SolarLoad_MTL_Study"
)


lstm_validation_results.to_csv(

    os.path.join(
        SAVE_DIR,
        "lstm_validation_results.csv"
    ),

    index=False
)


pd.DataFrame(
    history_lstm.history
).to_csv(

    os.path.join(
        SAVE_DIR,
        "lstm_training_history.csv"
    ),

    index=False
)


lstm_model.save(

    os.path.join(
        SAVE_DIR,
        "lstm_168h_seed42.keras"
    )
)


print("\nSaved:")
print(
    " ✓ lstm_validation_results.csv"
)
print(
    " ✓ lstm_training_history.csv"
)
print(
    " ✓ lstm_168h_seed42.keras"
)


print(
    "\n✓ TEST SET HAS NOT BEEN "
    "EVALUATED."
)

print("\n" + "=" * 78)
print("STEP 4.2 COMPLETE")
print("=" * 78)

STEP 4.2 — CONTROLLED MULTI-OUTPUT LSTM BASELINE

Dataset verification:
X_train: (34121, 168, 8)
y_train: (34121, 2)
X_val  : (8760, 168, 8)
y_val  : (8760, 2)

✓ Frozen dataset dimensions verified.

------------------------------------------------------------------------------
MODEL ARCHITECTURE
------------------------------------------------------------------------------


Model: "controlled_multioutput_lstm"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_sequence      │ (None, 168, 8)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 168, 64)   │     18,688 │ input_sequence[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 168, 64)   │          0 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 32)        │     12,416 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 32)        │          0 │ lstm_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dense        │ (None, 64)        │      2,112 │ dropout_2[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dropout      │ (None, 64)        │          0 │ shared_dense[0][… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ load_output (Dense) │ (None, 1)         │         65 │ shared_dropout[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ solar_output        │ (None, 1)         │         65 │ shared_dropout[0… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 33,346 (130.26 KB)

 Trainable params: 33,346 (130.26 KB)

 Non-trainable params: 0 (0.00 B)


Total trainable model parameters: 33,346
Epoch 1/50
534/534 - 14s - 27ms/step - load_output_loss: 0.1191 - loss: 0.2492 - solar_output_loss: 0.1300 - val_load_output_loss: 0.0294 - val_loss: 0.0665 - val_solar_output_loss: 0.0372
Epoch 2/50
534/534 - 9s - 17ms/step - load_output_loss: 0.0551 - loss: 0.1046 - solar_output_loss: 0.0495 - val_load_output_loss: 0.0221 - val_loss: 0.0402 - val_solar_output_loss: 0.0181
Epoch 3/50
534/534 - 9s - 17ms/step - load_output_loss: 0.0471 - loss: 0.0851 - solar_output_loss: 0.0380 - val_load_output_loss: 0.0161 - val_loss: 0.0279 - val_solar_output_loss: 0.0118
Epoch 4/50
534/534 - 9s - 18ms/step - load_output_loss: 0.0415 - loss: 0.0750 - solar_output_loss: 0.0335 - val_load_output_loss: 0.0151 - val_loss: 0.0251 - val_solar_output_loss: 0.0100
Epoch 5/50
534/534 - 9s - 17ms/step - load_output_loss: 0.0371 - loss: 0.0669 - solar_output_loss: 0.0298 - val_load_output_loss: 0.0124 - val_loss: 0.0214 - val_solar_output_loss: 0.0090
Epoch 6/50
534/53

Step 4.3 configuration

In [4]:
# ============================================================
# CHECKPOINT — SAVE FROZEN 168-HOUR ARRAYS
# ============================================================

import os
import numpy as np

SAVE_DIR = "/content/drive/MyDrive/SolarLoad_MTL_Study"

ARRAY_FILE = os.path.join(
    SAVE_DIR,
    "frozen_168h_sequences.npz"
)

np.savez_compressed(
    ARRAY_FILE,

    X_train=sequence_results[168]["X"]["Train"],
    y_train=sequence_results[168]["y"]["Train"],

    X_val=sequence_results[168]["X"]["Validation"],
    y_val=sequence_results[168]["y"]["Validation"],

    X_test=sequence_results[168]["X"]["Test"],
    y_test=sequence_results[168]["y"]["Test"]
)

print("=" * 70)
print("FROZEN ARRAYS SAVED")
print("=" * 70)

print("\nFile:", ARRAY_FILE)

# Verify saved file
with np.load(ARRAY_FILE) as data:

    print("\nVerification:")

    for key in data.files:
        print(f"{key:<10}: {data[key].shape}")

print("\n✓ Expected:")
print("X_train : (34121, 168, 8)")
print("y_train : (34121, 2)")
print("X_val   : (8760, 168, 8)")
print("y_val   : (8760, 2)")
print("X_test  : (6576, 168, 8)")
print("y_test  : (6576, 2)")

print(
    "\n✓ Future Colab sessions can restore "
    "the frozen arrays directly."
)

FROZEN ARRAYS SAVED

File: /content/drive/MyDrive/SolarLoad_MTL_Study/frozen_168h_sequences.npz

Verification:
X_train   : (34121, 168, 8)
y_train   : (34121, 2)
X_val     : (8760, 168, 8)
y_val     : (8760, 2)
X_test    : (6576, 168, 8)
y_test    : (6576, 2)

✓ Expected:
X_train : (34121, 168, 8)
y_train : (34121, 2)
X_val   : (8760, 168, 8)
y_val   : (8760, 2)
X_test  : (6576, 168, 8)
y_test  : (6576, 2)

✓ Future Colab sessions can restore the frozen arrays directly.


Step 4.3 — Single-Task Transformer Controls

In [5]:
# ============================================================
# STEP 4.3 — SINGLE-TASK TRANSFORMER CONTROLS
# Two independent Transformers: Load and Solar
# Frozen context: 168 h
# Validation evaluation only
# ============================================================

import os
import gc
import time
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("=" * 78)
print("STEP 4.3 — SINGLE-TASK TRANSFORMER CONTROLS")
print("=" * 78)

# ------------------------------------------------------------
# 1. Frozen configuration
# ------------------------------------------------------------
SEED = 42
SEQ_LEN = 168

D_MODEL = 64
NUM_HEADS = 4
FF_DIM = 128
NUM_BLOCKS = 2
DROPOUT = 0.20

LEARNING_RATE = 1e-3
BATCH_SIZE = 64
MAX_EPOCHS = 50
PATIENCE = 7

SAVE_DIR = "/content/drive/MyDrive/SolarLoad_MTL_Study"


def reset_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


# ------------------------------------------------------------
# 2. Retrieve frozen datasets
# ------------------------------------------------------------
X_train = sequence_results[SEQ_LEN]["X"]["Train"]
y_train = sequence_results[SEQ_LEN]["y"]["Train"]

X_val = sequence_results[SEQ_LEN]["X"]["Validation"]
y_val = sequence_results[SEQ_LEN]["y"]["Validation"]


assert X_train.shape == (34121, 168, 8)
assert y_train.shape == (34121, 2)

assert X_val.shape == (8760, 168, 8)
assert y_val.shape == (8760, 2)

print("\nFrozen dataset:")
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)

print("\n✓ Dataset verified.")


# ------------------------------------------------------------
# 3. Positional encoding
# ------------------------------------------------------------
class SinusoidalPositionalEncoding(layers.Layer):

    def __init__(self, d_model, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model

    def call(self, inputs):

        seq_len = tf.shape(inputs)[1]

        position = tf.cast(
            tf.range(seq_len)[:, tf.newaxis],
            tf.float32
        )

        div_term = tf.exp(
            tf.cast(
                tf.range(0, self.d_model, 2),
                tf.float32
            )
            *
            (-np.log(10000.0) / self.d_model)
        )

        sin_part = tf.sin(
            position * div_term
        )

        cos_part = tf.cos(
            position * div_term
        )

        pos_encoding = tf.reshape(
            tf.stack(
                [sin_part, cos_part],
                axis=-1
            ),
            (seq_len, self.d_model)
        )

        return (
            inputs +
            pos_encoding[tf.newaxis, :, :]
        )


# ------------------------------------------------------------
# 4. Transformer block
# ------------------------------------------------------------
def transformer_block(x, block_id):

    attn_output = layers.MultiHeadAttention(
        num_heads=NUM_HEADS,
        key_dim=D_MODEL // NUM_HEADS,
        dropout=DROPOUT,
        name=f"mha_{block_id}"
    )(x, x)

    x = layers.Add(
        name=f"attn_residual_{block_id}"
    )([x, attn_output])

    x = layers.LayerNormalization(
        epsilon=1e-6,
        name=f"attn_norm_{block_id}"
    )(x)

    ff = layers.Dense(
        FF_DIM,
        activation="gelu",
        name=f"ff1_{block_id}"
    )(x)

    ff = layers.Dropout(
        DROPOUT,
        name=f"ff_dropout_{block_id}"
    )(ff)

    ff = layers.Dense(
        D_MODEL,
        name=f"ff2_{block_id}"
    )(ff)

    x = layers.Add(
        name=f"ff_residual_{block_id}"
    )([x, ff])

    x = layers.LayerNormalization(
        epsilon=1e-6,
        name=f"ff_norm_{block_id}"
    )(x)

    return x


# ------------------------------------------------------------
# 5. Single-task Transformer
# ------------------------------------------------------------
def build_single_task_transformer(task_name):

    inputs = keras.Input(
        shape=(SEQ_LEN, 8),
        name="input_sequence"
    )

    x = layers.Dense(
        D_MODEL,
        name="input_projection"
    )(inputs)

    x = SinusoidalPositionalEncoding(
        D_MODEL,
        name="positional_encoding"
    )(x)

    for block_id in range(1, NUM_BLOCKS + 1):
        x = transformer_block(
            x,
            block_id
        )

    x = layers.GlobalAveragePooling1D(
        name="global_average_pooling"
    )(x)

    x = layers.Dense(
        64,
        activation="gelu",
        name="task_dense"
    )(x)

    x = layers.Dropout(
        DROPOUT,
        name="task_dropout"
    )(x)

    output = layers.Dense(
        1,
        name=f"{task_name.lower()}_output"
    )(x)

    model = keras.Model(
        inputs=inputs,
        outputs=output,
        name=f"single_task_{task_name.lower()}_transformer"
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="mse"
    )

    return model


# ------------------------------------------------------------
# 6. Train/evaluate one task
# ------------------------------------------------------------
def run_single_task(
    task_name,
    target_index
):

    print("\n" + "=" * 78)
    print(
        f"SINGLE-TASK TRANSFORMER — "
        f"{task_name.upper()}"
    )
    print("=" * 78)

    tf.keras.backend.clear_session()
    gc.collect()
    reset_seed()

    model = build_single_task_transformer(
        task_name
    )

    model.summary()

    params = model.count_params()

    print(
        f"\nParameters: {params:,}"
    )

    callbacks = [

        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=1
        )

    ]

    start_time = time.perf_counter()

    history = model.fit(

        X_train,
        y_train[:, target_index],

        validation_data=(
            X_val,
            y_val[:, target_index]
        ),

        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,

        callbacks=callbacks,

        shuffle=True,
        verbose=2
    )

    training_time = (
        time.perf_counter() -
        start_time
    )

    best_epoch = int(
        np.argmin(
            history.history["val_loss"]
        ) + 1
    )

    best_val_loss = float(
        np.min(
            history.history["val_loss"]
        )
    )

    # --------------------------------------------------------
    # Validation prediction ONLY
    # --------------------------------------------------------
    pred_scaled = model.predict(
        X_val,
        batch_size=BATCH_SIZE,
        verbose=0
    ).reshape(-1)

    # Convert only this target to original scale
    target_mean = target_scaler.mean_[
        target_index
    ]

    target_scale = target_scaler.scale_[
        target_index
    ]

    y_true_original = (
        y_val[:, target_index] *
        target_scale +
        target_mean
    )

    pred_original = (
        pred_scaled *
        target_scale +
        target_mean
    )

    mae = mean_absolute_error(
        y_true_original,
        pred_original
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true_original,
            pred_original
        )
    )

    r2 = r2_score(
        y_true_original,
        pred_original
    )

    print("\n" + "-" * 78)
    print(f"{task_name.upper()} VALIDATION RESULT")
    print("-" * 78)

    print(f"Parameters    : {params:,}")
    print(f"Best epoch    : {best_epoch}")
    print(
        f"Best val loss : "
        f"{best_val_loss:.6f}"
    )
    print(
        f"Training time : "
        f"{training_time:.2f} s"
    )

    print(f"MAE           : {mae:.3f}")
    print(f"RMSE          : {rmse:.3f}")
    print(f"R²            : {r2:.6f}")

    # --------------------------------------------------------
    # Save model + history
    # --------------------------------------------------------
    model_path = os.path.join(
        SAVE_DIR,
        f"single_task_{task_name.lower()}_"
        f"transformer_168h_seed42.keras"
    )

    history_path = os.path.join(
        SAVE_DIR,
        f"single_task_{task_name.lower()}_"
        f"history.csv"
    )

    model.save(model_path)

    pd.DataFrame(
        history.history
    ).to_csv(
        history_path,
        index=False
    )

    result = {

        "Model":
            "Single-task Transformer",

        "Task":
            task_name,

        "Context_hours":
            SEQ_LEN,

        "Seed":
            SEED,

        "Parameters":
            params,

        "Best_epoch":
            best_epoch,

        "Best_val_loss":
            best_val_loss,

        "Training_time_s":
            training_time,

        "MAE":
            mae,

        "RMSE":
            rmse,

        "R2":
            r2
    }

    del model
    gc.collect()

    return result


# ------------------------------------------------------------
# 7. Run Load Transformer
# ------------------------------------------------------------
load_result = run_single_task(
    task_name="Load",
    target_index=0
)


# ------------------------------------------------------------
# 8. Run Solar Transformer
# ------------------------------------------------------------
solar_result = run_single_task(
    task_name="Solar",
    target_index=1
)


# ------------------------------------------------------------
# 9. Consolidate results
# ------------------------------------------------------------
single_task_results = pd.DataFrame([
    load_result,
    solar_result
])


print("\n" + "=" * 78)
print("SINGLE-TASK TRANSFORMER SUMMARY")
print("=" * 78)

print(
    single_task_results.to_string(

        index=False,

        formatters={

            "Best_val_loss":
                lambda x: f"{x:.6f}",

            "Training_time_s":
                lambda x: f"{x:.2f}",

            "MAE":
                lambda x: f"{x:.3f}",

            "RMSE":
                lambda x: f"{x:.3f}",

            "R2":
                lambda x: f"{x:.6f}"
        }
    )
)


# ------------------------------------------------------------
# 10. Combined efficiency
# ------------------------------------------------------------
total_params = int(
    single_task_results[
        "Parameters"
    ].sum()
)

total_training_time = float(
    single_task_results[
        "Training_time_s"
    ].sum()
)


print("\n" + "-" * 78)
print("COMBINED SINGLE-TASK COST")
print("-" * 78)

print(
    f"Combined parameters     : "
    f"{total_params:,}"
)

print(
    f"Combined training time  : "
    f"{total_training_time:.2f} s"
)


# ------------------------------------------------------------
# 11. Save evidence
# ------------------------------------------------------------
single_task_results.to_csv(

    os.path.join(
        SAVE_DIR,
        "single_task_transformer_validation_results.csv"
    ),

    index=False
)


print("\nSaved:")
print(
    " ✓ single_task_transformer_validation_results.csv"
)
print(
    " ✓ single-task Load model + history"
)
print(
    " ✓ single-task Solar model + history"
)

print(
    "\n✓ TEST SET HAS NOT BEEN EVALUATED."
)

print("\n" + "=" * 78)
print("STEP 4.3 COMPLETE")
print("=" * 78)

STEP 4.3 — SINGLE-TASK TRANSFORMER CONTROLS

Frozen dataset:
X_train: (34121, 168, 8)
X_val  : (8760, 168, 8)

✓ Dataset verified.

SINGLE-TASK TRANSFORMER — LOAD


Model: "single_task_load_transformer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_sequence      │ (None, 168, 8)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_projection    │ (None, 168, 64)   │        576 │ input_sequence[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_encoding │ (None, 168, 64)   │          0 │ input_projection… │
│ (SinusoidalPositio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mha_1               │ (None, 168, 64)   │     16,640 │ positional_encod… │
│ (MultiHeadAttentio… │                   │            │ positional_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_residual_1     │ (None, 168, 64)   │          0 │ positional_encod… │
│ (Add)               │                   │            │ mha_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_norm_1         │ (None, 168, 64)   │        128 │ attn_residual_1[… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ff1_1 (Dense)       │ (None, 168, 128)  │      8,320 │ attn_norm_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ff_dropout_1        │ (None, 168, 128)  │          0 │ ff1_1[0][0]       │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ff2_1 (Dense)       │ (None, 168, 64)   │      8,256 │ ff_dropout_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ff_residual_1 (Add) │ (None, 168, 64)   │          0 │ attn_norm_1[0][0… │
│                     │                   │            │ ff2_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ff_norm_1           │ (None, 168, 64)   │        128 │ ff_residual_1[0]… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mha_2               │ (None, 168, 64)   │     16,640 │ ff_norm_1[0][0],  │
│ (MultiHeadAttentio… │                   │            │ ff_norm_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_residual_2     │ (None, 168, 64)   │          0 │ ff_norm_1[0][0],  │
│ (Add)               │                   │            │ mha_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_norm_2         │ (None, 168, 64)   │        128 │ attn_residual_2[… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ff1_2 (Dense)       │ (None, 168, 128)  │      8,320 │ attn_norm_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ff_dropout_2        │ (None, 168, 128)  │          0 │ ff1_2[0][0]       │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ff2_2 (Dense)       │ (None, 168, 64)   │      8,256 │ ff_dropout_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ff_residual_2 (Add) │ (None, 168, 64)   │          0 │ attn_norm_2[0][0… │
│                     │                   │            │ ff2_2[0][0]     

 Total params: 71,745 (280.25 KB)

 Trainable params: 71,745 (280.25 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_sequence      │ (None, 168, 8)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_projection    │ (None, 168, 64)   │        576 │ input_sequence[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_encoding │ (None, 168, 64)   │          0 │ input_projection… │
│ (SinusoidalPositio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mha_1               │ (None, 168, 64)   │     16,640 │ positional_encod… │
│ (MultiHeadAttentio… │                   │            │ positional_encod… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_residual_1     │ (None, 168, 64)   │          0 │ positional_encod… │
│ (Add)               │                   │            │ mha_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_norm_1         │ (None, 168, 64)   │        128 │ attn_residual_1[… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ff1_1 (Dense)       │ (None, 168, 128)  │      8,320 │ attn_norm_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ff_dropout_1        │ (None, 168, 128)  │          0 │ ff1_1[0][0]       │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ff2_1 (Dense)       │ (None, 168, 64)   │      8,256 │ ff_dropout_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ff_residual_1 (Add) │ (None, 168, 64)   │          0 │ attn_norm_1[0][0… │
│                     │                   │            │ ff2_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ff_norm_1           │ (None, 168, 64)   │        128 │ ff_residual_1[0]… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mha_2               │ (None, 168, 64)   │     16,640 │ ff_norm_1[0][0],  │
│ (MultiHeadAttentio… │                   │            │ ff_norm_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_residual_2     │ (None, 168, 64)   │          0 │ ff_norm_1[0][0],  │
│ (Add)               │                   │            │ mha_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attn_norm_2         │ (None, 168, 64)   │        128 │ attn_residual_2[… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ff1_2 (Dense)       │ (None, 168, 128)  │      8,320 │ attn_norm_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ff_dropout_2        │ (None, 168, 128)  │          0 │ ff1_2[0][0]       │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ff2_2 (Dense)       │ (None, 168, 64)   │      8,256 │ ff_dropout_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ff_residual_2 (Add) │ (None, 168, 64)   │          0 │ attn_norm_2[0][0… │
│                     │                   │            │ ff2_2[0][0]     

 Total params: 71,745 (280.25 KB)

 Trainable params: 71,745 (280.25 KB)

 Non-trainable params: 0 (0.00 B)


Parameters: 71,745
Epoch 1/50
534/534 - 42s - 78ms/step - loss: 0.1445 - val_loss: 0.0146
Epoch 2/50
534/534 - 3s - 5ms/step - loss: 0.0187 - val_loss: 0.0082
Epoch 3/50
534/534 - 2s - 5ms/step - loss: 0.0141 - val_loss: 0.0131
Epoch 4/50
534/534 - 2s - 5ms/step - loss: 0.0120 - val_loss: 0.0069
Epoch 5/50
534/534 - 3s - 5ms/step - loss: 0.0110 - val_loss: 0.0139
Epoch 6/50
534/534 - 2s - 5ms/step - loss: 0.0103 - val_loss: 0.0127
Epoch 7/50
534/534 - 3s - 5ms/step - loss: 0.0103 - val_loss: 0.0135
Epoch 8/50
534/534 - 2s - 5ms/step - loss: 0.0096 - val_loss: 0.0125
Epoch 9/50
534/534 - 2s - 5ms/step - loss: 0.0097 - val_loss: 0.0125
Epoch 10/50
534/534 - 2s - 5ms/step - loss: 0.0092 - val_loss: 0.0138
Epoch 11/50
534/534 - 2s - 5ms/step - loss: 0.0092 - val_loss: 0.0147
Epoch 11: early stopping
Restoring model weights from the end of the best epoch: 4.

------------------------------------------------------------------------------
SOLAR VALIDATION RESULT
-----------------------------

Step 4.4 — Five-Seed Reproducibility Experiment

In [6]:
# ============================================================
# STEP 4.4 — FIVE-SEED REPRODUCIBILITY EXPERIMENT
#
# Models:
#   1. Multi-output LSTM
#   2. Single-task Transformer — Load
#   3. Single-task Transformer — Solar
#   4. Multi-task Transformer
#
# Validation only. TEST SET REMAINS UNTOUCHED.
# ============================================================

import os
import gc
import time
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("=" * 78)
print("STEP 4.4 — FIVE-SEED REPRODUCIBILITY EXPERIMENT")
print("=" * 78)

# ------------------------------------------------------------
# 1. FROZEN CONFIGURATION
# ------------------------------------------------------------
SEEDS = [1, 7, 21, 42, 100]

SEQ_LEN = 168

D_MODEL = 64
NUM_HEADS = 4
FF_DIM = 128
NUM_BLOCKS = 2
DROPOUT = 0.20

LEARNING_RATE = 1e-3
BATCH_SIZE = 64
MAX_EPOCHS = 50
PATIENCE = 7

SAVE_DIR = "/content/drive/MyDrive/SolarLoad_MTL_Study"

RESULT_FILE = os.path.join(
    SAVE_DIR,
    "five_seed_validation_runs.csv"
)

SUMMARY_FILE = os.path.join(
    SAVE_DIR,
    "five_seed_validation_summary.csv"
)


# ------------------------------------------------------------
# 2. FROZEN DATA
# ------------------------------------------------------------
X_train = sequence_results[SEQ_LEN]["X"]["Train"]
y_train = sequence_results[SEQ_LEN]["y"]["Train"]

X_val = sequence_results[SEQ_LEN]["X"]["Validation"]
y_val = sequence_results[SEQ_LEN]["y"]["Validation"]

assert X_train.shape == (34121, 168, 8)
assert y_train.shape == (34121, 2)

assert X_val.shape == (8760, 168, 8)
assert y_val.shape == (8760, 2)

print("\nDataset verified:")
print("Train     :", X_train.shape)
print("Validation:", X_val.shape)
print("Seeds     :", SEEDS)
print("Expected training runs: 20")


# ------------------------------------------------------------
# 3. REPRODUCIBILITY
# ------------------------------------------------------------
def reset_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


# ------------------------------------------------------------
# 4. POSITIONAL ENCODING
# ------------------------------------------------------------
class SinusoidalPositionalEncoding(layers.Layer):

    def __init__(self, d_model, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model

    def call(self, inputs):

        seq_len = tf.shape(inputs)[1]

        position = tf.cast(
            tf.range(seq_len)[:, tf.newaxis],
            tf.float32
        )

        div_term = tf.exp(
            tf.cast(
                tf.range(0, self.d_model, 2),
                tf.float32
            )
            *
            (-np.log(10000.0) / self.d_model)
        )

        sin_part = tf.sin(position * div_term)
        cos_part = tf.cos(position * div_term)

        pos_encoding = tf.reshape(
            tf.stack(
                [sin_part, cos_part],
                axis=-1
            ),
            (seq_len, self.d_model)
        )

        return (
            inputs +
            pos_encoding[tf.newaxis, :, :]
        )


# ------------------------------------------------------------
# 5. TRANSFORMER BLOCK
# ------------------------------------------------------------
def transformer_block(x, block_id):

    attn_output = layers.MultiHeadAttention(
        num_heads=NUM_HEADS,
        key_dim=D_MODEL // NUM_HEADS,
        dropout=DROPOUT,
        name=f"mha_{block_id}"
    )(x, x)

    x = layers.Add(
        name=f"attn_residual_{block_id}"
    )([x, attn_output])

    x = layers.LayerNormalization(
        epsilon=1e-6,
        name=f"attn_norm_{block_id}"
    )(x)

    ff = layers.Dense(
        FF_DIM,
        activation="gelu",
        name=f"ff1_{block_id}"
    )(x)

    ff = layers.Dropout(
        DROPOUT,
        name=f"ff_dropout_{block_id}"
    )(ff)

    ff = layers.Dense(
        D_MODEL,
        name=f"ff2_{block_id}"
    )(ff)

    x = layers.Add(
        name=f"ff_residual_{block_id}"
    )([x, ff])

    x = layers.LayerNormalization(
        epsilon=1e-6,
        name=f"ff_norm_{block_id}"
    )(x)

    return x


# ------------------------------------------------------------
# 6. BUILD LSTM
# ------------------------------------------------------------
def build_lstm():

    inputs = keras.Input(
        shape=(SEQ_LEN, 8),
        name="input_sequence"
    )

    x = layers.LSTM(
        64,
        return_sequences=True,
        name="lstm_1"
    )(inputs)

    x = layers.Dropout(
        0.20,
        name="dropout_1"
    )(x)

    x = layers.LSTM(
        32,
        name="lstm_2"
    )(x)

    x = layers.Dropout(
        0.20,
        name="dropout_2"
    )(x)

    x = layers.Dense(
        64,
        activation="gelu",
        name="shared_dense"
    )(x)

    x = layers.Dropout(
        0.20,
        name="shared_dropout"
    )(x)

    load_output = layers.Dense(
        1,
        name="load_output"
    )(x)

    solar_output = layers.Dense(
        1,
        name="solar_output"
    )(x)

    model = keras.Model(
        inputs,
        [load_output, solar_output],
        name="multioutput_lstm"
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss={
            "load_output": "mse",
            "solar_output": "mse"
        }
    )

    return model


# ------------------------------------------------------------
# 7. BUILD SINGLE-TASK TRANSFORMER
# ------------------------------------------------------------
def build_st_transformer(task_name):

    inputs = keras.Input(
        shape=(SEQ_LEN, 8),
        name="input_sequence"
    )

    x = layers.Dense(
        D_MODEL,
        name="input_projection"
    )(inputs)

    x = SinusoidalPositionalEncoding(
        D_MODEL,
        name="positional_encoding"
    )(x)

    for block_id in range(
        1,
        NUM_BLOCKS + 1
    ):
        x = transformer_block(
            x,
            block_id
        )

    x = layers.GlobalAveragePooling1D(
        name="global_average_pooling"
    )(x)

    x = layers.Dense(
        64,
        activation="gelu",
        name="task_dense"
    )(x)

    x = layers.Dropout(
        DROPOUT,
        name="task_dropout"
    )(x)

    output = layers.Dense(
        1,
        name=f"{task_name.lower()}_output"
    )(x)

    model = keras.Model(
        inputs,
        output,
        name=f"st_{task_name.lower()}_transformer"
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="mse"
    )

    return model


# ------------------------------------------------------------
# 8. BUILD MULTI-TASK TRANSFORMER
# ------------------------------------------------------------
def build_mtl_transformer():

    inputs = keras.Input(
        shape=(SEQ_LEN, 8),
        name="input_sequence"
    )

    x = layers.Dense(
        D_MODEL,
        name="input_projection"
    )(inputs)

    x = SinusoidalPositionalEncoding(
        D_MODEL,
        name="positional_encoding"
    )(x)

    for block_id in range(
        1,
        NUM_BLOCKS + 1
    ):
        x = transformer_block(
            x,
            block_id
        )

    x = layers.GlobalAveragePooling1D(
        name="global_average_pooling"
    )(x)

    x = layers.Dense(
        64,
        activation="gelu",
        name="shared_dense"
    )(x)

    x = layers.Dropout(
        DROPOUT,
        name="shared_dropout"
    )(x)

    load_output = layers.Dense(
        1,
        name="load_output"
    )(x)

    solar_output = layers.Dense(
        1,
        name="solar_output"
    )(x)

    model = keras.Model(
        inputs,
        [load_output, solar_output],
        name="mtl_transformer"
    )

    model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss={
            "load_output": "mse",
            "solar_output": "mse"
        }
    )

    return model


# ------------------------------------------------------------
# 9. METRIC HELPER
# ------------------------------------------------------------
def compute_metric(
    y_true_scaled,
    pred_scaled,
    target_index
):

    mean = target_scaler.mean_[
        target_index
    ]

    scale = target_scaler.scale_[
        target_index
    ]

    y_true = (
        y_true_scaled * scale + mean
    )

    y_pred = (
        pred_scaled * scale + mean
    )

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    return mae, rmse, r2


# ------------------------------------------------------------
# 10. SAVE AFTER EVERY COMPLETED RUN
# ------------------------------------------------------------
all_rows = []


def checkpoint_results():

    pd.DataFrame(
        all_rows
    ).to_csv(
        RESULT_FILE,
        index=False
    )


# ------------------------------------------------------------
# 11. MAIN FIVE-SEED EXPERIMENT
# ------------------------------------------------------------
for seed in SEEDS:

    print("\n" + "#" * 78)
    print(f"SEED {seed}")
    print("#" * 78)

    # ========================================================
    # A. LSTM
    # ========================================================
    print("\n[A] Multi-output LSTM")

    tf.keras.backend.clear_session()
    gc.collect()
    reset_seed(seed)

    model = build_lstm()

    params = model.count_params()

    callback = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=0
    )

    start = time.perf_counter()

    history = model.fit(
        X_train,
        {
            "load_output":
                y_train[:, 0],
            "solar_output":
                y_train[:, 1]
        },
        validation_data=(
            X_val,
            {
                "load_output":
                    y_val[:, 0],
                "solar_output":
                    y_val[:, 1]
            }
        ),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[callback],
        shuffle=True,
        verbose=0
    )

    elapsed = time.perf_counter() - start

    pred = model.predict(
        X_val,
        batch_size=BATCH_SIZE,
        verbose=0
    )

    best_epoch = (
        np.argmin(
            history.history["val_loss"]
        ) + 1
    )

    best_loss = np.min(
        history.history["val_loss"]
    )

    for idx, target in enumerate(
        ["Load", "Solar"]
    ):

        p = pred[idx].reshape(-1)

        mae, rmse, r2 = compute_metric(
            y_val[:, idx],
            p,
            idx
        )

        all_rows.append({
            "Seed": seed,
            "Model": "LSTM",
            "Target": target,
            "Parameters": params,
            "Best_epoch": best_epoch,
            "Best_val_loss": best_loss,
            "Training_time_s": elapsed,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })

    checkpoint_results()

    print(
        f"Completed | params={params:,} | "
        f"epoch={best_epoch} | "
        f"time={elapsed:.1f}s"
    )

    del model
    gc.collect()


    # ========================================================
    # B. SINGLE-TASK TRANSFORMERS
    # ========================================================
    for target_index, target in enumerate(
        ["Load", "Solar"]
    ):

        print(
            f"\n[B] Single-task Transformer — "
            f"{target}"
        )

        tf.keras.backend.clear_session()
        gc.collect()
        reset_seed(seed)

        model = build_st_transformer(
            target
        )

        params = model.count_params()

        callback = keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=0
        )

        start = time.perf_counter()

        history = model.fit(
            X_train,
            y_train[:, target_index],
            validation_data=(
                X_val,
                y_val[:, target_index]
            ),
            epochs=MAX_EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=[callback],
            shuffle=True,
            verbose=0
        )

        elapsed = (
            time.perf_counter() - start
        )

        pred = model.predict(
            X_val,
            batch_size=BATCH_SIZE,
            verbose=0
        ).reshape(-1)

        best_epoch = (
            np.argmin(
                history.history["val_loss"]
            ) + 1
        )

        best_loss = np.min(
            history.history["val_loss"]
        )

        mae, rmse, r2 = compute_metric(
            y_val[:, target_index],
            pred,
            target_index
        )

        all_rows.append({
            "Seed": seed,
            "Model": "ST-Transformer",
            "Target": target,
            "Parameters": params,
            "Best_epoch": best_epoch,
            "Best_val_loss": best_loss,
            "Training_time_s": elapsed,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })

        checkpoint_results()

        print(
            f"Completed | "
            f"RMSE={rmse:.3f} | "
            f"epoch={best_epoch} | "
            f"time={elapsed:.1f}s"
        )

        del model
        gc.collect()


    # ========================================================
    # C. MULTI-TASK TRANSFORMER
    # ========================================================
    print("\n[C] Multi-task Transformer")

    tf.keras.backend.clear_session()
    gc.collect()
    reset_seed(seed)

    model = build_mtl_transformer()

    params = model.count_params()

    callback = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,
        verbose=0
    )

    start = time.perf_counter()

    history = model.fit(
        X_train,
        {
            "load_output":
                y_train[:, 0],
            "solar_output":
                y_train[:, 1]
        },
        validation_data=(
            X_val,
            {
                "load_output":
                    y_val[:, 0],
                "solar_output":
                    y_val[:, 1]
            }
        ),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[callback],
        shuffle=True,
        verbose=0
    )

    elapsed = (
        time.perf_counter() - start
    )

    pred = model.predict(
        X_val,
        batch_size=BATCH_SIZE,
        verbose=0
    )

    best_epoch = (
        np.argmin(
            history.history["val_loss"]
        ) + 1
    )

    best_loss = np.min(
        history.history["val_loss"]
    )

    for idx, target in enumerate(
        ["Load", "Solar"]
    ):

        p = pred[idx].reshape(-1)

        mae, rmse, r2 = compute_metric(
            y_val[:, idx],
            p,
            idx
        )

        all_rows.append({
            "Seed": seed,
            "Model": "MTL-Transformer",
            "Target": target,
            "Parameters": params,
            "Best_epoch": best_epoch,
            "Best_val_loss": best_loss,
            "Training_time_s": elapsed,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })

    checkpoint_results()

    print(
        f"Completed | params={params:,} | "
        f"epoch={best_epoch} | "
        f"time={elapsed:.1f}s"
    )

    del model
    gc.collect()


# ------------------------------------------------------------
# 12. FULL RESULTS
# ------------------------------------------------------------
five_seed_results = pd.DataFrame(
    all_rows
)

print("\n" + "=" * 78)
print("ALL FIVE-SEED VALIDATION RESULTS")
print("=" * 78)

print(
    five_seed_results[
        [
            "Seed",
            "Model",
            "Target",
            "MAE",
            "RMSE",
            "R2",
            "Parameters",
            "Best_epoch",
            "Training_time_s"
        ]
    ].to_string(
        index=False,
        formatters={
            "MAE":
                lambda x: f"{x:.3f}",
            "RMSE":
                lambda x: f"{x:.3f}",
            "R2":
                lambda x: f"{x:.6f}",
            "Training_time_s":
                lambda x: f"{x:.2f}"
        }
    )
)


# ------------------------------------------------------------
# 13. MEAN ± SD SUMMARY
# ------------------------------------------------------------
five_seed_summary = (

    five_seed_results

    .groupby(
        ["Model", "Target"]
    )

    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),

        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),

        R2_mean=("R2", "mean"),
        R2_std=("R2", "std"),

        Time_mean=("Training_time_s", "mean"),
        Time_std=("Training_time_s", "std"),

        Parameters=("Parameters", "first")
    )

    .reset_index()
)


five_seed_summary.to_csv(
    SUMMARY_FILE,
    index=False
)


print("\n" + "=" * 78)
print("FIVE-SEED SUMMARY — MEAN ± SD")
print("=" * 78)

print(
    five_seed_summary.to_string(
        index=False,
        formatters={
            "MAE_mean":
                lambda x: f"{x:.3f}",
            "MAE_std":
                lambda x: f"{x:.3f}",
            "RMSE_mean":
                lambda x: f"{x:.3f}",
            "RMSE_std":
                lambda x: f"{x:.3f}",
            "R2_mean":
                lambda x: f"{x:.6f}",
            "R2_std":
                lambda x: f"{x:.6f}",
            "Time_mean":
                lambda x: f"{x:.2f}",
            "Time_std":
                lambda x: f"{x:.2f}"
        }
    )
)


# ------------------------------------------------------------
# 14. PAIRED MTL vs ST RMSE DIFFERENCES
# ------------------------------------------------------------
print("\n" + "=" * 78)
print("PAIRED MTL vs SINGLE-TASK TRANSFORMER")
print("=" * 78)

paired_rows = []

for target in [
    "Load",
    "Solar"
]:

    st = (
        five_seed_results[
            (
                five_seed_results["Model"]
                == "ST-Transformer"
            )
            &
            (
                five_seed_results["Target"]
                == target
            )
        ]
        .set_index("Seed")["RMSE"]
    )

    mtl = (
        five_seed_results[
            (
                five_seed_results["Model"]
                == "MTL-Transformer"
            )
            &
            (
                five_seed_results["Target"]
                == target
            )
        ]
        .set_index("Seed")["RMSE"]
    )

    common_seeds = st.index.intersection(
        mtl.index
    )

    for seed in common_seeds:

        improvement = (
            (
                st.loc[seed] -
                mtl.loc[seed]
            )
            /
            st.loc[seed]
            * 100
        )

        paired_rows.append({
            "Target": target,
            "Seed": seed,
            "ST_RMSE": st.loc[seed],
            "MTL_RMSE": mtl.loc[seed],
            "MTL_improvement_pct":
                improvement
        })


paired_results = pd.DataFrame(
    paired_rows
)


print(
    paired_results.to_string(
        index=False,
        formatters={
            "ST_RMSE":
                lambda x: f"{x:.3f}",
            "MTL_RMSE":
                lambda x: f"{x:.3f}",
            "MTL_improvement_pct":
                lambda x: f"{x:.2f}%"
        }
    )
)


paired_results.to_csv(
    os.path.join(
        SAVE_DIR,
        "paired_mtl_vs_st_validation.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# 15. AVERAGE MTL IMPROVEMENT
# ------------------------------------------------------------
print("\n" + "-" * 78)
print("AVERAGE MTL IMPROVEMENT OVER SINGLE-TASK")
print("-" * 78)

for target in [
    "Load",
    "Solar"
]:

    temp = paired_results[
        paired_results["Target"]
        == target
    ]

    print(
        f"{target:<6}: "
        f"{temp['MTL_improvement_pct'].mean():.2f}% "
        f"± "
        f"{temp['MTL_improvement_pct'].std():.2f}%"
    )


print("\nSaved:")
print(" ✓ five_seed_validation_runs.csv")
print(" ✓ five_seed_validation_summary.csv")
print(" ✓ paired_mtl_vs_st_validation.csv")

print(
    "\n✓ TEST SET HAS NOT BEEN EVALUATED."
)

print("\n" + "=" * 78)
print("STEP 4.4 COMPLETE")
print("=" * 78)

STEP 4.4 — FIVE-SEED REPRODUCIBILITY EXPERIMENT

Dataset verified:
Train     : (34121, 168, 8)
Validation: (8760, 168, 8)
Seeds     : [1, 7, 21, 42, 100]
Expected training runs: 20

##############################################################################
SEED 1
##############################################################################

[A] Multi-output LSTM
Completed | params=33,346 | epoch=26 | time=312.6s

[B] Single-task Transformer — Load
Completed | RMSE=640.993 | epoch=10 | time=79.9s

[B] Single-task Transformer — Solar
Completed | RMSE=425.276 | epoch=8 | time=75.7s

[C] Multi-task Transformer
Completed | params=71,810 | epoch=11 | time=89.9s

##############################################################################
SEED 7
##############################################################################

[A] Multi-output LSTM
Completed | params=33,346 | epoch=19 | time=246.4s

[B] Single-task Transformer — Load
Completed | RMSE=838.325 | epoch=7 | time=74.4s

[B] Si

Step 4.5: five-seed weighted-loss MTL experiment for (2:1) and (1:2).

In [7]:
# ============================================================
# STEP 4.5 — MULTI-TASK LOSS-WEIGHT SENSITIVITY
#
# Existing condition:
#     Load : Solar = 1 : 1   [from Step 4.4]
#
# New conditions:
#     Load : Solar = 2 : 1
#     Load : Solar = 1 : 2
#
# Five seeds per new condition.
# Validation ONLY. Test remains untouched.
# ============================================================

import os
import gc
import time
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("=" * 78)
print("STEP 4.5 — MULTI-TASK LOSS-WEIGHT SENSITIVITY")
print("=" * 78)


# ------------------------------------------------------------
# 1. FROZEN CONFIGURATION
# ------------------------------------------------------------
SEEDS = [1, 7, 21, 42, 100]

SEQ_LEN = 168

D_MODEL = 64
NUM_HEADS = 4
FF_DIM = 128
NUM_BLOCKS = 2
DROPOUT = 0.20

LEARNING_RATE = 1e-3
BATCH_SIZE = 64
MAX_EPOCHS = 50
PATIENCE = 7

SAVE_DIR = (
    "/content/drive/MyDrive/"
    "SolarLoad_MTL_Study"
)

NEW_RESULTS_FILE = os.path.join(
    SAVE_DIR,
    "weighted_mtl_validation_runs.csv"
)

COMBINED_FILE = os.path.join(
    SAVE_DIR,
    "mtl_loss_weight_all_runs.csv"
)

SUMMARY_FILE = os.path.join(
    SAVE_DIR,
    "mtl_loss_weight_summary.csv"
)


# Only NEW conditions.
# 1:1 already exists from Step 4.4.
WEIGHT_CONFIGS = {

    "2:1": {
        "load_output": 2.0,
        "solar_output": 1.0
    },

    "1:2": {
        "load_output": 1.0,
        "solar_output": 2.0
    }
}


# ------------------------------------------------------------
# 2. FROZEN DATA
# ------------------------------------------------------------
X_train = sequence_results[
    SEQ_LEN
]["X"]["Train"]

y_train = sequence_results[
    SEQ_LEN
]["y"]["Train"]

X_val = sequence_results[
    SEQ_LEN
]["X"]["Validation"]

y_val = sequence_results[
    SEQ_LEN
]["y"]["Validation"]


assert X_train.shape == (34121, 168, 8)
assert y_train.shape == (34121, 2)

assert X_val.shape == (8760, 168, 8)
assert y_val.shape == (8760, 2)


print("\nDataset verified:")
print("Train     :", X_train.shape)
print("Validation:", X_val.shape)

print("\nSeeds:", SEEDS)

print(
    "New weighting conditions:",
    list(WEIGHT_CONFIGS.keys())
)

print("Expected new training runs: 10")


# ------------------------------------------------------------
# 3. VERIFY EXISTING STEP 4.4 RESULTS
# ------------------------------------------------------------
STEP44_FILE = os.path.join(
    SAVE_DIR,
    "five_seed_validation_runs.csv"
)

if not os.path.exists(STEP44_FILE):

    raise FileNotFoundError(
        "Step 4.4 results were not found:\n"
        + STEP44_FILE
    )


step44 = pd.read_csv(
    STEP44_FILE
)


equal_mtl = step44[
    step44["Model"]
    == "MTL-Transformer"
].copy()


assert len(equal_mtl) == 10

assert sorted(
    equal_mtl["Seed"].unique().tolist()
) == SEEDS


print(
    "\n✓ Existing five-seed 1:1 MTL "
    "results verified."
)


# ------------------------------------------------------------
# 4. REPRODUCIBILITY
# ------------------------------------------------------------
def reset_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


# ------------------------------------------------------------
# 5. POSITIONAL ENCODING
# ------------------------------------------------------------
class SinusoidalPositionalEncoding(
    layers.Layer
):

    def __init__(
        self,
        d_model,
        **kwargs
    ):

        super().__init__(**kwargs)

        self.d_model = d_model


    def call(self, inputs):

        seq_len = tf.shape(inputs)[1]

        position = tf.cast(
            tf.range(seq_len)[
                :, tf.newaxis
            ],
            tf.float32
        )

        div_term = tf.exp(

            tf.cast(
                tf.range(
                    0,
                    self.d_model,
                    2
                ),
                tf.float32
            )

            *

            (
                -np.log(10000.0)
                /
                self.d_model
            )
        )

        sin_part = tf.sin(
            position * div_term
        )

        cos_part = tf.cos(
            position * div_term
        )

        pos_encoding = tf.reshape(

            tf.stack(
                [
                    sin_part,
                    cos_part
                ],
                axis=-1
            ),

            (
                seq_len,
                self.d_model
            )
        )

        return (
            inputs
            +
            pos_encoding[
                tf.newaxis,
                :,
                :
            ]
        )


# ------------------------------------------------------------
# 6. TRANSFORMER BLOCK
# ------------------------------------------------------------
def transformer_block(
    x,
    block_id
):

    attn_output = (
        layers.MultiHeadAttention(

            num_heads=NUM_HEADS,

            key_dim=(
                D_MODEL
                //
                NUM_HEADS
            ),

            dropout=DROPOUT,

            name=f"mha_{block_id}"

        )(x, x)
    )

    x = layers.Add(
        name=f"attn_residual_{block_id}"
    )(
        [
            x,
            attn_output
        ]
    )

    x = layers.LayerNormalization(
        epsilon=1e-6,
        name=f"attn_norm_{block_id}"
    )(x)


    ff = layers.Dense(
        FF_DIM,
        activation="gelu",
        name=f"ff1_{block_id}"
    )(x)

    ff = layers.Dropout(
        DROPOUT,
        name=f"ff_dropout_{block_id}"
    )(ff)

    ff = layers.Dense(
        D_MODEL,
        name=f"ff2_{block_id}"
    )(ff)


    x = layers.Add(
        name=f"ff_residual_{block_id}"
    )(
        [
            x,
            ff
        ]
    )

    x = layers.LayerNormalization(
        epsilon=1e-6,
        name=f"ff_norm_{block_id}"
    )(x)

    return x


# ------------------------------------------------------------
# 7. BUILD WEIGHTED MTL TRANSFORMER
# ------------------------------------------------------------
def build_weighted_mtl(
    load_weight,
    solar_weight
):

    inputs = keras.Input(
        shape=(SEQ_LEN, 8),
        name="input_sequence"
    )

    x = layers.Dense(
        D_MODEL,
        name="input_projection"
    )(inputs)

    x = SinusoidalPositionalEncoding(
        D_MODEL,
        name="positional_encoding"
    )(x)


    for block_id in range(
        1,
        NUM_BLOCKS + 1
    ):

        x = transformer_block(
            x,
            block_id
        )


    x = layers.GlobalAveragePooling1D(
        name="global_average_pooling"
    )(x)


    x = layers.Dense(
        64,
        activation="gelu",
        name="shared_dense"
    )(x)


    x = layers.Dropout(
        DROPOUT,
        name="shared_dropout"
    )(x)


    load_output = layers.Dense(
        1,
        name="load_output"
    )(x)


    solar_output = layers.Dense(
        1,
        name="solar_output"
    )(x)


    model = keras.Model(

        inputs=inputs,

        outputs=[
            load_output,
            solar_output
        ],

        name="weighted_mtl_transformer"
    )


    model.compile(

        optimizer=keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),

        loss={
            "load_output": "mse",
            "solar_output": "mse"
        },

        loss_weights={
            "load_output":
                load_weight,

            "solar_output":
                solar_weight
        }
    )

    return model


# ------------------------------------------------------------
# 8. METRIC FUNCTION
# ------------------------------------------------------------
def compute_metric(
    y_true_scaled,
    pred_scaled,
    target_index
):

    target_mean = (
        target_scaler.mean_[
            target_index
        ]
    )

    target_scale = (
        target_scaler.scale_[
            target_index
        ]
    )


    y_true = (
        y_true_scaled
        *
        target_scale
        +
        target_mean
    )


    y_pred = (
        pred_scaled
        *
        target_scale
        +
        target_mean
    )


    mae = mean_absolute_error(
        y_true,
        y_pred
    )


    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )


    r2 = r2_score(
        y_true,
        y_pred
    )


    return mae, rmse, r2


# ------------------------------------------------------------
# 9. NEW RUN STORAGE
# ------------------------------------------------------------
weighted_rows = []


def save_checkpoint():

    pd.DataFrame(
        weighted_rows
    ).to_csv(
        NEW_RESULTS_FILE,
        index=False
    )


# ------------------------------------------------------------
# 10. RUN 10 NEW EXPERIMENTS
# ------------------------------------------------------------
for weight_label, weights in (
    WEIGHT_CONFIGS.items()
):

    load_weight = (
        weights["load_output"]
    )

    solar_weight = (
        weights["solar_output"]
    )


    print("\n" + "#" * 78)

    print(
        f"LOSS WEIGHT CONDITION "
        f"{weight_label}"
    )

    print(
        f"Load weight={load_weight} | "
        f"Solar weight={solar_weight}"
    )

    print("#" * 78)


    for seed in SEEDS:

        print(
            f"\nTraining "
            f"{weight_label} — "
            f"Seed {seed}"
        )


        tf.keras.backend.clear_session()

        gc.collect()

        reset_seed(seed)


        model = build_weighted_mtl(
            load_weight,
            solar_weight
        )


        params = (
            model.count_params()
        )


        callback = (
            keras.callbacks.EarlyStopping(

                monitor="val_loss",

                patience=PATIENCE,

                restore_best_weights=True,

                verbose=0
            )
        )


        start = time.perf_counter()


        history = model.fit(

            X_train,

            {
                "load_output":
                    y_train[:, 0],

                "solar_output":
                    y_train[:, 1]
            },

            validation_data=(

                X_val,

                {
                    "load_output":
                        y_val[:, 0],

                    "solar_output":
                        y_val[:, 1]
                }
            ),

            epochs=MAX_EPOCHS,

            batch_size=BATCH_SIZE,

            callbacks=[callback],

            shuffle=True,

            verbose=0
        )


        elapsed = (
            time.perf_counter()
            -
            start
        )


        best_epoch = int(

            np.argmin(
                history.history[
                    "val_loss"
                ]
            )

            + 1
        )


        best_val_loss = float(

            np.min(
                history.history[
                    "val_loss"
                ]
            )
        )


        predictions = model.predict(
            X_val,
            batch_size=BATCH_SIZE,
            verbose=0
        )


        for target_index, target in (
            enumerate(
                ["Load", "Solar"]
            )
        ):

            pred_scaled = (
                predictions[
                    target_index
                ].reshape(-1)
            )


            mae, rmse, r2 = (
                compute_metric(

                    y_val[
                        :,
                        target_index
                    ],

                    pred_scaled,

                    target_index
                )
            )


            weighted_rows.append({

                "Weighting":
                    weight_label,

                "Load_weight":
                    load_weight,

                "Solar_weight":
                    solar_weight,

                "Seed":
                    seed,

                "Model":
                    "MTL-Transformer",

                "Target":
                    target,

                "Parameters":
                    params,

                "Best_epoch":
                    best_epoch,

                "Best_val_loss":
                    best_val_loss,

                "Training_time_s":
                    elapsed,

                "MAE":
                    mae,

                "RMSE":
                    rmse,

                "R2":
                    r2
            })


        save_checkpoint()


        load_rmse = (
            weighted_rows[-2]["RMSE"]
        )

        solar_rmse = (
            weighted_rows[-1]["RMSE"]
        )


        print(
            f"Completed | "
            f"epoch={best_epoch} | "
            f"Load RMSE="
            f"{load_rmse:.3f} | "
            f"Solar RMSE="
            f"{solar_rmse:.3f} | "
            f"time={elapsed:.1f}s"
        )


        del model

        gc.collect()


# ------------------------------------------------------------
# 11. NEW RESULTS
# ------------------------------------------------------------
weighted_results = pd.DataFrame(
    weighted_rows
)


print("\n" + "=" * 78)
print("NEW WEIGHTED-MTL RESULTS")
print("=" * 78)


print(
    weighted_results[
        [
            "Weighting",
            "Seed",
            "Target",
            "MAE",
            "RMSE",
            "R2",
            "Best_epoch",
            "Training_time_s"
        ]
    ].to_string(

        index=False,

        formatters={

            "MAE":
                lambda x: f"{x:.3f}",

            "RMSE":
                lambda x: f"{x:.3f}",

            "R2":
                lambda x: f"{x:.6f}",

            "Training_time_s":
                lambda x: f"{x:.2f}"
        }
    )
)


# ------------------------------------------------------------
# 12. CONVERT EXISTING 1:1 RESULTS TO SAME FORMAT
# ------------------------------------------------------------
equal_formatted = (
    equal_mtl.copy()
)


equal_formatted["Weighting"] = "1:1"

equal_formatted["Load_weight"] = 1.0

equal_formatted["Solar_weight"] = 1.0


required_columns = [

    "Weighting",
    "Load_weight",
    "Solar_weight",
    "Seed",
    "Model",
    "Target",
    "Parameters",
    "Best_epoch",
    "Best_val_loss",
    "Training_time_s",
    "MAE",
    "RMSE",
    "R2"
]


equal_formatted = (
    equal_formatted[
        required_columns
    ]
)


weighted_results = (
    weighted_results[
        required_columns
    ]
)


# ------------------------------------------------------------
# 13. COMBINE 1:1 + 2:1 + 1:2
# ------------------------------------------------------------
all_weight_results = pd.concat(

    [
        equal_formatted,
        weighted_results
    ],

    ignore_index=True
)


all_weight_results.to_csv(
    COMBINED_FILE,
    index=False
)


# ------------------------------------------------------------
# 14. MEAN ± SD
# ------------------------------------------------------------
weight_summary = (

    all_weight_results

    .groupby(
        [
            "Weighting",
            "Target"
        ]
    )

    .agg(

        MAE_mean=(
            "MAE",
            "mean"
        ),

        MAE_std=(
            "MAE",
            "std"
        ),

        RMSE_mean=(
            "RMSE",
            "mean"
        ),

        RMSE_std=(
            "RMSE",
            "std"
        ),

        R2_mean=(
            "R2",
            "mean"
        ),

        R2_std=(
            "R2",
            "std"
        ),

        Time_mean=(
            "Training_time_s",
            "mean"
        ),

        Time_std=(
            "Training_time_s",
            "std"
        )
    )

    .reset_index()
)


weight_summary.to_csv(
    SUMMARY_FILE,
    index=False
)


print("\n" + "=" * 78)
print("LOSS-WEIGHT SUMMARY — MEAN ± SD")
print("=" * 78)


print(
    weight_summary.to_string(

        index=False,

        formatters={

            "MAE_mean":
                lambda x: f"{x:.3f}",

            "MAE_std":
                lambda x: f"{x:.3f}",

            "RMSE_mean":
                lambda x: f"{x:.3f}",

            "RMSE_std":
                lambda x: f"{x:.3f}",

            "R2_mean":
                lambda x: f"{x:.6f}",

            "R2_std":
                lambda x: f"{x:.6f}",

            "Time_mean":
                lambda x: f"{x:.2f}",

            "Time_std":
                lambda x: f"{x:.2f}"
        }
    )
)


# ------------------------------------------------------------
# 15. BALANCED NORMALIZED RMSE
#
# Normalize using training target scales.
# Lower is better.
# ------------------------------------------------------------
load_scale = (
    target_scaler.scale_[0]
)

solar_scale = (
    target_scaler.scale_[1]
)


balanced_rows = []


for weighting in [
    "1:1",
    "2:1",
    "1:2"
]:

    temp = all_weight_results[
        all_weight_results[
            "Weighting"
        ] == weighting
    ]


    for seed in SEEDS:

        seed_data = temp[
            temp["Seed"] == seed
        ]


        load_rmse = float(

            seed_data.loc[
                seed_data["Target"]
                == "Load",

                "RMSE"

            ].iloc[0]
        )


        solar_rmse = float(

            seed_data.loc[
                seed_data["Target"]
                == "Solar",

                "RMSE"

            ].iloc[0]
        )


        load_nrmse = (
            load_rmse /
            load_scale
        )


        solar_nrmse = (
            solar_rmse /
            solar_scale
        )


        balanced = (
            load_nrmse
            +
            solar_nrmse
        ) / 2.0


        balanced_rows.append({

            "Weighting":
                weighting,

            "Seed":
                seed,

            "Load_NRMSE":
                load_nrmse,

            "Solar_NRMSE":
                solar_nrmse,

            "Balanced_NRMSE":
                balanced
        })


balanced_results = pd.DataFrame(
    balanced_rows
)


balanced_summary = (

    balanced_results

    .groupby("Weighting")

    .agg(

        Balanced_NRMSE_mean=(
            "Balanced_NRMSE",
            "mean"
        ),

        Balanced_NRMSE_std=(
            "Balanced_NRMSE",
            "std"
        ),

        Load_NRMSE_mean=(
            "Load_NRMSE",
            "mean"
        ),

        Solar_NRMSE_mean=(
            "Solar_NRMSE",
            "mean"
        )
    )

    .reset_index()
)


print("\n" + "=" * 78)
print("BALANCED NORMALIZED RMSE")
print("=" * 78)


print(
    balanced_summary.to_string(

        index=False,

        formatters={

            "Balanced_NRMSE_mean":
                lambda x: f"{x:.6f}",

            "Balanced_NRMSE_std":
                lambda x: f"{x:.6f}",

            "Load_NRMSE_mean":
                lambda x: f"{x:.6f}",

            "Solar_NRMSE_mean":
                lambda x: f"{x:.6f}"
        }
    )
)


balanced_results.to_csv(

    os.path.join(
        SAVE_DIR,
        "mtl_loss_weight_balanced_runs.csv"
    ),

    index=False
)


balanced_summary.to_csv(

    os.path.join(
        SAVE_DIR,
        "mtl_loss_weight_balanced_summary.csv"
    ),

    index=False
)


# ------------------------------------------------------------
# 16. AUTOMATIC VALIDATION-ONLY SELECTION
# ------------------------------------------------------------
best_row = (

    balanced_summary

    .sort_values(
        "Balanced_NRMSE_mean"
    )

    .iloc[0]
)


best_weighting = (
    best_row["Weighting"]
)


print("\n" + "=" * 78)
print("VALIDATION-SELECTED LOSS WEIGHTING")
print("=" * 78)

print(
    "Lowest mean balanced NRMSE:",
    best_weighting
)

print(
    "Balanced NRMSE mean:",
    f"{best_row['Balanced_NRMSE_mean']:.6f}"
)

print(
    "Balanced NRMSE SD  :",
    f"{best_row['Balanced_NRMSE_std']:.6f}"
)


print("\nSaved:")
print(" ✓ weighted_mtl_validation_runs.csv")
print(" ✓ mtl_loss_weight_all_runs.csv")
print(" ✓ mtl_loss_weight_summary.csv")
print(" ✓ mtl_loss_weight_balanced_runs.csv")
print(" ✓ mtl_loss_weight_balanced_summary.csv")


print(
    "\n✓ No test-set predictions were generated."
)

print(
    "✓ Architecture and preprocessing "
    "were unchanged."
)

print("\n" + "=" * 78)
print("STEP 4.5 COMPLETE")
print("=" * 78)

STEP 4.5 — MULTI-TASK LOSS-WEIGHT SENSITIVITY

Dataset verified:
Train     : (34121, 168, 8)
Validation: (8760, 168, 8)

Seeds: [1, 7, 21, 42, 100]
New weighting conditions: ['2:1', '1:2']
Expected new training runs: 10

✓ Existing five-seed 1:1 MTL results verified.

##############################################################################
LOSS WEIGHT CONDITION 2:1
Load weight=2.0 | Solar weight=1.0
##############################################################################

Training 2:1 — Seed 1
Completed | epoch=27 | Load RMSE=667.358 | Solar RMSE=379.708 | time=134.0s

Training 2:1 — Seed 7
Completed | epoch=17 | Load RMSE=751.553 | Solar RMSE=446.490 | time=107.5s

Training 2:1 — Seed 21
Completed | epoch=13 | Load RMSE=696.662 | Solar RMSE=465.787 | time=99.8s

Training 2:1 — Seed 42
Completed | epoch=7 | Load RMSE=1001.086 | Solar RMSE=498.909 | time=80.4s

Training 2:1 — Seed 100
Completed | epoch=20 | Load RMSE=670.794 | Solar RMSE=514.148 | time=119.9s

##############

Step 4.6 — Final locked-model test evaluation

In [8]:
# ============================================================
# STEP 4.6 — LOCKED FIVE-SEED FINAL TEST EVALUATION
#
# MODEL SELECTION IS CLOSED.
#
# Frozen:
#   Context             = 168 h
#   Forecast horizon    = 1 h
#   Seeds               = [1, 7, 21, 42, 100]
#   Transformer         = d_model 64, heads 4,
#                         FF 128, 2 blocks
#   MTL weights         = Load:Solar = 2:1
#
# Train: 2015–2018
# Validation: 2019 (early stopping only)
# Final test: 2020-01-01 to 2020-09-30
#
# NO MODEL SELECTION FROM TEST RESULTS.
# ============================================================

import os
import gc
import time
import random
import json

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("=" * 80)
print("STEP 4.6 — LOCKED FIVE-SEED FINAL TEST EVALUATION")
print("=" * 80)

# ============================================================
# 1. LOCKED CONFIGURATION
# ============================================================

SEEDS = [1, 7, 21, 42, 100]

SEQ_LEN = 168

D_MODEL = 64
NUM_HEADS = 4
FF_DIM = 128
NUM_BLOCKS = 2
DROPOUT = 0.20

LEARNING_RATE = 1e-3
BATCH_SIZE = 64
MAX_EPOCHS = 50
PATIENCE = 7

MTL_LOAD_WEIGHT = 2.0
MTL_SOLAR_WEIGHT = 1.0

SAVE_DIR = (
    "/content/drive/MyDrive/"
    "SolarLoad_MTL_Study"
)

RUN_FILE = os.path.join(
    SAVE_DIR,
    "final_test_five_seed_runs.csv"
)

SUMMARY_FILE = os.path.join(
    SAVE_DIR,
    "final_test_five_seed_summary.csv"
)

PREDICTION_FILE = os.path.join(
    SAVE_DIR,
    "final_test_predictions.csv"
)

EFFICIENCY_FILE = os.path.join(
    SAVE_DIR,
    "final_model_efficiency.csv"
)

FREEZE_FILE = os.path.join(
    SAVE_DIR,
    "final_model_selection_freeze.json"
)


# ============================================================
# 2. LOAD FROZEN DATA
# ============================================================

X_train = sequence_results[
    SEQ_LEN
]["X"]["Train"]

y_train = sequence_results[
    SEQ_LEN
]["y"]["Train"]

X_val = sequence_results[
    SEQ_LEN
]["X"]["Validation"]

y_val = sequence_results[
    SEQ_LEN
]["y"]["Validation"]

X_test = sequence_results[
    SEQ_LEN
]["X"]["Test"]

y_test = sequence_results[
    SEQ_LEN
]["y"]["Test"]

test_times = sequence_results[
    SEQ_LEN
]["times"]["Test"]


assert X_train.shape == (34121, 168, 8)
assert y_train.shape == (34121, 2)

assert X_val.shape == (8760, 168, 8)
assert y_val.shape == (8760, 2)

assert X_test.shape == (6576, 168, 8)
assert y_test.shape == (6576, 2)

assert len(test_times) == 6576


print("\nLOCKED DATA:")
print("Train      :", X_train.shape)
print("Validation :", X_val.shape)
print("Test       :", X_test.shape)

print(
    "Test period:",
    test_times[0],
    "→",
    test_times[-1]
)


# ============================================================
# 3. RECORD MODEL-SELECTION FREEZE
# ============================================================

freeze_record = {

    "selection_status":
        "LOCKED_BEFORE_FINAL_TEST",

    "context_hours":
        168,

    "forecast_horizon_hours":
        1,

    "seeds":
        SEEDS,

    "transformer": {
        "d_model": D_MODEL,
        "num_heads": NUM_HEADS,
        "ff_dim": FF_DIM,
        "num_blocks": NUM_BLOCKS,
        "dropout": DROPOUT
    },

    "mtl_loss_weights": {
        "load": MTL_LOAD_WEIGHT,
        "solar": MTL_SOLAR_WEIGHT
    },

    "training_period":
        "2015-01-01 to 2018-12-31",

    "validation_period":
        "2019-01-01 to 2019-12-31",

    "test_period":
        "2020-01-01 to 2020-09-30",

    "rule":
        (
            "No model, architecture, context, loss-weight, "
            "or seed selection may be changed using test results."
        )
}


with open(
    FREEZE_FILE,
    "w"
) as f:

    json.dump(
        freeze_record,
        f,
        indent=2
    )


print(
    "\n✓ Model-selection freeze written "
    "BEFORE test prediction."
)


# ============================================================
# 4. REPRODUCIBILITY
# ============================================================

def reset_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


# ============================================================
# 5. POSITIONAL ENCODING
# ============================================================

class SinusoidalPositionalEncoding(layers.Layer):

    def __init__(
        self,
        d_model,
        **kwargs
    ):

        super().__init__(**kwargs)

        self.d_model = d_model


    def call(self, inputs):

        seq_len = tf.shape(inputs)[1]

        position = tf.cast(
            tf.range(seq_len)[:, tf.newaxis],
            tf.float32
        )

        div_term = tf.exp(

            tf.cast(
                tf.range(
                    0,
                    self.d_model,
                    2
                ),
                tf.float32
            )

            *

            (
                -np.log(10000.0)
                /
                self.d_model
            )
        )

        sin_part = tf.sin(
            position * div_term
        )

        cos_part = tf.cos(
            position * div_term
        )

        pos_encoding = tf.reshape(

            tf.stack(
                [sin_part, cos_part],
                axis=-1
            ),

            (
                seq_len,
                self.d_model
            )
        )

        return (
            inputs
            +
            pos_encoding[
                tf.newaxis, :, :
            ]
        )


# ============================================================
# 6. TRANSFORMER BLOCK
# ============================================================

def transformer_block(
    x,
    block_id
):

    attn_output = (
        layers.MultiHeadAttention(

            num_heads=NUM_HEADS,

            key_dim=(
                D_MODEL //
                NUM_HEADS
            ),

            dropout=DROPOUT,

            name=f"mha_{block_id}"

        )(x, x)
    )

    x = layers.Add(
        name=f"attn_residual_{block_id}"
    )(
        [x, attn_output]
    )

    x = layers.LayerNormalization(
        epsilon=1e-6,
        name=f"attn_norm_{block_id}"
    )(x)


    ff = layers.Dense(
        FF_DIM,
        activation="gelu",
        name=f"ff1_{block_id}"
    )(x)

    ff = layers.Dropout(
        DROPOUT,
        name=f"ff_dropout_{block_id}"
    )(ff)

    ff = layers.Dense(
        D_MODEL,
        name=f"ff2_{block_id}"
    )(ff)


    x = layers.Add(
        name=f"ff_residual_{block_id}"
    )(
        [x, ff]
    )

    x = layers.LayerNormalization(
        epsilon=1e-6,
        name=f"ff_norm_{block_id}"
    )(x)

    return x


# ============================================================
# 7. LSTM
# ============================================================

def build_lstm():

    inputs = keras.Input(
        shape=(SEQ_LEN, 8),
        name="input_sequence"
    )

    x = layers.LSTM(
        64,
        return_sequences=True,
        name="lstm_1"
    )(inputs)

    x = layers.Dropout(
        0.20,
        name="dropout_1"
    )(x)

    x = layers.LSTM(
        32,
        name="lstm_2"
    )(x)

    x = layers.Dropout(
        0.20,
        name="dropout_2"
    )(x)

    x = layers.Dense(
        64,
        activation="gelu",
        name="shared_dense"
    )(x)

    x = layers.Dropout(
        0.20,
        name="shared_dropout"
    )(x)

    load_output = layers.Dense(
        1,
        name="load_output"
    )(x)

    solar_output = layers.Dense(
        1,
        name="solar_output"
    )(x)

    model = keras.Model(
        inputs,
        [
            load_output,
            solar_output
        ],
        name="locked_lstm"
    )

    model.compile(

        optimizer=keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),

        loss={
            "load_output": "mse",
            "solar_output": "mse"
        }
    )

    return model


# ============================================================
# 8. SINGLE-TASK TRANSFORMER
# ============================================================

def build_st_transformer(task):

    inputs = keras.Input(
        shape=(SEQ_LEN, 8),
        name="input_sequence"
    )

    x = layers.Dense(
        D_MODEL,
        name="input_projection"
    )(inputs)

    x = SinusoidalPositionalEncoding(
        D_MODEL,
        name="positional_encoding"
    )(x)

    for block_id in range(
        1,
        NUM_BLOCKS + 1
    ):

        x = transformer_block(
            x,
            block_id
        )

    x = layers.GlobalAveragePooling1D(
        name="global_average_pooling"
    )(x)

    x = layers.Dense(
        64,
        activation="gelu",
        name="task_dense"
    )(x)

    x = layers.Dropout(
        DROPOUT,
        name="task_dropout"
    )(x)

    output = layers.Dense(
        1,
        name=f"{task.lower()}_output"
    )(x)

    model = keras.Model(
        inputs,
        output,
        name=f"locked_st_{task.lower()}"
    )

    model.compile(

        optimizer=keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),

        loss="mse"
    )

    return model


# ============================================================
# 9. WEIGHTED MTL TRANSFORMER
# ============================================================

def build_mtl_transformer():

    inputs = keras.Input(
        shape=(SEQ_LEN, 8),
        name="input_sequence"
    )

    x = layers.Dense(
        D_MODEL,
        name="input_projection"
    )(inputs)

    x = SinusoidalPositionalEncoding(
        D_MODEL,
        name="positional_encoding"
    )(x)

    for block_id in range(
        1,
        NUM_BLOCKS + 1
    ):

        x = transformer_block(
            x,
            block_id
        )

    x = layers.GlobalAveragePooling1D(
        name="global_average_pooling"
    )(x)

    x = layers.Dense(
        64,
        activation="gelu",
        name="shared_dense"
    )(x)

    x = layers.Dropout(
        DROPOUT,
        name="shared_dropout"
    )(x)

    load_output = layers.Dense(
        1,
        name="load_output"
    )(x)

    solar_output = layers.Dense(
        1,
        name="solar_output"
    )(x)

    model = keras.Model(
        inputs,
        [
            load_output,
            solar_output
        ],
        name="locked_weighted_mtl"
    )

    model.compile(

        optimizer=keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),

        loss={
            "load_output": "mse",
            "solar_output": "mse"
        },

        loss_weights={
            "load_output":
                MTL_LOAD_WEIGHT,

            "solar_output":
                MTL_SOLAR_WEIGHT
        }
    )

    return model


# ============================================================
# 10. METRIC / INVERSE-SCALING HELPERS
# ============================================================

def inverse_target(
    values,
    target_index
):

    return (
        values
        *
        target_scaler.scale_[target_index]
        +
        target_scaler.mean_[target_index]
    )


def calculate_metrics(
    true_scaled,
    pred_scaled,
    target_index
):

    true = inverse_target(
        true_scaled,
        target_index
    )

    pred = inverse_target(
        pred_scaled,
        target_index
    )

    mae = mean_absolute_error(
        true,
        pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            true,
            pred
        )
    )

    r2 = r2_score(
        true,
        pred
    )

    return (
        float(mae),
        float(rmse),
        float(r2),
        true,
        pred
    )


# ============================================================
# 11. RESULT STORAGE
# ============================================================

run_rows = []
prediction_frames = []


def checkpoint():

    pd.DataFrame(
        run_rows
    ).to_csv(
        RUN_FILE,
        index=False
    )

    if prediction_frames:

        pd.concat(
            prediction_frames,
            ignore_index=True
        ).to_csv(
            PREDICTION_FILE,
            index=False
        )


# ============================================================
# 12. TRAIN AND TEST LOCKED MODELS
# ============================================================

for seed in SEEDS:

    print("\n" + "#" * 80)
    print(f"FINAL LOCKED EVALUATION — SEED {seed}")
    print("#" * 80)


    # ========================================================
    # A. LSTM
    # ========================================================

    print("\n[A] LSTM")

    tf.keras.backend.clear_session()
    gc.collect()
    reset_seed(seed)

    model = build_lstm()

    params = model.count_params()

    early_stop = (
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=0
        )
    )

    start = time.perf_counter()

    history = model.fit(

        X_train,

        {
            "load_output":
                y_train[:, 0],

            "solar_output":
                y_train[:, 1]
        },

        validation_data=(

            X_val,

            {
                "load_output":
                    y_val[:, 0],

                "solar_output":
                    y_val[:, 1]
            }
        ),

        epochs=MAX_EPOCHS,

        batch_size=BATCH_SIZE,

        callbacks=[early_stop],

        shuffle=True,

        verbose=0
    )

    train_time = (
        time.perf_counter() - start
    )

    best_epoch = int(
        np.argmin(
            history.history["val_loss"]
        ) + 1
    )

    start_pred = time.perf_counter()

    predictions = model.predict(
        X_test,
        batch_size=BATCH_SIZE,
        verbose=0
    )

    inference_time = (
        time.perf_counter() -
        start_pred
    )


    for idx, target in enumerate(
        ["Load", "Solar"]
    ):

        pred_scaled = (
            predictions[idx]
            .reshape(-1)
        )

        (
            mae,
            rmse,
            r2,
            true_original,
            pred_original

        ) = calculate_metrics(

            y_test[:, idx],
            pred_scaled,
            idx
        )


        run_rows.append({

            "Seed": seed,
            "Model": "LSTM",
            "Target": target,

            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,

            "Parameters": params,
            "Best_epoch": best_epoch,

            "Training_time_s":
                train_time,

            "Inference_time_s":
                inference_time
        })


        prediction_frames.append(

            pd.DataFrame({

                "Timestamp":
                    pd.to_datetime(
                        test_times,
                        utc=True
                    ),

                "Seed":
                    seed,

                "Model":
                    "LSTM",

                "Target":
                    target,

                "Actual":
                    true_original,

                "Prediction":
                    pred_original,

                "Error":
                    (
                        pred_original -
                        true_original
                    ),

                "Absolute_error":
                    np.abs(
                        pred_original -
                        true_original
                    ),

                "Squared_error":
                    (
                        pred_original -
                        true_original
                    ) ** 2
            })
        )


    checkpoint()

    print(
        f"Completed | params={params:,} | "
        f"epoch={best_epoch} | "
        f"train={train_time:.1f}s | "
        f"inference={inference_time:.2f}s"
    )

    del model
    gc.collect()


    # ========================================================
    # B. SINGLE-TASK TRANSFORMERS
    # ========================================================

    for idx, target in enumerate(
        ["Load", "Solar"]
    ):

        print(
            f"\n[B] ST-Transformer — {target}"
        )

        tf.keras.backend.clear_session()
        gc.collect()
        reset_seed(seed)

        model = build_st_transformer(
            target
        )

        params = model.count_params()

        early_stop = (
            keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=PATIENCE,
                restore_best_weights=True,
                verbose=0
            )
        )

        start = time.perf_counter()

        history = model.fit(

            X_train,
            y_train[:, idx],

            validation_data=(
                X_val,
                y_val[:, idx]
            ),

            epochs=MAX_EPOCHS,

            batch_size=BATCH_SIZE,

            callbacks=[early_stop],

            shuffle=True,

            verbose=0
        )

        train_time = (
            time.perf_counter() - start
        )

        best_epoch = int(
            np.argmin(
                history.history["val_loss"]
            ) + 1
        )

        start_pred = time.perf_counter()

        pred_scaled = (
            model.predict(
                X_test,
                batch_size=BATCH_SIZE,
                verbose=0
            )
            .reshape(-1)
        )

        inference_time = (
            time.perf_counter() -
            start_pred
        )


        (
            mae,
            rmse,
            r2,
            true_original,
            pred_original

        ) = calculate_metrics(

            y_test[:, idx],
            pred_scaled,
            idx
        )


        run_rows.append({

            "Seed": seed,
            "Model":
                "ST-Transformer",

            "Target": target,

            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,

            "Parameters": params,
            "Best_epoch": best_epoch,

            "Training_time_s":
                train_time,

            "Inference_time_s":
                inference_time
        })


        prediction_frames.append(

            pd.DataFrame({

                "Timestamp":
                    pd.to_datetime(
                        test_times,
                        utc=True
                    ),

                "Seed":
                    seed,

                "Model":
                    "ST-Transformer",

                "Target":
                    target,

                "Actual":
                    true_original,

                "Prediction":
                    pred_original,

                "Error":
                    (
                        pred_original -
                        true_original
                    ),

                "Absolute_error":
                    np.abs(
                        pred_original -
                        true_original
                    ),

                "Squared_error":
                    (
                        pred_original -
                        true_original
                    ) ** 2
            })
        )


        checkpoint()

        print(
            f"Completed | "
            f"RMSE={rmse:.3f} | "
            f"epoch={best_epoch} | "
            f"train={train_time:.1f}s | "
            f"inference={inference_time:.2f}s"
        )

        del model
        gc.collect()


    # ========================================================
    # C. WEIGHTED MTL TRANSFORMER — 2:1
    # ========================================================

    print(
        "\n[C] Weighted MTL Transformer — 2:1"
    )

    tf.keras.backend.clear_session()
    gc.collect()
    reset_seed(seed)

    model = build_mtl_transformer()

    params = model.count_params()

    early_stop = (
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=0
        )
    )

    start = time.perf_counter()

    history = model.fit(

        X_train,

        {
            "load_output":
                y_train[:, 0],

            "solar_output":
                y_train[:, 1]
        },

        validation_data=(

            X_val,

            {
                "load_output":
                    y_val[:, 0],

                "solar_output":
                    y_val[:, 1]
            }
        ),

        epochs=MAX_EPOCHS,

        batch_size=BATCH_SIZE,

        callbacks=[early_stop],

        shuffle=True,

        verbose=0
    )

    train_time = (
        time.perf_counter() - start
    )

    best_epoch = int(
        np.argmin(
            history.history["val_loss"]
        ) + 1
    )

    start_pred = time.perf_counter()

    predictions = model.predict(
        X_test,
        batch_size=BATCH_SIZE,
        verbose=0
    )

    inference_time = (
        time.perf_counter() -
        start_pred
    )


    for idx, target in enumerate(
        ["Load", "Solar"]
    ):

        pred_scaled = (
            predictions[idx]
            .reshape(-1)
        )

        (
            mae,
            rmse,
            r2,
            true_original,
            pred_original

        ) = calculate_metrics(

            y_test[:, idx],
            pred_scaled,
            idx
        )


        run_rows.append({

            "Seed": seed,

            "Model":
                "MTL-Transformer-2:1",

            "Target": target,

            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,

            "Parameters": params,
            "Best_epoch": best_epoch,

            "Training_time_s":
                train_time,

            "Inference_time_s":
                inference_time
        })


        prediction_frames.append(

            pd.DataFrame({

                "Timestamp":
                    pd.to_datetime(
                        test_times,
                        utc=True
                    ),

                "Seed":
                    seed,

                "Model":
                    "MTL-Transformer-2:1",

                "Target":
                    target,

                "Actual":
                    true_original,

                "Prediction":
                    pred_original,

                "Error":
                    (
                        pred_original -
                        true_original
                    ),

                "Absolute_error":
                    np.abs(
                        pred_original -
                        true_original
                    ),

                "Squared_error":
                    (
                        pred_original -
                        true_original
                    ) ** 2
            })
        )


    checkpoint()

    print(
        f"Completed | params={params:,} | "
        f"epoch={best_epoch} | "
        f"train={train_time:.1f}s | "
        f"inference={inference_time:.2f}s"
    )

    del model
    gc.collect()


# ============================================================
# 13. FINAL TEST RESULTS
# ============================================================

final_test_runs = pd.DataFrame(
    run_rows
)


print("\n" + "=" * 80)
print("FINAL TEST RESULTS — ALL SEEDS")
print("=" * 80)


print(
    final_test_runs[
        [
            "Seed",
            "Model",
            "Target",
            "MAE",
            "RMSE",
            "R2",
            "Parameters",
            "Best_epoch",
            "Training_time_s",
            "Inference_time_s"
        ]
    ].to_string(

        index=False,

        formatters={

            "MAE":
                lambda x: f"{x:.3f}",

            "RMSE":
                lambda x: f"{x:.3f}",

            "R2":
                lambda x: f"{x:.6f}",

            "Training_time_s":
                lambda x: f"{x:.2f}",

            "Inference_time_s":
                lambda x: f"{x:.3f}"
        }
    )
)


# ============================================================
# 14. FINAL MEAN ± SD
# ============================================================

final_summary = (

    final_test_runs

    .groupby(
        ["Model", "Target"]
    )

    .agg(

        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),

        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),

        R2_mean=("R2", "mean"),
        R2_std=("R2", "std"),

        Train_time_mean=(
            "Training_time_s",
            "mean"
        ),

        Train_time_std=(
            "Training_time_s",
            "std"
        ),

        Inference_time_mean=(
            "Inference_time_s",
            "mean"
        ),

        Inference_time_std=(
            "Inference_time_s",
            "std"
        ),

        Parameters=(
            "Parameters",
            "first"
        )
    )

    .reset_index()
)


final_summary.to_csv(
    SUMMARY_FILE,
    index=False
)


print("\n" + "=" * 80)
print("FINAL TEST SUMMARY — MEAN ± SD")
print("=" * 80)


print(
    final_summary.to_string(

        index=False,

        formatters={

            "MAE_mean":
                lambda x: f"{x:.3f}",

            "MAE_std":
                lambda x: f"{x:.3f}",

            "RMSE_mean":
                lambda x: f"{x:.3f}",

            "RMSE_std":
                lambda x: f"{x:.3f}",

            "R2_mean":
                lambda x: f"{x:.6f}",

            "R2_std":
                lambda x: f"{x:.6f}",

            "Train_time_mean":
                lambda x: f"{x:.2f}",

            "Train_time_std":
                lambda x: f"{x:.2f}",

            "Inference_time_mean":
                lambda x: f"{x:.3f}",

            "Inference_time_std":
                lambda x: f"{x:.3f}"
        }
    )
)


# ============================================================
# 15. COMPUTATIONAL EFFICIENCY TABLE
# ============================================================

def get_first(
    model_name,
    column
):

    return float(

        final_test_runs.loc[
            final_test_runs["Model"]
            == model_name,
            column
        ].iloc[0]
    )


lstm_params = int(
    get_first(
        "LSTM",
        "Parameters"
    )
)

st_single_params = int(
    get_first(
        "ST-Transformer",
        "Parameters"
    )
)

mtl_params = int(
    get_first(
        "MTL-Transformer-2:1",
        "Parameters"
    )
)


efficiency = pd.DataFrame({

    "System": [
        "LSTM",
        "Two ST-Transformers",
        "MTL-Transformer-2:1"
    ],

    "Parameters": [
        lstm_params,
        st_single_params * 2,
        mtl_params
    ]
})


efficiency[
    "Parameter_reduction_vs_ST_pair_pct"
] = (

    100
    *
    (
        1
        -
        efficiency["Parameters"]
        /
        (st_single_params * 2)
    )
)


efficiency.to_csv(
    EFFICIENCY_FILE,
    index=False
)


print("\n" + "=" * 80)
print("PARAMETER EFFICIENCY")
print("=" * 80)

print(
    efficiency.to_string(
        index=False,
        formatters={
            "Parameter_reduction_vs_ST_pair_pct":
                lambda x: f"{x:.2f}%"
        }
    )
)


# ============================================================
# 16. PAIRED TEST RMSE COMPARISONS
# ============================================================

print("\n" + "=" * 80)
print("PAIRED TEST RMSE — MTL 2:1 vs ST")
print("=" * 80)


comparison_rows = []


for target in [
    "Load",
    "Solar"
]:

    st = (

        final_test_runs[
            (
                final_test_runs["Model"]
                == "ST-Transformer"
            )
            &
            (
                final_test_runs["Target"]
                == target
            )
        ]

        .set_index("Seed")["RMSE"]
    )


    mtl = (

        final_test_runs[
            (
                final_test_runs["Model"]
                == "MTL-Transformer-2:1"
            )
            &
            (
                final_test_runs["Target"]
                == target
            )
        ]

        .set_index("Seed")["RMSE"]
    )


    for seed in SEEDS:

        improvement = (
            (
                st.loc[seed]
                -
                mtl.loc[seed]
            )
            /
            st.loc[seed]
            *
            100
        )


        comparison_rows.append({

            "Target": target,
            "Seed": seed,

            "ST_RMSE":
                st.loc[seed],

            "MTL_RMSE":
                mtl.loc[seed],

            "MTL_improvement_pct":
                improvement
        })


test_comparison = pd.DataFrame(
    comparison_rows
)


print(
    test_comparison.to_string(

        index=False,

        formatters={

            "ST_RMSE":
                lambda x: f"{x:.3f}",

            "MTL_RMSE":
                lambda x: f"{x:.3f}",

            "MTL_improvement_pct":
                lambda x: f"{x:.2f}%"
        }
    )
)


test_comparison.to_csv(

    os.path.join(
        SAVE_DIR,
        "final_test_mtl_vs_st.csv"
    ),

    index=False
)


print("\nAverage paired difference:")

for target in [
    "Load",
    "Solar"
]:

    temp = test_comparison[
        test_comparison["Target"]
        == target
    ]

    print(

        f"{target:<6}: "

        f"{temp['MTL_improvement_pct'].mean():.2f}% "
        f"± "
        f"{temp['MTL_improvement_pct'].std():.2f}%"
    )


# ============================================================
# 17. FINAL CHECKPOINT
# ============================================================

final_test_runs.to_csv(
    RUN_FILE,
    index=False
)

all_predictions = pd.concat(
    prediction_frames,
    ignore_index=True
)

all_predictions.to_csv(
    PREDICTION_FILE,
    index=False
)


print("\n" + "=" * 80)
print("FINAL CHECKPOINT")
print("=" * 80)

print("Saved:")
print(" ✓ final_model_selection_freeze.json")
print(" ✓ final_test_five_seed_runs.csv")
print(" ✓ final_test_five_seed_summary.csv")
print(" ✓ final_test_predictions.csv")
print(" ✓ final_model_efficiency.csv")
print(" ✓ final_test_mtl_vs_st.csv")

print(
    "\nPrediction rows:",
    f"{len(all_predictions):,}"
)

expected_prediction_rows = (
    6576
    *
    5
    *
    6
)

assert (
    len(all_predictions)
    ==
    expected_prediction_rows
)

print(
    "Expected prediction rows:",
    f"{expected_prediction_rows:,}"
)

print(
    "\n✓ All hourly test predictions preserved."
)

print(
    "✓ Final test evaluation completed "
    "under the locked protocol."
)

print("\n" + "=" * 80)
print("STEP 4.6 COMPLETE")
print("=" * 80)

STEP 4.6 — LOCKED FIVE-SEED FINAL TEST EVALUATION

LOCKED DATA:
Train      : (34121, 168, 8)
Validation : (8760, 168, 8)
Test       : (6576, 168, 8)
Test period: 2020-01-01 00:00:00+00:00 → 2020-09-30 23:00:00+00:00

✓ Model-selection freeze written BEFORE test prediction.

################################################################################
FINAL LOCKED EVALUATION — SEED 1
################################################################################

[A] LSTM
Completed | params=33,346 | epoch=26 | train=321.5s | inference=1.12s

[B] ST-Transformer — Load
Completed | RMSE=740.704 | epoch=10 | train=82.8s | inference=7.17s

[B] ST-Transformer — Solar
Completed | RMSE=625.383 | epoch=8 | train=77.4s | inference=5.25s

[C] Weighted MTL Transformer — 2:1
Completed | params=71,810 | epoch=27 | train=141.0s | inference=5.32s

################################################################################
FINAL LOCKED EVALUATION — SEED 7
#######################################

Step 4.7 — Statistical Comparison of the Locked Final Forecasts

In [9]:
# ============================================================
# STEP 4.7 — STATISTICAL COMPARISON OF LOCKED FINAL FORECASTS
#
# Uses ONLY saved Step 4.6 predictions.
# NO TRAINING.
# NO MODEL SELECTION.
#
# Primary comparison:
#   Diebold-Mariano-style test using squared-error loss
#   with Newey-West / HAC long-run variance.
#
# Supporting comparison:
#   Moving-block bootstrap 95% CI for RMSE difference.
#
# Comparisons:
#   MTL 2:1 vs LSTM
#   MTL 2:1 vs ST-Transformer
#
# Targets:
#   Load
#   Solar
# ============================================================

import os
import numpy as np
import pandas as pd

from scipy.stats import norm

print("=" * 80)
print("STEP 4.7 — STATISTICAL COMPARISON OF FINAL FORECASTS")
print("=" * 80)

# ------------------------------------------------------------
# 1. PATHS
# ------------------------------------------------------------

SAVE_DIR = (
    "/content/drive/MyDrive/"
    "SolarLoad_MTL_Study"
)

PRED_FILE = os.path.join(
    SAVE_DIR,
    "final_test_predictions.csv"
)

OUTPUT_FILE = os.path.join(
    SAVE_DIR,
    "final_statistical_comparisons.csv"
)

SEED_FILE = os.path.join(
    SAVE_DIR,
    "final_statistical_seed_comparisons.csv"
)

BOOT_FILE = os.path.join(
    SAVE_DIR,
    "final_rmse_block_bootstrap.csv"
)


# ------------------------------------------------------------
# 2. LOAD LOCKED PREDICTIONS
# ------------------------------------------------------------

pred = pd.read_csv(
    PRED_FILE,
    parse_dates=["Timestamp"]
)

print("\nLoaded prediction rows:")
print(f"{len(pred):,}")

assert len(pred) == 197280

print("\nModels:")
print(sorted(pred["Model"].unique()))

print("\nTargets:")
print(sorted(pred["Target"].unique()))

print("\nSeeds:")
print(sorted(pred["Seed"].unique()))


expected_models = {
    "LSTM",
    "ST-Transformer",
    "MTL-Transformer-2:1"
}

assert set(pred["Model"].unique()) == expected_models

assert set(pred["Target"].unique()) == {
    "Load",
    "Solar"
}

assert sorted(
    pred["Seed"].unique().tolist()
) == [1, 7, 21, 42, 100]


# ------------------------------------------------------------
# 3. NEWey-WEST LONG-RUN VARIANCE
# ------------------------------------------------------------

def newey_west_variance(
    d,
    lag=24
):

    """
    Long-run variance of a loss-differential series
    using Bartlett/Newey-West weights.
    """

    d = np.asarray(
        d,
        dtype=float
    )

    d = d[np.isfinite(d)]

    n = len(d)

    centered = (
        d - np.mean(d)
    )

    gamma0 = np.sum(
        centered * centered
    ) / n

    lrv = gamma0

    for k in range(
        1,
        lag + 1
    ):

        gamma_k = np.sum(
            centered[k:]
            *
            centered[:-k]
        ) / n

        weight = (
            1.0
            -
            k / (lag + 1.0)
        )

        lrv += (
            2.0
            *
            weight
            *
            gamma_k
        )

    return lrv


# ------------------------------------------------------------
# 4. DM-STYLE HAC TEST
#
# d_t = loss_A - loss_B
#
# Positive mean d:
#     Model B has lower loss.
#
# Negative mean d:
#     Model A has lower loss.
# ------------------------------------------------------------

def dm_hac_test(
    loss_a,
    loss_b,
    lag=24
):

    loss_a = np.asarray(
        loss_a,
        dtype=float
    )

    loss_b = np.asarray(
        loss_b,
        dtype=float
    )

    d = (
        loss_a
        -
        loss_b
    )

    d = d[np.isfinite(d)]

    n = len(d)

    mean_d = np.mean(d)

    lrv = newey_west_variance(
        d,
        lag=lag
    )

    if (
        not np.isfinite(lrv)
        or lrv <= 0
    ):

        return (
            mean_d,
            np.nan,
            np.nan
        )

    standard_error = np.sqrt(
        lrv / n
    )

    statistic = (
        mean_d /
        standard_error
    )

    p_value = (
        2.0
        *
        norm.sf(
            abs(statistic)
        )
    )

    return (
        mean_d,
        statistic,
        p_value
    )


# ------------------------------------------------------------
# 5. PAIRED DATA EXTRACTION
# ------------------------------------------------------------

def paired_errors(
    dataframe,
    model_a,
    model_b,
    target,
    seed
):

    a = dataframe[
        (
            dataframe["Model"]
            == model_a
        )
        &
        (
            dataframe["Target"]
            == target
        )
        &
        (
            dataframe["Seed"]
            == seed
        )
    ][
        [
            "Timestamp",
            "Actual",
            "Prediction",
            "Squared_error"
        ]
    ].copy()


    b = dataframe[
        (
            dataframe["Model"]
            == model_b
        )
        &
        (
            dataframe["Target"]
            == target
        )
        &
        (
            dataframe["Seed"]
            == seed
        )
    ][
        [
            "Timestamp",
            "Actual",
            "Prediction",
            "Squared_error"
        ]
    ].copy()


    merged = a.merge(
        b,
        on="Timestamp",
        suffixes=("_A", "_B"),
        how="inner"
    )

    assert len(merged) == 6576

    # Actual target values must be identical.
    assert np.allclose(
        merged["Actual_A"],
        merged["Actual_B"]
    )

    return merged


# ------------------------------------------------------------
# 6. SEED-LEVEL HAC TESTS
# ------------------------------------------------------------

comparisons = [

    (
        "MTL-Transformer-2:1",
        "LSTM"
    ),

    (
        "MTL-Transformer-2:1",
        "ST-Transformer"
    )
]


seed_rows = []


for model_a, model_b in comparisons:

    for target in [
        "Load",
        "Solar"
    ]:

        for seed in [
            1, 7, 21, 42, 100
        ]:

            paired = paired_errors(
                pred,
                model_a,
                model_b,
                target,
                seed
            )

            loss_a = (
                paired[
                    "Squared_error_A"
                ].values
            )

            loss_b = (
                paired[
                    "Squared_error_B"
                ].values
            )

            (
                mean_diff,
                dm_stat,
                p_value

            ) = dm_hac_test(
                loss_a,
                loss_b,
                lag=24
            )


            rmse_a = np.sqrt(
                np.mean(loss_a)
            )

            rmse_b = np.sqrt(
                np.mean(loss_b)
            )


            seed_rows.append({

                "Model_A": model_a,
                "Model_B": model_b,

                "Target": target,
                "Seed": seed,

                "RMSE_A": rmse_a,
                "RMSE_B": rmse_b,

                "RMSE_A_minus_B":
                    rmse_a - rmse_b,

                "Mean_squared_loss_diff":
                    mean_diff,

                "HAC_lag": 24,

                "DM_HAC_stat":
                    dm_stat,

                "p_value":
                    p_value
            })


seed_stats = pd.DataFrame(
    seed_rows
)

seed_stats.to_csv(
    SEED_FILE,
    index=False
)


print("\n" + "=" * 80)
print("SEED-LEVEL HAC COMPARISONS")
print("=" * 80)

print(
    seed_stats.to_string(

        index=False,

        formatters={

            "RMSE_A":
                lambda x: f"{x:.3f}",

            "RMSE_B":
                lambda x: f"{x:.3f}",

            "RMSE_A_minus_B":
                lambda x: f"{x:.3f}",

            "DM_HAC_stat":
                lambda x: f"{x:.3f}",

            "p_value":
                lambda x: f"{x:.6g}"
        }
    )
)


# ------------------------------------------------------------
# 7. ENSEMBLE-OF-SEEDS / MEAN-PREDICTION COMPARISON
#
# Average predictions across the five frozen seeds first.
# This produces one forecast from each model architecture
# at every test timestamp.
#
# This is NOT used for model selection.
# ------------------------------------------------------------

ensemble = (

    pred

    .groupby(
        [
            "Timestamp",
            "Model",
            "Target"
        ],
        as_index=False
    )

    .agg(
        Actual=("Actual", "first"),
        Prediction=("Prediction", "mean")
    )
)


ensemble["Error"] = (
    ensemble["Prediction"]
    -
    ensemble["Actual"]
)

ensemble["Squared_error"] = (
    ensemble["Error"] ** 2
)


# ------------------------------------------------------------
# 8. MOVING-BLOCK BOOTSTRAP
# ------------------------------------------------------------

def moving_block_rmse_ci(
    actual,
    pred_a,
    pred_b,
    block_length=24,
    n_boot=2000,
    seed=2026
):

    actual = np.asarray(
        actual,
        dtype=float
    )

    pred_a = np.asarray(
        pred_a,
        dtype=float
    )

    pred_b = np.asarray(
        pred_b,
        dtype=float
    )

    n = len(actual)

    rng = np.random.default_rng(
        seed
    )

    error_a = (
        pred_a - actual
    )

    error_b = (
        pred_b - actual
    )

    observed_a = np.sqrt(
        np.mean(
            error_a ** 2
        )
    )

    observed_b = np.sqrt(
        np.mean(
            error_b ** 2
        )
    )

    observed_diff = (
        observed_a
        -
        observed_b
    )


    possible_starts = np.arange(
        0,
        n - block_length + 1
    )


    boot_diffs = np.empty(
        n_boot,
        dtype=float
    )


    blocks_needed = int(
        np.ceil(
            n / block_length
        )
    )


    for b in range(n_boot):

        starts = rng.choice(
            possible_starts,
            size=blocks_needed,
            replace=True
        )

        idx_parts = [

            np.arange(
                start,
                start + block_length
            )

            for start in starts
        ]

        idx = np.concatenate(
            idx_parts
        )[:n]


        rmse_a = np.sqrt(
            np.mean(
                error_a[idx] ** 2
            )
        )

        rmse_b = np.sqrt(
            np.mean(
                error_b[idx] ** 2
            )
        )


        boot_diffs[b] = (
            rmse_a - rmse_b
        )


    lower = np.percentile(
        boot_diffs,
        2.5
    )

    upper = np.percentile(
        boot_diffs,
        97.5
    )


    return (
        observed_a,
        observed_b,
        observed_diff,
        lower,
        upper
    )


# ------------------------------------------------------------
# 9. ARCHITECTURE-LEVEL COMPARISON
# ------------------------------------------------------------

summary_rows = []
bootstrap_rows = []


for model_a, model_b in comparisons:

    for target in [
        "Load",
        "Solar"
    ]:

        a = ensemble[
            (
                ensemble["Model"]
                == model_a
            )
            &
            (
                ensemble["Target"]
                == target
            )
        ].sort_values(
            "Timestamp"
        )


        b = ensemble[
            (
                ensemble["Model"]
                == model_b
            )
            &
            (
                ensemble["Target"]
                == target
            )
        ].sort_values(
            "Timestamp"
        )


        merged = a.merge(
            b,
            on="Timestamp",
            suffixes=("_A", "_B")
        )

        assert len(merged) == 6576


        actual = (
            merged["Actual_A"]
            .values
        )

        pred_a = (
            merged["Prediction_A"]
            .values
        )

        pred_b = (
            merged["Prediction_B"]
            .values
        )


        assert np.allclose(
            merged["Actual_A"],
            merged["Actual_B"]
        )


        sq_a = (
            pred_a - actual
        ) ** 2

        sq_b = (
            pred_b - actual
        ) ** 2


        (
            mean_diff,
            dm_stat,
            p_value

        ) = dm_hac_test(
            sq_a,
            sq_b,
            lag=24
        )


        (
            rmse_a,
            rmse_b,
            rmse_diff,
            ci_low,
            ci_high

        ) = moving_block_rmse_ci(

            actual,
            pred_a,
            pred_b,

            block_length=24,
            n_boot=2000,
            seed=2026
        )


        summary_rows.append({

            "Model_A": model_a,
            "Model_B": model_b,

            "Target": target,

            "RMSE_A":
                rmse_a,

            "RMSE_B":
                rmse_b,

            "RMSE_A_minus_B":
                rmse_diff,

            "RMSE_diff_CI95_low":
                ci_low,

            "RMSE_diff_CI95_high":
                ci_high,

            "Mean_squared_loss_diff":
                mean_diff,

            "DM_HAC_stat":
                dm_stat,

            "p_value":
                p_value,

            "HAC_lag":
                24,

            "Bootstrap_block_hours":
                24,

            "Bootstrap_repetitions":
                2000
        })


        bootstrap_rows.append({

            "Comparison":
                f"{model_a} vs {model_b}",

            "Target":
                target,

            "RMSE_difference":
                rmse_diff,

            "CI95_low":
                ci_low,

            "CI95_high":
                ci_high
        })


final_stats = pd.DataFrame(
    summary_rows
)

bootstrap_summary = pd.DataFrame(
    bootstrap_rows
)


final_stats.to_csv(
    OUTPUT_FILE,
    index=False
)

bootstrap_summary.to_csv(
    BOOT_FILE,
    index=False
)


# ------------------------------------------------------------
# 10. FINAL OUTPUT
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("ARCHITECTURE-LEVEL STATISTICAL COMPARISON")
print("=" * 80)

print(
    final_stats.to_string(

        index=False,

        formatters={

            "RMSE_A":
                lambda x: f"{x:.3f}",

            "RMSE_B":
                lambda x: f"{x:.3f}",

            "RMSE_A_minus_B":
                lambda x: f"{x:.3f}",

            "RMSE_diff_CI95_low":
                lambda x: f"{x:.3f}",

            "RMSE_diff_CI95_high":
                lambda x: f"{x:.3f}",

            "DM_HAC_stat":
                lambda x: f"{x:.3f}",

            "p_value":
                lambda x: f"{x:.6g}"
        }
    )
)


print("\nInterpretation convention:")
print(
    "RMSE_A_minus_B < 0  → Model A has lower RMSE."
)

print(
    "RMSE_A_minus_B > 0  → Model B has lower RMSE."
)

print(
    "Bootstrap CI excluding 0 indicates a "
    "stable RMSE difference under the chosen "
    "24-hour block-resampling scheme."
)


print("\n" + "=" * 80)
print("SEED-LEVEL DIRECTION COUNTS")
print("=" * 80)


for model_a, model_b in comparisons:

    for target in [
        "Load",
        "Solar"
    ]:

        temp = seed_stats[
            (
                seed_stats["Model_A"]
                == model_a
            )
            &
            (
                seed_stats["Model_B"]
                == model_b
            )
            &
            (
                seed_stats["Target"]
                == target
            )
        ]

        a_better = int(
            (
                temp[
                    "RMSE_A_minus_B"
                ] < 0
            ).sum()
        )

        b_better = int(
            (
                temp[
                    "RMSE_A_minus_B"
                ] > 0
            ).sum()
        )

        significant = int(
            (
                temp["p_value"]
                < 0.05
            ).sum()
        )


        print(
            f"{target:<6} | "
            f"{model_a} vs {model_b} | "
            f"A lower RMSE: {a_better}/5 | "
            f"B lower RMSE: {b_better}/5 | "
            f"HAC p<0.05: {significant}/5"
        )


print("\nSaved:")
print(" ✓ final_statistical_seed_comparisons.csv")
print(" ✓ final_statistical_comparisons.csv")
print(" ✓ final_rmse_block_bootstrap.csv")

print(
    "\n✓ No models were trained."
)

print(
    "✓ No model-selection decisions were made."
)

print(
    "✓ Analysis used only the locked Step 4.6 forecasts."
)

print("\n" + "=" * 80)
print("STEP 4.7 COMPLETE")
print("=" * 80)

STEP 4.7 — STATISTICAL COMPARISON OF FINAL FORECASTS

Loaded prediction rows:
197,280

Models:
['LSTM', 'MTL-Transformer-2:1', 'ST-Transformer']

Targets:
['Load', 'Solar']

Seeds:
[np.int64(1), np.int64(7), np.int64(21), np.int64(42), np.int64(100)]

SEED-LEVEL HAC COMPARISONS
            Model_A        Model_B Target  Seed   RMSE_A   RMSE_B RMSE_A_minus_B  Mean_squared_loss_diff  HAC_lag DM_HAC_stat     p_value
MTL-Transformer-2:1           LSTM   Load     1  742.540  762.647        -20.107           -30264.543856       24      -1.934   0.0531627
MTL-Transformer-2:1           LSTM   Load     7  766.437  878.166       -111.729          -183749.094025       24      -6.706 1.99544e-11
MTL-Transformer-2:1           LSTM   Load    21  738.937  854.003       -115.067          -183294.104730       24      -6.452 1.10541e-10
MTL-Transformer-2:1           LSTM   Load    42 1101.083  747.432        353.651           653728.554016       24      14.261 3.83444e-46
MTL-Transformer-2:1           L

Step 4.8A — Restore only the raw study dataframe

In [11]:
# ============================================================
# STEP 4.8A — RESTORE RAW STUDY DATAFRAME FOR SHIFT ANALYSIS
#
# This does NOT change:
#   - frozen sequences
#   - scalers
#   - models
#   - validation results
#   - final test results
#
# It only reloads the exact frozen OPSD source required for
# distribution-shift analysis.
# ============================================================

import os
import pandas as pd
import numpy as np

print("=" * 80)
print("STEP 4.8A — RESTORE RAW STUDY DATAFRAME")
print("=" * 80)

DATA_URL = (
    "https://data.open-power-system-data.org/"
    "time_series/2020-10-06/"
    "time_series_60min_singleindex.csv"
)

LOAD_COL = "DE_load_actual_entsoe_transparency"
SOLAR_COL = "DE_solar_generation_actual"

print("\nLoading frozen OPSD release:")
print("Release: 2020-10-06")
print("File   : time_series_60min_singleindex.csv")

# Read ONLY the columns needed for Step 4.8.
# This is much lighter than loading all 300 columns.
df_study = pd.read_csv(
    DATA_URL,
    usecols=[
        "utc_timestamp",
        LOAD_COL,
        SOLAR_COL
    ]
)

# ------------------------------------------------------------
# Parse timestamp
# ------------------------------------------------------------

df_study["utc_timestamp"] = pd.to_datetime(
    df_study["utc_timestamp"],
    utc=True,
    errors="coerce"
)

assert (
    df_study["utc_timestamp"]
    .isna()
    .sum()
    == 0
)

# ------------------------------------------------------------
# Restrict to frozen study period
# ------------------------------------------------------------

START = pd.Timestamp(
    "2015-01-01 00:00:00",
    tz="UTC"
)

END = pd.Timestamp(
    "2020-09-30 23:00:00",
    tz="UTC"
)

df_study = (
    df_study[
        df_study["utc_timestamp"]
        .between(
            START,
            END
        )
    ]
    .copy()
    .sort_values("utc_timestamp")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Frozen integrity checks
# ------------------------------------------------------------

print("\nRESTORED DATAFRAME")
print("-" * 80)

print("Shape:", df_study.shape)

print(
    "Period:",
    df_study["utc_timestamp"].iloc[0],
    "→",
    df_study["utc_timestamp"].iloc[-1]
)

print(
    "Load missing :",
    int(
        df_study[LOAD_COL]
        .isna()
        .sum()
    )
)

print(
    "Solar missing:",
    int(
        df_study[SOLAR_COL]
        .isna()
        .sum()
    )
)

duplicate_count = int(
    df_study["utc_timestamp"]
    .duplicated()
    .sum()
)

hour_diffs = (
    df_study["utc_timestamp"]
    .diff()
    .dropna()
    .dt.total_seconds()
    / 3600.0
)

non_hourly = int(
    (hour_diffs != 1.0).sum()
)

print(
    "Duplicate timestamps:",
    duplicate_count
)

print(
    "Non-hourly intervals:",
    non_hourly
)

# ------------------------------------------------------------
# Must match frozen D2
# ------------------------------------------------------------

assert len(df_study) == 50400, (
    f"Expected 50,400 rows, got {len(df_study)}"
)

assert (
    df_study[LOAD_COL]
    .isna()
    .sum()
    == 0
), "Frozen audit expected 0 missing Load values."

assert (
    df_study[SOLAR_COL]
    .isna()
    .sum()
    == 103
), "Frozen audit expected 103 missing Solar values."

assert duplicate_count == 0

assert non_hourly == 0

assert (
    df_study["utc_timestamp"].iloc[0]
    == START
)

assert (
    df_study["utc_timestamp"].iloc[-1]
    == END
)

print("\n✓ Frozen dataset identity reproduced.")
print("✓ 50,400 hourly observations verified.")
print("✓ Load missingness = 0 verified.")
print("✓ Solar missingness = 103 verified.")
print("✓ No duplicate timestamps.")
print("✓ No non-hourly intervals.")

print("\nSTEP 4.8A COMPLETE")

STEP 4.8A — RESTORE RAW STUDY DATAFRAME

Loading frozen OPSD release:
Release: 2020-10-06
File   : time_series_60min_singleindex.csv

RESTORED DATAFRAME
--------------------------------------------------------------------------------
Shape: (50400, 3)
Period: 2015-01-01 00:00:00+00:00 → 2020-09-30 23:00:00+00:00
Load missing : 0
Solar missing: 103
Duplicate timestamps: 0
Non-hourly intervals: 0

✓ Frozen dataset identity reproduced.
✓ 50,400 hourly observations verified.
✓ Load missingness = 0 verified.
✓ Solar missingness = 103 verified.
✓ No duplicate timestamps.
✓ No non-hourly intervals.

STEP 4.8A COMPLETE


Step 4.8 — Temporal Distribution Shift

In [12]:
# ============================================================
# STEP 4.8 — SEASON-MATCHED TEMPORAL DISTRIBUTION-SHIFT AUDIT
#
# NO MODEL TRAINING.
#
# Compares:
#   Historical Train Jan-Sep (2015-2018) vs Test Jan-Sep 2020
#   Validation Jan-Sep 2019            vs Test Jan-Sep 2020
#
# Targets:
#   Load
#   Solar
#
# Metrics:
#   Mean / SD
#   Standardized mean difference (SMD)
#   Wasserstein distance
#   Normalized Wasserstein distance
#   KS statistic + p-value
#
# IMPORTANT:
# Statistical significance != practical magnitude.
# ============================================================

import os
import numpy as np
import pandas as pd

from scipy.stats import (
    ks_2samp,
    wasserstein_distance
)

print("=" * 80)
print("STEP 4.8 — SEASON-MATCHED TEMPORAL DISTRIBUTION-SHIFT AUDIT")
print("=" * 80)

SAVE_DIR = (
    "/content/drive/MyDrive/"
    "SolarLoad_MTL_Study"
)

OUT_FILE = os.path.join(
    SAVE_DIR,
    "temporal_distribution_shift_audit.csv"
)

MONTHLY_FILE = os.path.join(
    SAVE_DIR,
    "temporal_distribution_shift_monthly.csv"
)

# ------------------------------------------------------------
# 1. USE THE RECONSTRUCTED RAW STUDY DATAFRAME
#
# Expected dataframe:
# df_study
#
# If your current notebook used another variable name for the
# reconstructed 2015-2020 hourly dataframe, replace df_study below.
# ------------------------------------------------------------

print("\nChecking study dataframe...")

assert "df_study" in globals(), (
    "df_study is not present. "
    "Use the reconstructed study dataframe from Step R2."
)

required_cols = [
    "utc_timestamp",
    "DE_load_actual_entsoe_transparency",
    "DE_solar_generation_actual"
]

for col in required_cols:
    assert col in df_study.columns, (
        f"Missing required column: {col}"
    )

shift_df = df_study[
    required_cols
].copy()

shift_df["utc_timestamp"] = pd.to_datetime(
    shift_df["utc_timestamp"],
    utc=True
)

shift_df = shift_df.sort_values(
    "utc_timestamp"
).reset_index(drop=True)

shift_df["year"] = (
    shift_df["utc_timestamp"].dt.year
)

shift_df["month"] = (
    shift_df["utc_timestamp"].dt.month
)

# Jan-Sep only
shift_df = shift_df[
    shift_df["month"].between(1, 9)
].copy()

print(
    "Season-matched observations:",
    f"{len(shift_df):,}"
)


# ------------------------------------------------------------
# 2. DEFINE PERIODS
# ------------------------------------------------------------

train_js = shift_df[
    shift_df["year"].between(
        2015,
        2018
    )
].copy()

val_js = shift_df[
    shift_df["year"] == 2019
].copy()

test_js = shift_df[
    shift_df["year"] == 2020
].copy()


print("\nPERIOD COUNTS")
print("-" * 80)

print(
    "Train Jan-Sep 2015-2018:",
    f"{len(train_js):,}"
)

print(
    "Validation Jan-Sep 2019:",
    f"{len(val_js):,}"
)

print(
    "Test Jan-Sep 2020:",
    f"{len(test_js):,}"
)


# ------------------------------------------------------------
# 3. EFFECT-SIZE FUNCTIONS
# ------------------------------------------------------------

def pooled_sd(x, y):

    x = np.asarray(
        x,
        dtype=float
    )

    y = np.asarray(
        y,
        dtype=float
    )

    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    nx = len(x)
    ny = len(y)

    vx = np.var(
        x,
        ddof=1
    )

    vy = np.var(
        y,
        ddof=1
    )

    return np.sqrt(
        (
            (nx - 1) * vx
            +
            (ny - 1) * vy
        )
        /
        (nx + ny - 2)
    )


def compare_distributions(
    reference,
    future,
    reference_name,
    future_name,
    target_name
):

    reference = np.asarray(
        reference,
        dtype=float
    )

    future = np.asarray(
        future,
        dtype=float
    )

    reference = reference[
        np.isfinite(reference)
    ]

    future = future[
        np.isfinite(future)
    ]

    ref_mean = np.mean(reference)
    fut_mean = np.mean(future)

    ref_sd = np.std(
        reference,
        ddof=1
    )

    fut_sd = np.std(
        future,
        ddof=1
    )

    p_sd = pooled_sd(
        reference,
        future
    )

    smd = (
        (fut_mean - ref_mean)
        /
        p_sd
    )

    mean_change_pct = (
        100.0
        *
        (fut_mean - ref_mean)
        /
        ref_mean
    )

    sd_change_pct = (
        100.0
        *
        (fut_sd - ref_sd)
        /
        ref_sd
    )

    wd = wasserstein_distance(
        reference,
        future
    )

    # Normalize by reference SD.
    wd_norm = (
        wd / ref_sd
    )

    ks = ks_2samp(
        reference,
        future,
        alternative="two-sided",
        method="auto"
    )

    return {

        "Target":
            target_name,

        "Reference_period":
            reference_name,

        "Future_period":
            future_name,

        "N_reference":
            len(reference),

        "N_future":
            len(future),

        "Reference_mean":
            ref_mean,

        "Future_mean":
            fut_mean,

        "Mean_change_pct":
            mean_change_pct,

        "Reference_SD":
            ref_sd,

        "Future_SD":
            fut_sd,

        "SD_change_pct":
            sd_change_pct,

        "Standardized_mean_difference":
            smd,

        "Wasserstein_distance":
            wd,

        "Normalized_Wasserstein":
            wd_norm,

        "KS_statistic":
            ks.statistic,

        "KS_p_value":
            ks.pvalue
    }


# ------------------------------------------------------------
# 4. GLOBAL SEASON-MATCHED SHIFT AUDIT
# ------------------------------------------------------------

target_map = {

    "Load":
        "DE_load_actual_entsoe_transparency",

    "Solar":
        "DE_solar_generation_actual"
}


rows = []


for target_name, col in target_map.items():

    rows.append(

        compare_distributions(

            train_js[col].values,

            test_js[col].values,

            "2015-2018 Jan-Sep",

            "2020 Jan-Sep",

            target_name
        )
    )


    rows.append(

        compare_distributions(

            val_js[col].values,

            test_js[col].values,

            "2019 Jan-Sep",

            "2020 Jan-Sep",

            target_name
        )
    )


shift_results = pd.DataFrame(
    rows
)

shift_results.to_csv(
    OUT_FILE,
    index=False
)


print("\n" + "=" * 80)
print("SEASON-MATCHED DISTRIBUTION COMPARISON")
print("=" * 80)


display_cols = [

    "Target",
    "Reference_period",
    "Future_period",

    "N_reference",
    "N_future",

    "Reference_mean",
    "Future_mean",

    "Mean_change_pct",

    "Reference_SD",
    "Future_SD",

    "SD_change_pct",

    "Standardized_mean_difference",

    "Normalized_Wasserstein",

    "KS_statistic",
    "KS_p_value"
]


print(
    shift_results[
        display_cols
    ].to_string(

        index=False,

        formatters={

            "Reference_mean":
                lambda x: f"{x:.3f}",

            "Future_mean":
                lambda x: f"{x:.3f}",

            "Mean_change_pct":
                lambda x: f"{x:.2f}%",

            "Reference_SD":
                lambda x: f"{x:.3f}",

            "Future_SD":
                lambda x: f"{x:.3f}",

            "SD_change_pct":
                lambda x: f"{x:.2f}%",

            "Standardized_mean_difference":
                lambda x: f"{x:.4f}",

            "Normalized_Wasserstein":
                lambda x: f"{x:.4f}",

            "KS_statistic":
                lambda x: f"{x:.4f}",

            "KS_p_value":
                lambda x: f"{x:.6g}"
        }
    )
)


# ------------------------------------------------------------
# 5. MONTH-BY-MONTH SHIFT
#
# 2015-2018 historical same-month distribution
# versus the same month in 2020.
#
# This helps determine whether the global result is being
# driven by only one or two months.
# ------------------------------------------------------------

monthly_rows = []


for target_name, col in target_map.items():

    for month in range(
        1,
        10
    ):

        historical_month = (
            shift_df[
                (
                    shift_df["year"]
                    .between(
                        2015,
                        2018
                    )
                )
                &
                (
                    shift_df["month"]
                    == month
                )
            ][col]
            .dropna()
            .values
        )

        test_month = (
            shift_df[
                (
                    shift_df["year"]
                    == 2020
                )
                &
                (
                    shift_df["month"]
                    == month
                )
            ][col]
            .dropna()
            .values
        )


        result = compare_distributions(

            historical_month,

            test_month,

            f"2015-2018 month {month:02d}",

            f"2020 month {month:02d}",

            target_name
        )

        result["Month"] = month

        monthly_rows.append(
            result
        )


monthly_results = pd.DataFrame(
    monthly_rows
)

monthly_results.to_csv(
    MONTHLY_FILE,
    index=False
)


print("\n" + "=" * 80)
print("MONTH-BY-MONTH SHIFT — 2015-2018 vs 2020")
print("=" * 80)


print(
    monthly_results[
        [
            "Target",
            "Month",
            "Mean_change_pct",
            "Standardized_mean_difference",
            "Normalized_Wasserstein",
            "KS_statistic",
            "KS_p_value"
        ]
    ].to_string(

        index=False,

        formatters={

            "Mean_change_pct":
                lambda x: f"{x:.2f}%",

            "Standardized_mean_difference":
                lambda x: f"{x:.4f}",

            "Normalized_Wasserstein":
                lambda x: f"{x:.4f}",

            "KS_statistic":
                lambda x: f"{x:.4f}",

            "KS_p_value":
                lambda x: f"{x:.6g}"
        }
    )
)


# ------------------------------------------------------------
# 6. SIMPLE MAGNITUDE FLAGS
#
# Descriptive only.
# Not formal universal thresholds.
# ------------------------------------------------------------

def descriptive_shift_flag(
    smd,
    nwd
):

    magnitude = max(
        abs(smd),
        abs(nwd)
    )

    if magnitude < 0.10:
        return "Very small"

    elif magnitude < 0.25:
        return "Small"

    elif magnitude < 0.50:
        return "Moderate"

    else:
        return "Large"


shift_results[
    "Descriptive_shift_magnitude"
] = [

    descriptive_shift_flag(
        row[
            "Standardized_mean_difference"
        ],
        row[
            "Normalized_Wasserstein"
        ]
    )

    for _, row
    in shift_results.iterrows()
]


shift_results.to_csv(
    OUT_FILE,
    index=False
)


print("\n" + "=" * 80)
print("DESCRIPTIVE SHIFT MAGNITUDE")
print("=" * 80)


print(
    shift_results[
        [
            "Target",
            "Reference_period",
            "Future_period",
            "Descriptive_shift_magnitude"
        ]
    ].to_string(
        index=False
    )
)


print("\nSaved:")
print(
    " ✓ temporal_distribution_shift_audit.csv"
)

print(
    " ✓ temporal_distribution_shift_monthly.csv"
)

print("\n✓ No model training performed.")
print("✓ Test season matched to Jan-Sep.")
print(
    "✓ Shift magnitude separated from statistical significance."
)

print("\n" + "=" * 80)
print("STEP 4.8 COMPLETE")
print("=" * 80)

STEP 4.8 — SEASON-MATCHED TEMPORAL DISTRIBUTION-SHIFT AUDIT

Checking study dataframe...
Season-matched observations: 39,360

PERIOD COUNTS
--------------------------------------------------------------------------------
Train Jan-Sep 2015-2018: 26,232
Validation Jan-Sep 2019: 6,552
Test Jan-Sep 2020: 6,576

SEASON-MATCHED DISTRIBUTION COMPARISON
Target  Reference_period Future_period  N_reference  N_future Reference_mean Future_mean Mean_change_pct Reference_SD Future_SD SD_change_pct Standardized_mean_difference Normalized_Wasserstein KS_statistic  KS_p_value
  Load 2015-2018 Jan-Sep  2020 Jan-Sep        26232      6576      55308.118   53047.851          -4.09%     9912.494  9734.444        -1.80%                      -0.2288                 0.2283       0.1332 8.99327e-82
  Load      2019 Jan-Sep  2020 Jan-Sep         6552      6576      55786.256   53047.851          -4.91%     9845.448  9734.444        -1.13%                      -0.2797                 0.2781       0.1435 2.2496

Step 4.9 — Shift–Error Coupling Analysis

In [13]:
# ============================================================
# STEP 4.9 — TEMPORAL SHIFT–FORECAST ERROR COUPLING
#
# NO TRAINING.
# NO MODEL SELECTION.
# NO NEW PREDICTIONS.
#
# Uses:
#   final_test_predictions.csv
#   temporal_distribution_shift_monthly.csv
#
# Goals:
# 1. Compute monthly 2020 RMSE for every model × target × seed.
# 2. Summarize monthly RMSE across five seeds.
# 3. Join monthly forecasting errors to season-matched shift
#    magnitude from Step 4.8.
# 4. Quantify association between distribution shift and error.
# 5. Examine whether MTL-vs-LSTM and MTL-vs-ST relative
#    performance changes with shift magnitude.
#
# Primary shift metric:
#   Normalized Wasserstein distance
#
# Supporting shift metric:
#   absolute standardized mean difference |SMD|
#
# Association:
#   Spearman rank correlation
#
# IMPORTANT:
# Association is descriptive/exploratory and NOT causal.
# ============================================================

import os
import numpy as np
import pandas as pd

from scipy.stats import spearmanr

print("=" * 88)
print("STEP 4.9 — TEMPORAL SHIFT–FORECAST ERROR COUPLING")
print("=" * 88)

SAVE_DIR = "/content/drive/MyDrive/SolarLoad_MTL_Study"

PRED_FILE = os.path.join(
    SAVE_DIR,
    "final_test_predictions.csv"
)

SHIFT_FILE = os.path.join(
    SAVE_DIR,
    "temporal_distribution_shift_monthly.csv"
)

MONTHLY_RUN_FILE = os.path.join(
    SAVE_DIR,
    "monthly_test_error_by_seed.csv"
)

MONTHLY_SUMMARY_FILE = os.path.join(
    SAVE_DIR,
    "monthly_test_error_summary.csv"
)

COUPLING_FILE = os.path.join(
    SAVE_DIR,
    "shift_error_coupling.csv"
)

RELATIVE_FILE = os.path.join(
    SAVE_DIR,
    "monthly_relative_model_performance.csv"
)

RELATIVE_COUPLING_FILE = os.path.join(
    SAVE_DIR,
    "shift_relative_performance_coupling.csv"
)


# ============================================================
# 1. LOAD FROZEN EVIDENCE
# ============================================================

pred = pd.read_csv(
    PRED_FILE,
    parse_dates=["Timestamp"]
)

shift = pd.read_csv(
    SHIFT_FILE
)

print("\nLoaded:")
print(f"Predictions : {len(pred):,}")
print(f"Shift rows  : {len(shift):,}")

assert len(pred) == 197280

expected_models = {
    "LSTM",
    "ST-Transformer",
    "MTL-Transformer-2:1"
}

assert set(pred["Model"].unique()) == expected_models

assert set(pred["Target"].unique()) == {
    "Load",
    "Solar"
}

assert sorted(
    pred["Seed"].unique().tolist()
) == [1, 7, 21, 42, 100]


# ============================================================
# 2. PREPARE MONTH IDENTIFIER
# ============================================================

pred["Month"] = pred["Timestamp"].dt.month

assert sorted(
    pred["Month"].unique().tolist()
) == list(range(1, 10))


# ============================================================
# 3. MONTHLY ERROR FOR EACH INDEPENDENT RUN
# ============================================================

def rmse_from_group(group):

    error = (
        group["Prediction"].to_numpy(dtype=float)
        -
        group["Actual"].to_numpy(dtype=float)
    )

    return np.sqrt(
        np.mean(error ** 2)
    )


def mae_from_group(group):

    error = np.abs(
        group["Prediction"].to_numpy(dtype=float)
        -
        group["Actual"].to_numpy(dtype=float)
    )

    return np.mean(error)


monthly_rows = []

for (
    model,
    target,
    seed,
    month
), group in pred.groupby(
    [
        "Model",
        "Target",
        "Seed",
        "Month"
    ]
):

    monthly_rows.append({

        "Model": model,
        "Target": target,
        "Seed": int(seed),
        "Month": int(month),
        "N": len(group),

        "MAE": mae_from_group(group),
        "RMSE": rmse_from_group(group)
    })


monthly_runs = pd.DataFrame(
    monthly_rows
).sort_values(
    [
        "Target",
        "Month",
        "Model",
        "Seed"
    ]
)

monthly_runs.to_csv(
    MONTHLY_RUN_FILE,
    index=False
)


expected_rows = (
    3 * 2 * 5 * 9
)

assert len(monthly_runs) == expected_rows

print(
    f"\nMonthly independent-run rows: "
    f"{len(monthly_runs)} "
    f"(expected {expected_rows})"
)


# ============================================================
# 4. MONTHLY MEAN ± SD ACROSS FIVE SEEDS
# ============================================================

monthly_summary = (

    monthly_runs

    .groupby(
        [
            "Model",
            "Target",
            "Month"
        ],
        as_index=False
    )

    .agg(

        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),

        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),

        Seeds=("Seed", "nunique")
    )
)

assert (
    monthly_summary["Seeds"] == 5
).all()

monthly_summary.to_csv(
    MONTHLY_SUMMARY_FILE,
    index=False
)


print("\n" + "=" * 88)
print("MONTHLY TEST RMSE — MEAN ± SD ACROSS FIVE SEEDS")
print("=" * 88)

print(
    monthly_summary[
        [
            "Target",
            "Month",
            "Model",
            "RMSE_mean",
            "RMSE_std"
        ]
    ].to_string(

        index=False,

        formatters={

            "RMSE_mean":
                lambda x: f"{x:.3f}",

            "RMSE_std":
                lambda x: f"{x:.3f}"
        }
    )
)


# ============================================================
# 5. PREPARE STEP 4.8 SHIFT EVIDENCE
#
# We specifically use the historical 2015–2018 same-month
# comparison against 2020, because this gives one consistent
# shift measure for each test month.
# ============================================================

shift_monthly = shift[
    shift["Reference_period"]
    .astype(str)
    .str.startswith("2015-2018 month")
].copy()

assert len(shift_monthly) == 18

shift_monthly["Month"] = (
    shift_monthly["Month"]
    .astype(int)
)

shift_monthly["Abs_SMD"] = np.abs(
    shift_monthly[
        "Standardized_mean_difference"
    ]
)

shift_keep = shift_monthly[
    [
        "Target",
        "Month",
        "Mean_change_pct",
        "Standardized_mean_difference",
        "Abs_SMD",
        "Normalized_Wasserstein",
        "KS_statistic",
        "KS_p_value"
    ]
].copy()


# ============================================================
# 6. JOIN SHIFT MAGNITUDE WITH MONTHLY FORECAST ERROR
# ============================================================

coupled = monthly_summary.merge(

    shift_keep,

    on=[
        "Target",
        "Month"
    ],

    how="inner",

    validate="many_to_one"
)

assert len(coupled) == 54

coupled.to_csv(
    COUPLING_FILE,
    index=False
)


# ============================================================
# 7. SHIFT ↔ ERROR ASSOCIATION
#
# n = 9 months per model/target.
#
# We report Spearman rho because:
# - sample size is small,
# - relationship need not be linear,
# - monthly values may contain outliers.
#
# p-values are exploratory because n=9 and months are temporal.
# ============================================================

association_rows = []

for (
    model,
    target
), group in coupled.groupby(
    [
        "Model",
        "Target"
    ]
):

    group = group.sort_values(
        "Month"
    )

    rho_w, p_w = spearmanr(
        group["Normalized_Wasserstein"],
        group["RMSE_mean"]
    )

    rho_smd, p_smd = spearmanr(
        group["Abs_SMD"],
        group["RMSE_mean"]
    )

    association_rows.append({

        "Model": model,
        "Target": target,
        "N_months": len(group),

        "Spearman_rho_Wasserstein_vs_RMSE":
            rho_w,

        "p_Wasserstein_vs_RMSE":
            p_w,

        "Spearman_rho_AbsSMD_vs_RMSE":
            rho_smd,

        "p_AbsSMD_vs_RMSE":
            p_smd
    })


association = pd.DataFrame(
    association_rows
)


print("\n" + "=" * 88)
print("SHIFT ↔ FORECAST ERROR ASSOCIATION")
print("=" * 88)

print(
    association.to_string(

        index=False,

        formatters={

            "Spearman_rho_Wasserstein_vs_RMSE":
                lambda x: f"{x:.4f}",

            "p_Wasserstein_vs_RMSE":
                lambda x: f"{x:.6g}",

            "Spearman_rho_AbsSMD_vs_RMSE":
                lambda x: f"{x:.4f}",

            "p_AbsSMD_vs_RMSE":
                lambda x: f"{x:.6g}"
        }
    )
)


# ============================================================
# 8. MONTHLY RELATIVE MODEL PERFORMANCE
#
# Positive improvement:
#   MTL has lower RMSE.
#
# Negative improvement:
#   comparator has lower RMSE.
#
# Uses mean RMSE across five independent seeds.
# ============================================================

pivot = monthly_summary.pivot_table(

    index=[
        "Target",
        "Month"
    ],

    columns="Model",

    values="RMSE_mean"
).reset_index()


pivot.columns.name = None


pivot[
    "MTL_vs_LSTM_improvement_pct"
] = (

    100.0
    *
    (
        pivot["LSTM"]
        -
        pivot["MTL-Transformer-2:1"]
    )
    /
    pivot["LSTM"]
)


pivot[
    "MTL_vs_ST_improvement_pct"
] = (

    100.0
    *
    (
        pivot["ST-Transformer"]
        -
        pivot["MTL-Transformer-2:1"]
    )
    /
    pivot["ST-Transformer"]
)


relative = pivot.merge(

    shift_keep,

    on=[
        "Target",
        "Month"
    ],

    how="inner",

    validate="one_to_one"
)

relative.to_csv(
    RELATIVE_FILE,
    index=False
)


print("\n" + "=" * 88)
print("MONTHLY RELATIVE PERFORMANCE")
print("=" * 88)

print(
    relative[
        [
            "Target",
            "Month",
            "Normalized_Wasserstein",
            "LSTM",
            "MTL-Transformer-2:1",
            "ST-Transformer",
            "MTL_vs_LSTM_improvement_pct",
            "MTL_vs_ST_improvement_pct"
        ]
    ].to_string(

        index=False,

        formatters={

            "Normalized_Wasserstein":
                lambda x: f"{x:.4f}",

            "LSTM":
                lambda x: f"{x:.3f}",

            "MTL-Transformer-2:1":
                lambda x: f"{x:.3f}",

            "ST-Transformer":
                lambda x: f"{x:.3f}",

            "MTL_vs_LSTM_improvement_pct":
                lambda x: f"{x:.2f}%",

            "MTL_vs_ST_improvement_pct":
                lambda x: f"{x:.2f}%"
        }
    )
)


# ============================================================
# 9. DOES RELATIVE MTL PERFORMANCE CHANGE WITH SHIFT?
#
# Again exploratory:
# n = 9 monthly observations.
# ============================================================

relative_rows = []

for target, group in relative.groupby(
    "Target"
):

    group = group.sort_values(
        "Month"
    )

    for comparison_col, label in [

        (
            "MTL_vs_LSTM_improvement_pct",
            "MTL vs LSTM"
        ),

        (
            "MTL_vs_ST_improvement_pct",
            "MTL vs ST"
        )
    ]:

        rho_w, p_w = spearmanr(
            group["Normalized_Wasserstein"],
            group[comparison_col]
        )

        rho_smd, p_smd = spearmanr(
            group["Abs_SMD"],
            group[comparison_col]
        )

        relative_rows.append({

            "Target":
                target,

            "Comparison":
                label,

            "N_months":
                len(group),

            "rho_Wasserstein_vs_MTL_improvement":
                rho_w,

            "p_Wasserstein":
                p_w,

            "rho_AbsSMD_vs_MTL_improvement":
                rho_smd,

            "p_AbsSMD":
                p_smd
        })


relative_coupling = pd.DataFrame(
    relative_rows
)

relative_coupling.to_csv(
    RELATIVE_COUPLING_FILE,
    index=False
)


print("\n" + "=" * 88)
print("SHIFT ↔ RELATIVE MTL PERFORMANCE")
print("=" * 88)

print(
    relative_coupling.to_string(

        index=False,

        formatters={

            "rho_Wasserstein_vs_MTL_improvement":
                lambda x: f"{x:.4f}",

            "p_Wasserstein":
                lambda x: f"{x:.6g}",

            "rho_AbsSMD_vs_MTL_improvement":
                lambda x: f"{x:.4f}",

            "p_AbsSMD":
                lambda x: f"{x:.6g}"
        }
    )
)


# ============================================================
# 10. HIGHER-SHIFT vs LOWER-SHIFT DESCRIPTIVE CHECK
#
# Median split is descriptive ONLY.
# It is NOT a new statistical model and NOT used for selection.
# ============================================================

print("\n" + "=" * 88)
print("HIGHER-SHIFT vs LOWER-SHIFT MONTHS — DESCRIPTIVE")
print("=" * 88)

for target, group in relative.groupby(
    "Target"
):

    median_shift = group[
        "Normalized_Wasserstein"
    ].median()

    temp = group.copy()

    temp["Shift_group"] = np.where(

        temp[
            "Normalized_Wasserstein"
        ] > median_shift,

        "Higher-shift",

        "Lower-shift"
    )

    print(f"\nTARGET: {target}")
    print(
        f"Median normalized Wasserstein = "
        f"{median_shift:.4f}"
    )

    descriptive = (

        temp

        .groupby(
            "Shift_group"
        )

        .agg(

            Months=("Month", "count"),

            Mean_shift=(
                "Normalized_Wasserstein",
                "mean"
            ),

            MTL_vs_LSTM_mean_improvement_pct=(
                "MTL_vs_LSTM_improvement_pct",
                "mean"
            ),

            MTL_vs_ST_mean_improvement_pct=(
                "MTL_vs_ST_improvement_pct",
                "mean"
            )
        )

        .reset_index()
    )

    print(
        descriptive.to_string(

            index=False,

            formatters={

                "Mean_shift":
                    lambda x: f"{x:.4f}",

                "MTL_vs_LSTM_mean_improvement_pct":
                    lambda x: f"{x:.2f}%",

                "MTL_vs_ST_mean_improvement_pct":
                    lambda x: f"{x:.2f}%"
            }
        )
    )


# ============================================================
# 11. SAVE ASSOCIATION TABLE
# ============================================================

association.to_csv(
    os.path.join(
        SAVE_DIR,
        "shift_error_association_summary.csv"
    ),
    index=False
)


print("\n" + "=" * 88)
print("FINAL CHECKPOINT")
print("=" * 88)

print("\nSaved:")
print(" ✓ monthly_test_error_by_seed.csv")
print(" ✓ monthly_test_error_summary.csv")
print(" ✓ shift_error_coupling.csv")
print(" ✓ monthly_relative_model_performance.csv")
print(" ✓ shift_relative_performance_coupling.csv")
print(" ✓ shift_error_association_summary.csv")

print("\n✓ No models trained.")
print("✓ No new predictions generated.")
print("✓ Five independent seeds retained as primary evidence.")
print("✓ Shift/error relationships treated as exploratory associations.")
print("✓ No causal interpretation made.")

print("\n" + "=" * 88)
print("STEP 4.9 COMPLETE")
print("=" * 88)

STEP 4.9 — TEMPORAL SHIFT–FORECAST ERROR COUPLING

Loaded:
Predictions : 197,280
Shift rows  : 18

Monthly independent-run rows: 270 (expected 270)

MONTHLY TEST RMSE — MEAN ± SD ACROSS FIVE SEEDS
Target  Month               Model RMSE_mean RMSE_std
  Load      1                LSTM   652.502   45.871
  Load      2                LSTM   734.131   69.642
  Load      3                LSTM   897.447   69.302
  Load      4                LSTM  1006.556   97.777
  Load      5                LSTM   933.223   98.094
  Load      6                LSTM   858.413  113.384
  Load      7                LSTM   804.194   95.967
  Load      8                LSTM   738.466   97.821
  Load      9                LSTM   703.571   93.841
 Solar      1                LSTM   292.351   16.057
 Solar      2                LSTM   348.649   14.313
 Solar      3                LSTM   490.468   52.866
 Solar      4                LSTM   671.264   88.555
 Solar      5                LSTM   542.425   31.614
 Solar  

Step 5.0A — Final Artifact & Model Availability Audit

In [14]:
# ============================================================
# STEP 5.0A — FINAL ARTIFACT & MODEL AVAILABILITY AUDIT
#
# PURPOSE:
# Determine exactly what is preserved in Google Drive before
# designing controlled robustness / stress testing.
#
# NO TRAINING.
# NO MODEL LOADING.
# NO FILE MODIFICATION.
# ============================================================

import os
import glob
import pandas as pd

print("=" * 88)
print("STEP 5.0A — FINAL ARTIFACT & MODEL AVAILABILITY AUDIT")
print("=" * 88)

SAVE_DIR = "/content/drive/MyDrive/SolarLoad_MTL_Study"

assert os.path.isdir(SAVE_DIR), (
    f"Project directory not found: {SAVE_DIR}"
)

# ------------------------------------------------------------
# 1. IMPORTANT FROZEN FILES
# ------------------------------------------------------------

important_files = [

    # Dataset / preprocessing
    "experiment_manifest.json",
    "feature_scaler.joblib",
    "target_scaler.joblib",
    "frozen_168h_sequences.npz",
    "sequence_audit.csv",
    "split_audit.csv",

    # Baselines / development
    "baseline_results.csv",
    "context_screening_results.csv",
    "context_balanced_scores.csv",

    # LSTM development
    "lstm_validation_results.csv",
    "lstm_training_history.csv",
    "lstm_168h_seed42.keras",

    # Reproducibility / weighting
    "five_seed_validation_runs.csv",
    "five_seed_validation_summary.csv",
    "paired_mtl_vs_st_validation.csv",
    "weighted_mtl_validation_runs.csv",
    "mtl_loss_weight_all_runs.csv",
    "mtl_loss_weight_summary.csv",
    "mtl_loss_weight_balanced_runs.csv",
    "mtl_loss_weight_balanced_summary.csv",

    # Final locked test
    "final_model_selection_freeze.json",
    "final_test_five_seed_runs.csv",
    "final_test_five_seed_summary.csv",
    "final_test_predictions.csv",
    "final_model_efficiency.csv",
    "final_test_mtl_vs_st.csv",

    # Statistics
    "final_statistical_seed_comparisons.csv",
    "final_statistical_comparisons.csv",
    "final_rmse_block_bootstrap.csv",

    # Shift analysis
    "temporal_distribution_shift_audit.csv",
    "temporal_distribution_shift_monthly.csv",

    # Shift-error coupling
    "monthly_test_error_by_seed.csv",
    "monthly_test_error_summary.csv",
    "shift_error_coupling.csv",
    "monthly_relative_model_performance.csv",
    "shift_relative_performance_coupling.csv",
    "shift_error_association_summary.csv"
]


audit_rows = []

for name in important_files:

    path = os.path.join(
        SAVE_DIR,
        name
    )

    exists = os.path.isfile(path)

    size = (
        os.path.getsize(path)
        if exists
        else 0
    )

    audit_rows.append({

        "File": name,
        "Exists": exists,
        "Size_bytes": size,
        "Size_MB": size / (1024 ** 2)
    })


audit = pd.DataFrame(
    audit_rows
)


print("\n" + "=" * 88)
print("FROZEN ARTIFACT CHECK")
print("=" * 88)

print(
    audit.to_string(

        index=False,

        formatters={
            "Size_MB":
                lambda x: f"{x:.3f}"
        }
    )
)


missing = audit[
    ~audit["Exists"]
]


print("\nArtifact summary:")
print(
    f"Present : "
    f"{int(audit['Exists'].sum())}"
    f"/{len(audit)}"
)

print(
    f"Missing : "
    f"{len(missing)}"
)


# ------------------------------------------------------------
# 2. SEARCH RECURSIVELY FOR MODEL FILES
# ------------------------------------------------------------

model_extensions = [
    "*.keras",
    "*.h5",
    "*.hdf5",
    "*.weights.h5",
    "*.ckpt",
    "*.pt",
    "*.pth"
]

model_files = []

for pattern in model_extensions:

    model_files.extend(

        glob.glob(
            os.path.join(
                SAVE_DIR,
                "**",
                pattern
            ),
            recursive=True
        )
    )


# Include TensorFlow checkpoint index files separately.
model_files.extend(

    glob.glob(
        os.path.join(
            SAVE_DIR,
            "**",
            "*.index"
        ),
        recursive=True
    )
)


model_files = sorted(
    set(model_files)
)


print("\n" + "=" * 88)
print("MODEL / WEIGHT FILES FOUND")
print("=" * 88)

if len(model_files) == 0:

    print(
        "No saved model/weight files found."
    )

else:

    for path in model_files:

        relative = os.path.relpath(
            path,
            SAVE_DIR
        )

        size_mb = (
            os.path.getsize(path)
            / (1024 ** 2)
        )

        print(
            f"{relative:<70} "
            f"{size_mb:>10.3f} MB"
        )


print(
    f"\nTotal model/weight files found: "
    f"{len(model_files)}"
)


# ------------------------------------------------------------
# 3. LOOK SPECIFICALLY FOR FINAL FIVE-SEED MODELS
# ------------------------------------------------------------

seed_tokens = [
    "1",
    "7",
    "21",
    "42",
    "100"
]

keywords = [
    "final",
    "seed",
    "lstm",
    "st",
    "transformer",
    "mtl"
]


candidate_final_models = []

for path in model_files:

    name = os.path.basename(
        path
    ).lower()

    if any(
        keyword in name
        for keyword in keywords
    ):

        candidate_final_models.append(
            path
        )


print("\n" + "=" * 88)
print("CANDIDATE FINAL MODEL FILES")
print("=" * 88)

if not candidate_final_models:

    print(
        "No obvious final five-seed model files detected."
    )

else:

    for path in candidate_final_models:

        print(
            os.path.relpath(
                path,
                SAVE_DIR
            )
        )


# ------------------------------------------------------------
# 4. CHECK WHETHER ALL FIVE SEEDS APPEAR IN SAVED MODEL NAMES
# ------------------------------------------------------------

print("\n" + "=" * 88)
print("SEED TOKEN AUDIT IN MODEL FILENAMES")
print("=" * 88)

basename_text = "\n".join(
    [
        os.path.basename(path).lower()
        for path in model_files
    ]
)


for seed in seed_tokens:

    possible_tokens = [
        f"seed{seed}",
        f"seed_{seed}",
        f"seed-{seed}"
    ]

    found = any(
        token in basename_text
        for token in possible_tokens
    )

    print(
        f"Seed {seed:<3}: "
        f"{'FOUND' if found else 'NOT FOUND'}"
    )


# ------------------------------------------------------------
# 5. VERIFY FINAL PREDICTION FILE
# ------------------------------------------------------------

prediction_file = os.path.join(
    SAVE_DIR,
    "final_test_predictions.csv"
)

print("\n" + "=" * 88)
print("FINAL PREDICTION EVIDENCE")
print("=" * 88)

if os.path.isfile(prediction_file):

    p = pd.read_csv(
        prediction_file,
        usecols=[
            "Seed",
            "Model",
            "Target"
        ]
    )

    print(
        f"Rows: {len(p):,}"
    )

    print(
        "Seeds:",
        sorted(
            p["Seed"]
            .unique()
            .tolist()
        )
    )

    print(
        "Models:",
        sorted(
            p["Model"]
            .unique()
            .tolist()
        )
    )

    print(
        "Targets:",
        sorted(
            p["Target"]
            .unique()
            .tolist()
        )
    )

    assert len(p) == 197280

    assert sorted(
        p["Seed"].unique().tolist()
    ) == [1, 7, 21, 42, 100]

    print(
        "\n✓ Complete locked prediction evidence "
        "is preserved."
    )

else:

    print(
        "WARNING: final_test_predictions.csv "
        "not found."
    )


# ------------------------------------------------------------
# 6. SAVE AUDIT
# ------------------------------------------------------------

audit_file = os.path.join(
    SAVE_DIR,
    "step_5_0A_artifact_audit.csv"
)

audit.to_csv(
    audit_file,
    index=False
)


print("\n" + "=" * 88)
print("DECISION INFORMATION")
print("=" * 88)

print(
    f"Model/weight files available: "
    f"{len(model_files)}"
)

print(
    f"Candidate final model files: "
    f"{len(candidate_final_models)}"
)

print(
    "\nDo NOT retrain anything yet."
)

print(
    "The next robustness protocol will be chosen "
    "from this audit."
)

print("\nSaved:")
print(
    " ✓ step_5_0A_artifact_audit.csv"
)

print("\n" + "=" * 88)
print("STEP 5.0A COMPLETE")
print("=" * 88)

STEP 5.0A — FINAL ARTIFACT & MODEL AVAILABILITY AUDIT

FROZEN ARTIFACT CHECK
                                   File  Exists  Size_bytes Size_MB
               experiment_manifest.json    True        2809   0.003
                  feature_scaler.joblib    True         983   0.001
                   target_scaler.joblib    True         983   0.001
              frozen_168h_sequences.npz    True     2642244   2.520
                     sequence_audit.csv    True         770   0.001
                        split_audit.csv    True         601   0.001
                   baseline_results.csv    True         799   0.001
          context_screening_results.csv    True         752   0.001
            context_balanced_scores.csv    True         241   0.000
            lstm_validation_results.csv    True         355   0.000
              lstm_training_history.csv    True        4545   0.004
                 lstm_168h_seed42.keras    True      453118   0.432
          five_seed_validation_runs.csv

Step 5.0B — Reconstruct and Save the Frozen Robustness Models

In [15]:
# ============================================================
# STEP 5.0B — ROBUSTNESS MODEL RECONSTRUCTION + CHECKPOINTING
#
# PURPOSE:
# Reconstruct a compact, predeclared 3-seed model set for
# controlled post-training input-degradation analysis.
#
# Seeds:
#   1, 42, 100
#
# Architectures:
#   1. Multi-output LSTM
#   2. Single-task Transformer — Load
#   3. Single-task Transformer — Solar
#   4. Weighted MTL Transformer — Load:Solar = 2:1
#
# IMPORTANT:
# - Uses frozen 168h sequences.
# - Uses frozen training/validation sets.
# - Test set is NOT used for model selection.
# - Every trained model is saved immediately.
# - No degradation is applied in this step.
# ============================================================

import os
import gc
import json
import time
import random

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

print("=" * 92)
print("STEP 5.0B — ROBUSTNESS MODEL RECONSTRUCTION + CHECKPOINTING")
print("=" * 92)

SAVE_DIR = "/content/drive/MyDrive/SolarLoad_MTL_Study"

ROBUST_DIR = os.path.join(
    SAVE_DIR,
    "robustness_models"
)

os.makedirs(
    ROBUST_DIR,
    exist_ok=True
)

NPZ_FILE = os.path.join(
    SAVE_DIR,
    "frozen_168h_sequences.npz"
)

SEEDS = [1, 42, 100]

# ------------------------------------------------------------
# FROZEN CONFIGURATION
# ------------------------------------------------------------

SEQ_LEN = 168
N_FEATURES = 8

D_MODEL = 64
NUM_HEADS = 4
FF_DIM = 128
NUM_BLOCKS = 2

DROPOUT = 0.20

LEARNING_RATE = 1e-3
BATCH_SIZE = 64
MAX_EPOCHS = 50
PATIENCE = 7

MTL_LOAD_WEIGHT = 2.0
MTL_SOLAR_WEIGHT = 1.0


# ============================================================
# 1. LOAD FROZEN SEQUENCES
# ============================================================

print("\nLoading frozen 168h sequences...")

data = np.load(
    NPZ_FILE
)

print("Available NPZ keys:")
print(data.files)

# Flexible key recovery.
def recover_array(possible_names):

    for name in possible_names:

        if name in data.files:
            return data[name]

    raise KeyError(
        f"None of these keys found: {possible_names}"
    )


X_train = recover_array([
    "X_train",
    "X_train_168"
])

y_train = recover_array([
    "y_train",
    "y_train_168"
])

X_val = recover_array([
    "X_val",
    "X_validation",
    "X_val_168"
])

y_val = recover_array([
    "y_val",
    "y_validation",
    "y_val_168"
])

X_test = recover_array([
    "X_test",
    "X_test_168"
])

y_test = recover_array([
    "y_test",
    "y_test_168"
])


print("\nFrozen arrays:")
print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)

assert X_train.shape == (34121, 168, 8)
assert y_train.shape == (34121, 2)

assert X_val.shape == (8760, 168, 8)
assert y_val.shape == (8760, 2)

assert X_test.shape == (6576, 168, 8)
assert y_test.shape == (6576, 2)


# ============================================================
# 2. DETERMINISM
# ============================================================

def set_seed(seed):

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


# ============================================================
# 3. POSITIONAL ENCODING
# ============================================================

@keras.utils.register_keras_serializable(
    package="SolarLoadStudy"
)
class PositionalEncoding(layers.Layer):

    def __init__(
        self,
        sequence_length,
        d_model,
        **kwargs
    ):

        super().__init__(**kwargs)

        self.sequence_length = sequence_length
        self.d_model = d_model

        position = np.arange(
            sequence_length
        )[:, np.newaxis]

        div_term = np.exp(
            np.arange(
                0,
                d_model,
                2
            )
            *
            (
                -np.log(10000.0)
                / d_model
            )
        )

        pe = np.zeros(
            (
                sequence_length,
                d_model
            ),
            dtype=np.float32
        )

        pe[:, 0::2] = np.sin(
            position * div_term
        )

        pe[:, 1::2] = np.cos(
            position * div_term
        )

        self.pe = tf.constant(
            pe[np.newaxis, ...],
            dtype=tf.float32
        )


    def call(self, inputs):

        return (
            inputs
            +
            self.pe[
                :,
                :tf.shape(inputs)[1],
                :
            ]
        )


    def get_config(self):

        config = super().get_config()

        config.update({
            "sequence_length":
                self.sequence_length,

            "d_model":
                self.d_model
        })

        return config


# ============================================================
# 4. TRANSFORMER BLOCK
# ============================================================

def transformer_block(
    x,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    ff_dim=FF_DIM,
    dropout=DROPOUT
):

    attn = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads,
        dropout=dropout
    )(
        x,
        x
    )

    attn = layers.Dropout(
        dropout
    )(attn)

    x = layers.LayerNormalization(
        epsilon=1e-6
    )(x + attn)

    ff = layers.Dense(
        ff_dim,
        activation="gelu"
    )(x)

    ff = layers.Dropout(
        dropout
    )(ff)

    ff = layers.Dense(
        d_model
    )(ff)

    ff = layers.Dropout(
        dropout
    )(ff)

    return layers.LayerNormalization(
        epsilon=1e-6
    )(x + ff)


# ============================================================
# 5. MODEL BUILDERS
# ============================================================

def build_lstm():

    inputs = keras.Input(
        shape=(SEQ_LEN, N_FEATURES),
        name="input"
    )

    x = layers.LSTM(
        64,
        return_sequences=True
    )(inputs)

    x = layers.Dropout(
        DROPOUT
    )(x)

    x = layers.LSTM(
        32
    )(x)

    x = layers.Dropout(
        DROPOUT
    )(x)

    x = layers.Dense(
        64,
        activation="gelu"
    )(x)

    x = layers.Dropout(
        DROPOUT
    )(x)

    load_output = layers.Dense(
        1,
        name="load"
    )(x)

    solar_output = layers.Dense(
        1,
        name="solar"
    )(x)

    model = keras.Model(
        inputs,
        [
            load_output,
            solar_output
        ],
        name="LSTM_multioutput"
    )

    model.compile(

        optimizer=keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),

        loss={
            "load": "mse",
            "solar": "mse"
        }
    )

    return model


def transformer_backbone():

    inputs = keras.Input(
        shape=(SEQ_LEN, N_FEATURES),
        name="input"
    )

    x = layers.Dense(
        D_MODEL
    )(inputs)

    x = PositionalEncoding(
        SEQ_LEN,
        D_MODEL
    )(x)

    for _ in range(NUM_BLOCKS):

        x = transformer_block(x)

    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(
        64,
        activation="gelu"
    )(x)

    x = layers.Dropout(
        DROPOUT
    )(x)

    return inputs, x


def build_single_task(
    target_name
):

    inputs, x = transformer_backbone()

    output = layers.Dense(
        1,
        name=target_name
    )(x)

    model = keras.Model(
        inputs,
        output,
        name=f"ST_{target_name}"
    )

    model.compile(

        optimizer=keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),

        loss="mse"
    )

    return model


def build_mtl():

    inputs, x = transformer_backbone()

    load_output = layers.Dense(
        1,
        name="load"
    )(x)

    solar_output = layers.Dense(
        1,
        name="solar"
    )(x)

    model = keras.Model(
        inputs,
        [
            load_output,
            solar_output
        ],
        name="MTL_Transformer_2to1"
    )

    model.compile(

        optimizer=keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),

        loss={
            "load": "mse",
            "solar": "mse"
        },

        loss_weights={
            "load": MTL_LOAD_WEIGHT,
            "solar": MTL_SOLAR_WEIGHT
        }
    )

    return model


# ============================================================
# 6. CALLBACK
# ============================================================

def make_callbacks():

    return [

        keras.callbacks.EarlyStopping(

            monitor="val_loss",

            patience=PATIENCE,

            restore_best_weights=True,

            verbose=0
        )
    ]


# ============================================================
# 7. TRAIN + IMMEDIATE CHECKPOINT
# ============================================================

run_rows = []


for seed in SEEDS:

    print("\n" + "=" * 92)
    print(f"SEED {seed}")
    print("=" * 92)


    # --------------------------------------------------------
    # A. LSTM
    # --------------------------------------------------------

    tf.keras.backend.clear_session()
    gc.collect()

    set_seed(seed)

    model = build_lstm()

    print(
        f"\nTraining LSTM — seed {seed}"
    )

    start = time.time()

    history = model.fit(

        X_train,

        {
            "load": y_train[:, 0],
            "solar": y_train[:, 1]
        },

        validation_data=(

            X_val,

            {
                "load": y_val[:, 0],
                "solar": y_val[:, 1]
            }
        ),

        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,

        callbacks=make_callbacks(),

        verbose=0
    )

    elapsed = time.time() - start

    best_epoch = (
        int(
            np.argmin(
                history.history["val_loss"]
            )
        )
        + 1
    )

    path = os.path.join(
        ROBUST_DIR,
        f"robust_LSTM_seed_{seed}.keras"
    )

    model.save(path)

    run_rows.append({
        "Seed": seed,
        "Model": "LSTM",
        "Best_epoch": best_epoch,
        "Training_seconds": elapsed,
        "Parameters": model.count_params(),
        "Saved_path": path
    })

    print(
        f"✓ saved: {os.path.basename(path)}"
    )


    del model
    gc.collect()


    # --------------------------------------------------------
    # B. SINGLE-TASK LOAD TRANSFORMER
    # --------------------------------------------------------

    tf.keras.backend.clear_session()
    gc.collect()

    set_seed(seed)

    model = build_single_task(
        "load"
    )

    print(
        f"\nTraining ST Load — seed {seed}"
    )

    start = time.time()

    history = model.fit(

        X_train,
        y_train[:, 0],

        validation_data=(
            X_val,
            y_val[:, 0]
        ),

        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,

        callbacks=make_callbacks(),

        verbose=0
    )

    elapsed = time.time() - start

    best_epoch = (
        int(
            np.argmin(
                history.history["val_loss"]
            )
        )
        + 1
    )

    path = os.path.join(
        ROBUST_DIR,
        f"robust_ST_Load_seed_{seed}.keras"
    )

    model.save(path)

    run_rows.append({
        "Seed": seed,
        "Model": "ST-Load",
        "Best_epoch": best_epoch,
        "Training_seconds": elapsed,
        "Parameters": model.count_params(),
        "Saved_path": path
    })

    print(
        f"✓ saved: {os.path.basename(path)}"
    )


    del model
    gc.collect()


    # --------------------------------------------------------
    # C. SINGLE-TASK SOLAR TRANSFORMER
    # --------------------------------------------------------

    tf.keras.backend.clear_session()
    gc.collect()

    set_seed(seed)

    model = build_single_task(
        "solar"
    )

    print(
        f"\nTraining ST Solar — seed {seed}"
    )

    start = time.time()

    history = model.fit(

        X_train,
        y_train[:, 1],

        validation_data=(
            X_val,
            y_val[:, 1]
        ),

        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,

        callbacks=make_callbacks(),

        verbose=0
    )

    elapsed = time.time() - start

    best_epoch = (
        int(
            np.argmin(
                history.history["val_loss"]
            )
        )
        + 1
    )

    path = os.path.join(
        ROBUST_DIR,
        f"robust_ST_Solar_seed_{seed}.keras"
    )

    model.save(path)

    run_rows.append({
        "Seed": seed,
        "Model": "ST-Solar",
        "Best_epoch": best_epoch,
        "Training_seconds": elapsed,
        "Parameters": model.count_params(),
        "Saved_path": path
    })

    print(
        f"✓ saved: {os.path.basename(path)}"
    )


    del model
    gc.collect()


    # --------------------------------------------------------
    # D. WEIGHTED MTL TRANSFORMER
    # --------------------------------------------------------

    tf.keras.backend.clear_session()
    gc.collect()

    set_seed(seed)

    model = build_mtl()

    print(
        f"\nTraining weighted MTL 2:1 — seed {seed}"
    )

    start = time.time()

    history = model.fit(

        X_train,

        {
            "load": y_train[:, 0],
            "solar": y_train[:, 1]
        },

        validation_data=(

            X_val,

            {
                "load": y_val[:, 0],
                "solar": y_val[:, 1]
            }
        ),

        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,

        callbacks=make_callbacks(),

        verbose=0
    )

    elapsed = time.time() - start

    best_epoch = (
        int(
            np.argmin(
                history.history["val_loss"]
            )
        )
        + 1
    )

    path = os.path.join(
        ROBUST_DIR,
        f"robust_MTL_2to1_seed_{seed}.keras"
    )

    model.save(path)

    run_rows.append({
        "Seed": seed,
        "Model": "MTL-2:1",
        "Best_epoch": best_epoch,
        "Training_seconds": elapsed,
        "Parameters": model.count_params(),
        "Saved_path": path
    })

    print(
        f"✓ saved: {os.path.basename(path)}"
    )


    del model
    gc.collect()


# ============================================================
# 8. SAVE RECONSTRUCTION AUDIT
# ============================================================

runs = pd.DataFrame(
    run_rows
)

RUN_FILE = os.path.join(
    SAVE_DIR,
    "robustness_model_reconstruction_runs.csv"
)

runs.to_csv(
    RUN_FILE,
    index=False
)


print("\n" + "=" * 92)
print("ROBUSTNESS MODEL RECONSTRUCTION SUMMARY")
print("=" * 92)

print(
    runs[
        [
            "Seed",
            "Model",
            "Best_epoch",
            "Training_seconds",
            "Parameters"
        ]
    ].to_string(

        index=False,

        formatters={
            "Training_seconds":
                lambda x: f"{x:.2f}"
        }
    )
)


# ============================================================
# 9. VERIFY ALL 12 MODEL FILES
# ============================================================

print("\n" + "=" * 92)
print("CHECKPOINT VERIFICATION")
print("=" * 92)

expected = []

for seed in SEEDS:

    expected.extend([

        f"robust_LSTM_seed_{seed}.keras",

        f"robust_ST_Load_seed_{seed}.keras",

        f"robust_ST_Solar_seed_{seed}.keras",

        f"robust_MTL_2to1_seed_{seed}.keras"
    ])


missing = []

for name in expected:

    path = os.path.join(
        ROBUST_DIR,
        name
    )

    exists = os.path.isfile(path)

    size_mb = (
        os.path.getsize(path)
        / (1024 ** 2)
        if exists
        else 0
    )

    print(
        f"{name:<45} "
        f"{'OK' if exists else 'MISSING':<8} "
        f"{size_mb:.3f} MB"
    )

    if not exists:
        missing.append(name)


assert len(missing) == 0, (
    f"Missing robustness models: {missing}"
)


# ============================================================
# 10. FREEZE ROBUSTNESS PROTOCOL
# ============================================================

protocol = {

    "purpose":
        "post-training controlled input degradation",

    "context_hours":
        168,

    "forecast_horizon_hours":
        1,

    "training_data":
        "clean frozen training sequences",

    "validation_data":
        "clean frozen validation sequences",

    "test_data":
        "frozen 2020 Jan-Sep test sequences",

    "seeds":
        SEEDS,

    "architectures": [
        "LSTM multi-output",
        "ST Transformer Load",
        "ST Transformer Solar",
        "weighted MTL Transformer 2:1"
    ],

    "degradation_levels": [
        0.0,
        0.10,
        0.30,
        0.50
    ],

    "degraded_features": [
        "historical Load",
        "historical Solar"
    ],

    "replacement":
        "training mean = 0 in standardized dynamic-feature space",

    "calendar_features":
        "unchanged",

    "mask_policy":
        "fixed shared masks across architectures and seeds",

    "selection":
        "none; protocol frozen before degradation evaluation"
}


PROTOCOL_FILE = os.path.join(
    SAVE_DIR,
    "robustness_protocol_freeze.json"
)

with open(
    PROTOCOL_FILE,
    "w"
) as f:

    json.dump(
        protocol,
        f,
        indent=2
    )


print("\n✓ All 12 robustness model files preserved.")
print("✓ Robustness protocol frozen BEFORE degradation evaluation.")

print("\nSaved:")
print(" ✓ robustness_model_reconstruction_runs.csv")
print(" ✓ robustness_protocol_freeze.json")
print(" ✓ robustness_models/*.keras")

print("\nIMPORTANT:")
print(
    "Do NOT run degradation evaluation yet "
    "until this checkpoint is verified."
)

print("\n" + "=" * 92)
print("STEP 5.0B COMPLETE")
print("=" * 92)

STEP 5.0B — ROBUSTNESS MODEL RECONSTRUCTION + CHECKPOINTING

Loading frozen 168h sequences...
Available NPZ keys:
['X_train', 'y_train', 'X_val', 'y_val', 'X_test', 'y_test']

Frozen arrays:
Train: (34121, 168, 8) (34121, 2)
Val  : (8760, 168, 8) (8760, 2)
Test : (6576, 168, 8) (6576, 2)

SEED 1

Training LSTM — seed 1
✓ saved: robust_LSTM_seed_1.keras

Training ST Load — seed 1
✓ saved: robust_ST_Load_seed_1.keras

Training ST Solar — seed 1
✓ saved: robust_ST_Solar_seed_1.keras

Training weighted MTL 2:1 — seed 1
✓ saved: robust_MTL_2to1_seed_1.keras

SEED 42

Training LSTM — seed 42
✓ saved: robust_LSTM_seed_42.keras

Training ST Load — seed 42
✓ saved: robust_ST_Load_seed_42.keras

Training ST Solar — seed 42
✓ saved: robust_ST_Solar_seed_42.keras

Training weighted MTL 2:1 — seed 42
✓ saved: robust_MTL_2to1_seed_42.keras

SEED 100

Training LSTM — seed 100
✓ saved: robust_LSTM_seed_100.keras

Training ST Load — seed 100
✓ saved: robust_ST_Load_seed_100.keras

Training ST Solar — s

Step 5.0C — Controlled Input-Degradation Stress Test

In [17]:
# ============================================================
# STEP 5.0C-FIX — COMPLETE ROBUSTNESS RESULT PROCESSING
#
# IMPORTANT:
# - NO TRAINING
# - NO MODEL LOADING
# - NO NEW PREDICTIONS
#
# This continues from the already-completed Step 5.0C
# evaluation and fixes the Mask_rate sorting/display error.
# ============================================================

import os
import numpy as np
import pandas as pd

print("=" * 96)
print("STEP 5.0C-FIX — COMPLETE ROBUSTNESS RESULT PROCESSING")
print("=" * 96)

SAVE_DIR = "/content/drive/MyDrive/SolarLoad_MTL_Study"


# ============================================================
# 1. VERIFY EXISTING RESULTS
# ============================================================

print("\nChecking existing Step 5.0C objects...")

assert "results" in globals(), (
    "The object 'results' is not in memory. "
    "Do NOT retrain. Load robustness_degradation_runs.csv instead."
)

assert "summary" in globals(), (
    "The object 'summary' is not in memory. "
    "Do NOT retrain. Load robustness_degradation_summary.csv instead."
)

print(f"Run-level rows : {len(results)}")
print(f"Summary rows   : {len(summary)}")

assert len(results) == 72, (
    f"Expected 72 run-level rows, found {len(results)}."
)

assert len(summary) == 24, (
    f"Expected 24 summary rows, found {len(summary)}."
)

print("✓ Existing robustness evaluation recovered.")


# ============================================================
# 2. RECONSTRUCT CLEAN-REFERENCE DEGRADATION TABLE
#
# Recalculate this safely rather than depending on the
# partially executed previous object.
# ============================================================

clean_reference = (
    summary[
        summary["Condition"] == "Clean"
    ][
        [
            "Model",
            "Target",
            "RMSE_mean"
        ]
    ]
    .rename(
        columns={
            "RMSE_mean": "Clean_RMSE"
        }
    )
)

assert len(clean_reference) == 6


degradation = summary.merge(
    clean_reference,
    on=[
        "Model",
        "Target"
    ],
    how="left",
    validate="many_to_one"
)


degradation[
    "RMSE_increase_pct"
] = (
    100.0
    *
    (
        degradation["RMSE_mean"]
        -
        degradation["Clean_RMSE"]
    )
    /
    degradation["Clean_RMSE"]
)


# Clean condition must equal 0% degradation.
clean_check = degradation[
    degradation["Condition"] == "Clean"
][
    "RMSE_increase_pct"
].to_numpy()

assert np.allclose(
    clean_check,
    0.0,
    atol=1e-10
)

print(
    "✓ Clean-reference RMSE degradation "
    "recalculated successfully."
)


# ============================================================
# 3. SAVE RELATIVE DEGRADATION
# ============================================================

DEGRADATION_FILE = os.path.join(
    SAVE_DIR,
    "robustness_relative_degradation.csv"
)

degradation.to_csv(
    DEGRADATION_FILE,
    index=False
)

print(
    "✓ robustness_relative_degradation.csv saved."
)


# ============================================================
# 4. CORRECTED RMSE DEGRADATION DISPLAY
#
# FIX:
# Sort while Mask_rate is still present.
# ============================================================

degradation_display = (
    degradation
    .sort_values(
        [
            "Target",
            "Model",
            "Mask_rate"
        ]
    )
    [
        [
            "Target",
            "Model",
            "Condition",
            "Mask_rate",
            "RMSE_mean",
            "RMSE_std",
            "Clean_RMSE",
            "RMSE_increase_pct"
        ]
    ]
)


print("\n" + "=" * 96)
print("RMSE INCREASE RELATIVE TO CLEAN")
print("=" * 96)

print(
    degradation_display.to_string(
        index=False,
        formatters={
            "Mask_rate":
                lambda x: f"{x:.0%}",

            "RMSE_mean":
                lambda x: f"{x:.3f}",

            "RMSE_std":
                lambda x: f"{x:.3f}",

            "Clean_RMSE":
                lambda x: f"{x:.3f}",

            "RMSE_increase_pct":
                lambda x: f"{x:+.2f}%"
        }
    )
)


# ============================================================
# 5. BUILD MTL RELATIVE PERFORMANCE TABLE
#
# Positive value:
#   MTL has lower RMSE than comparator.
#
# Negative value:
#   Comparator has lower RMSE than MTL.
# ============================================================

pivot = summary.pivot_table(
    index=[
        "Target",
        "Condition",
        "Mask_rate"
    ],
    columns="Model",
    values="RMSE_mean"
).reset_index()

pivot.columns.name = None


required_models = [
    "LSTM",
    "ST-Transformer",
    "MTL-Transformer-2:1"
]

for model_name in required_models:

    assert model_name in pivot.columns, (
        f"Missing model column: {model_name}"
    )


pivot[
    "MTL_vs_LSTM_improvement_pct"
] = (
    100.0
    *
    (
        pivot["LSTM"]
        -
        pivot["MTL-Transformer-2:1"]
    )
    /
    pivot["LSTM"]
)


pivot[
    "MTL_vs_ST_improvement_pct"
] = (
    100.0
    *
    (
        pivot["ST-Transformer"]
        -
        pivot["MTL-Transformer-2:1"]
    )
    /
    pivot["ST-Transformer"]
)


pivot = pivot.sort_values(
    [
        "Target",
        "Mask_rate"
    ]
).reset_index(
    drop=True
)


# ============================================================
# 6. SAVE MTL RELATIVE PERFORMANCE
# ============================================================

RELATIVE_FILE = os.path.join(
    SAVE_DIR,
    "robustness_mtl_relative_performance.csv"
)

pivot.to_csv(
    RELATIVE_FILE,
    index=False
)


print("\n" + "=" * 96)
print("MTL RELATIVE PERFORMANCE UNDER DEGRADATION")
print("=" * 96)

print(
    pivot.to_string(
        index=False,
        formatters={
            "Mask_rate":
                lambda x: f"{x:.0%}",

            "LSTM":
                lambda x: f"{x:.3f}",

            "ST-Transformer":
                lambda x: f"{x:.3f}",

            "MTL-Transformer-2:1":
                lambda x: f"{x:.3f}",

            "MTL_vs_LSTM_improvement_pct":
                lambda x: f"{x:+.2f}%",

            "MTL_vs_ST_improvement_pct":
                lambda x: f"{x:+.2f}%"
        }
    )
)


# ============================================================
# 7. SAVE / VERIFY MASK AUDIT
# ============================================================

MASK_FILE = os.path.join(
    SAVE_DIR,
    "robustness_mask_audit.csv"
)

if "mask_audit_rows" in globals():

    mask_audit = pd.DataFrame(
        mask_audit_rows
    )

    mask_audit.to_csv(
        MASK_FILE,
        index=False
    )

else:

    # Exact values from completed Step 5.0C output.
    mask_audit = pd.DataFrame({

        "Condition": [
            "Clean",
            "Mild_10pct",
            "Moderate_30pct",
            "Severe_50pct"
        ],

        "Requested_rate": [
            0.00,
            0.10,
            0.30,
            0.50
        ],

        "Actual_rate": [
            0.00000,
            0.10003,
            0.29989,
            0.49955
        ],

        "Masked_positions": [
            0,
            110505,
            331308,
            551890
        ],

        "Total_positions": [
            6576 * 168,
            6576 * 168,
            6576 * 168,
            6576 * 168
        ]
    })

    mask_audit.to_csv(
        MASK_FILE,
        index=False
    )

print(
    "✓ robustness_mask_audit.csv preserved."
)


# ============================================================
# 8. RE-SAVE PRIMARY RESULT TABLES
# ============================================================

RUN_FILE = os.path.join(
    SAVE_DIR,
    "robustness_degradation_runs.csv"
)

SUMMARY_FILE = os.path.join(
    SAVE_DIR,
    "robustness_degradation_summary.csv"
)

results.to_csv(
    RUN_FILE,
    index=False
)

summary.to_csv(
    SUMMARY_FILE,
    index=False
)

print(
    "✓ Run-level and summary results preserved."
)


# ============================================================
# 9. PRESERVE STRESS PREDICTIONS
# ============================================================

PRED_FILE = os.path.join(
    SAVE_DIR,
    "robustness_degradation_predictions.csv"
)

if "stress_predictions" in globals():

    print(
        f"\nStress predictions in memory: "
        f"{len(stress_predictions):,}"
    )

    assert len(
        stress_predictions
    ) == 473472

    stress_predictions.to_csv(
        PRED_FILE,
        index=False
    )

    print(
        "✓ 473,472 stress predictions saved."
    )

elif os.path.isfile(PRED_FILE):

    print(
        "\n✓ Stress prediction file already "
        "exists in Drive."
    )

else:

    print(
        "\nWARNING: stress prediction object/file "
        "not found."
    )


# ============================================================
# 10. FINAL RESULT INTEGRITY
# ============================================================

assert len(results) == 72


seed_integrity = (
    results
    .groupby(
        [
            "Model",
            "Target",
            "Condition"
        ]
    )["Seed"]
    .nunique()
)

assert seed_integrity.eq(
    3
).all()


assert np.isfinite(
    results[
        [
            "MAE",
            "RMSE",
            "R2"
        ]
    ].to_numpy()
).all()


assert len(pivot) == 8

assert len(degradation) == 24


print("\n✓ 72 run-level results verified.")
print(
    "✓ Exactly 3 seeds per "
    "model × target × condition verified."
)
print("✓ All metrics finite.")
print("✓ 24 degradation-summary rows verified.")
print("✓ 8 relative-comparison rows verified.")


# ============================================================
# 11. CHECK ALL EXPECTED FILES
# ============================================================

expected_files = [

    "robustness_mask_audit.csv",

    "robustness_degradation_runs.csv",

    "robustness_degradation_summary.csv",

    "robustness_relative_degradation.csv",

    "robustness_mtl_relative_performance.csv"
]


print("\n" + "=" * 96)
print("FILE CHECK")
print("=" * 96)

for filename in expected_files:

    path = os.path.join(
        SAVE_DIR,
        filename
    )

    exists = os.path.isfile(
        path
    )

    size = (
        os.path.getsize(path)
        if exists
        else 0
    )

    print(
        f"{filename:<52} "
        f"{'OK' if exists else 'MISSING':<8} "
        f"{size:,} bytes"
    )

    assert exists


# ============================================================
# 12. FINAL CHECKPOINT
# ============================================================

print("\n" + "=" * 96)
print("FINAL CHECKPOINT")
print("=" * 96)

print("\n✓ NO retraining performed.")
print("✓ NO models reloaded.")
print("✓ NO new predictions generated.")
print(
    "✓ Existing Step 5.0C evaluation "
    "was retained."
)
print(
    "✓ Mask_rate sorting/display bug "
    "was corrected."
)
print(
    "✓ Clean-relative degradation "
    "was recalculated."
)
print(
    "✓ MTL relative-performance table "
    "was generated."
)
print(
    "✓ Robustness artifacts were "
    "preserved in Google Drive."
)

print("\nSaved:")
print(" ✓ robustness_mask_audit.csv")
print(" ✓ robustness_degradation_runs.csv")
print(" ✓ robustness_degradation_summary.csv")
print(" ✓ robustness_relative_degradation.csv")
print(" ✓ robustness_mtl_relative_performance.csv")

if os.path.isfile(PRED_FILE):
    print(
        " ✓ robustness_degradation_predictions.csv"
    )

print("\n" + "=" * 96)
print("STEP 5.0C-FIX COMPLETE")
print("=" * 96)

STEP 5.0C-FIX — COMPLETE ROBUSTNESS RESULT PROCESSING

Checking existing Step 5.0C objects...
Run-level rows : 72
Summary rows   : 24
✓ Existing robustness evaluation recovered.
✓ Clean-reference RMSE degradation recalculated successfully.
✓ robustness_relative_degradation.csv saved.

RMSE INCREASE RELATIVE TO CLEAN
Target               Model      Condition Mask_rate RMSE_mean RMSE_std Clean_RMSE RMSE_increase_pct
  Load                LSTM          Clean        0%   795.612   70.798    795.612            +0.00%
  Load                LSTM     Mild_10pct       10%  2991.585  111.588    795.612          +276.01%
  Load                LSTM Moderate_30pct       30%  5302.739  193.977    795.612          +566.50%
  Load                LSTM   Severe_50pct       50%  6918.373  275.637    795.612          +769.57%
  Load MTL-Transformer-2:1          Clean        0%   922.870    9.523    922.870            +0.00%
  Load MTL-Transformer-2:1     Mild_10pct       10%  3255.799  223.261    922.870 

Step 5.1 — Seven-Day Temporal Occlusion Analysis

In [18]:
# ============================================================
# STEP 5.1 — SEVEN-DAY TEMPORAL OCCLUSION EXPLAINABILITY
#
# PURPOSE
# ------------------------------------------------------------
# Determine which 24-hour portions of the frozen 168-hour
# history are most influential for Load and Solar forecasting.
#
# NO TRAINING.
# NO MODEL SELECTION.
#
# Robustness/explainability cohort:
#   Seeds = [1, 42, 100]
#
# Models:
#   LSTM
#   ST Transformer — Load
#   ST Transformer — Solar
#   Weighted MTL Transformer 2:1
#
# OCCLUSION:
#   Seven non-overlapping 24-hour blocks.
#
#   Day_-1 = most recent 24 h
#   Day_-7 = oldest 24 h
#
# Only dynamic channels are occluded:
#   feature 0 = Load
#   feature 1 = Solar
#
# Replacement = 0 standardized units
#             = training mean
#
# Calendar features remain unchanged.
#
# Interpretation:
# Larger RMSE increase after occlusion means that temporal
# block contains more predictive information for that
# model/target.
#
# IMPORTANT:
# This is perturbation-based sensitivity analysis.
# It is NOT causal feature importance.
# ============================================================

import os
import gc
import json
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib

from tensorflow import keras
from tensorflow.keras import layers


print("=" * 100)
print("STEP 5.1 — SEVEN-DAY TEMPORAL OCCLUSION EXPLAINABILITY")
print("=" * 100)


# ============================================================
# 1. PATHS / FROZEN CONFIGURATION
# ============================================================

SAVE_DIR = "/content/drive/MyDrive/SolarLoad_MTL_Study"

ROBUST_DIR = os.path.join(
    SAVE_DIR,
    "robustness_models"
)

NPZ_FILE = os.path.join(
    SAVE_DIR,
    "frozen_168h_sequences.npz"
)

TARGET_SCALER_FILE = os.path.join(
    SAVE_DIR,
    "target_scaler.joblib"
)

SEEDS = [1, 42, 100]

SEQ_LEN = 168
BLOCK_HOURS = 24
N_BLOCKS = 7

assert SEQ_LEN == BLOCK_HOURS * N_BLOCKS


# ============================================================
# 2. LOAD FROZEN TEST DATA
# ============================================================

data = np.load(NPZ_FILE)

X_test = data["X_test"].astype(
    np.float32
)

y_test = data["y_test"].astype(
    np.float32
)

assert X_test.shape == (
    6576,
    168,
    8
)

assert y_test.shape == (
    6576,
    2
)

print("\nFrozen test data:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)


# ============================================================
# 3. TARGET SCALER
# ============================================================

target_scaler = joblib.load(
    TARGET_SCALER_FILE
)

print("\nTarget scaler:")
print("Mean :", target_scaler.mean_)
print("Scale:", target_scaler.scale_)


def inverse_target(
    values,
    target_index
):

    values = np.asarray(
        values
    ).reshape(-1)

    return (
        values
        * target_scaler.scale_[target_index]
        +
        target_scaler.mean_[target_index]
    )


actual_load = inverse_target(
    y_test[:, 0],
    0
)

actual_solar = inverse_target(
    y_test[:, 1],
    1
)


# ============================================================
# 4. SERIALIZABLE POSITIONAL ENCODING
# ============================================================

@keras.utils.register_keras_serializable(
    package="SolarLoadStudy"
)
class PositionalEncoding(layers.Layer):

    def __init__(
        self,
        sequence_length,
        d_model,
        **kwargs
    ):

        super().__init__(**kwargs)

        self.sequence_length = sequence_length
        self.d_model = d_model

        position = np.arange(
            sequence_length
        )[:, np.newaxis]

        div_term = np.exp(
            np.arange(
                0,
                d_model,
                2
            )
            *
            (
                -np.log(10000.0)
                / d_model
            )
        )

        pe = np.zeros(
            (
                sequence_length,
                d_model
            ),
            dtype=np.float32
        )

        pe[:, 0::2] = np.sin(
            position * div_term
        )

        pe[:, 1::2] = np.cos(
            position * div_term
        )

        self.pe = tf.constant(
            pe[np.newaxis, ...],
            dtype=tf.float32
        )


    def call(
        self,
        inputs
    ):

        return (
            inputs
            +
            self.pe[
                :,
                :tf.shape(inputs)[1],
                :
            ]
        )


    def get_config(self):

        config = super().get_config()

        config.update({
            "sequence_length":
                self.sequence_length,
            "d_model":
                self.d_model
        })

        return config


def load_model_safe(path):

    return keras.models.load_model(
        path,
        custom_objects={
            "PositionalEncoding":
                PositionalEncoding
        },
        compile=False
    )


# ============================================================
# 5. METRICS
# ============================================================

def calculate_metrics(
    actual,
    predicted
):

    actual = np.asarray(
        actual
    ).reshape(-1)

    predicted = np.asarray(
        predicted
    ).reshape(-1)

    error = predicted - actual

    mae = np.mean(
        np.abs(error)
    )

    rmse = np.sqrt(
        np.mean(error ** 2)
    )

    ss_res = np.sum(
        error ** 2
    )

    ss_tot = np.sum(
        (
            actual
            -
            np.mean(actual)
        ) ** 2
    )

    r2 = (
        1.0
        -
        ss_res / ss_tot
    )

    return mae, rmse, r2


# ============================================================
# 6. DEFINE TEMPORAL OCCLUSION BLOCKS
#
# Array ordering:
#   index 0   = oldest historical hour
#   index 167 = most recent historical hour
#
# Therefore:
#
# Day_-1 = indices 144:168
# Day_-2 = indices 120:144
# ...
# Day_-7 = indices   0:24
# ============================================================

blocks = []

for day_back in range(
    1,
    N_BLOCKS + 1
):

    end = (
        SEQ_LEN
        -
        (day_back - 1)
        * BLOCK_HOURS
    )

    start = (
        end
        -
        BLOCK_HOURS
    )

    blocks.append({
        "Block":
            f"Day_-{day_back}",

        "Day_back":
            day_back,

        "Start_index":
            start,

        "End_index_exclusive":
            end,

        "Hours":
            BLOCK_HOURS
    })


block_df = pd.DataFrame(
    blocks
)

print("\n" + "=" * 100)
print("TEMPORAL OCCLUSION BLOCKS")
print("=" * 100)

print(
    block_df.to_string(
        index=False
    )
)

assert len(block_df) == 7
assert block_df["Hours"].eq(24).all()

assert (
    block_df["Start_index"].min()
    == 0
)

assert (
    block_df["End_index_exclusive"].max()
    == 168
)


BLOCK_FILE = os.path.join(
    SAVE_DIR,
    "temporal_occlusion_blocks.csv"
)

block_df.to_csv(
    BLOCK_FILE,
    index=False
)


# ============================================================
# 7. CREATE OCCLUDED INPUTS
#
# IMPORTANT:
# Only features 0:2 are changed.
# Calendar features 2:8 remain exactly unchanged.
# ============================================================

occluded_inputs = {
    "Clean": X_test
}


for block in blocks:

    X_occ = X_test.copy()

    start = block["Start_index"]
    end = block["End_index_exclusive"]

    # Replace historical Load + Solar by
    # training mean (= 0 standardized units).
    X_occ[
        :,
        start:end,
        0:2
    ] = 0.0

    # Calendar integrity.
    assert np.array_equal(
        X_occ[:, :, 2:],
        X_test[:, :, 2:]
    )

    # Dynamic values outside the selected block
    # must remain unchanged.
    if start > 0:

        assert np.array_equal(
            X_occ[:, :start, 0:2],
            X_test[:, :start, 0:2]
        )

    if end < SEQ_LEN:

        assert np.array_equal(
            X_occ[:, end:, 0:2],
            X_test[:, end:, 0:2]
        )

    occluded_inputs[
        block["Block"]
    ] = X_occ


print(
    "\n✓ Seven 24-hour occlusion conditions created."
)

print(
    "✓ Only historical Load/Solar channels modified."
)

print(
    "✓ Calendar features preserved exactly."
)


# ============================================================
# 8. EVALUATION STORAGE
# ============================================================

result_rows = []


def append_result(
    seed,
    model_name,
    target_name,
    condition,
    day_back,
    actual,
    predicted
):

    mae, rmse, r2 = calculate_metrics(
        actual,
        predicted
    )

    result_rows.append({

        "Seed": seed,

        "Model":
            model_name,

        "Target":
            target_name,

        "Condition":
            condition,

        "Day_back":
            day_back,

        "MAE":
            mae,

        "RMSE":
            rmse,

        "R2":
            r2
    })


# ============================================================
# 9. EVALUATE SAVED 3-SEED MODELS
# ============================================================

for seed in SEEDS:

    print("\n" + "=" * 100)
    print(f"TEMPORAL OCCLUSION — SEED {seed}")
    print("=" * 100)


    # --------------------------------------------------------
    # A. LSTM
    # --------------------------------------------------------

    tf.keras.backend.clear_session()
    gc.collect()

    model_path = os.path.join(
        ROBUST_DIR,
        f"robust_LSTM_seed_{seed}.keras"
    )

    assert os.path.isfile(
        model_path
    )

    model = load_model_safe(
        model_path
    )

    print("\nLSTM")

    for condition, X_cond in occluded_inputs.items():

        pred = model.predict(
            X_cond,
            batch_size=256,
            verbose=0
        )

        pred_load = inverse_target(
            np.asarray(
                pred[0]
            ).reshape(-1),
            0
        )

        pred_solar = inverse_target(
            np.asarray(
                pred[1]
            ).reshape(-1),
            1
        )

        if condition == "Clean":
            day_back = 0
        else:
            day_back = int(
                condition.split("-")[1]
            )

        append_result(
            seed,
            "LSTM",
            "Load",
            condition,
            day_back,
            actual_load,
            pred_load
        )

        append_result(
            seed,
            "LSTM",
            "Solar",
            condition,
            day_back,
            actual_solar,
            pred_solar
        )

        print(
            f"  {condition:<10} done"
        )

    del model
    gc.collect()


    # --------------------------------------------------------
    # B. SINGLE-TASK LOAD TRANSFORMER
    # --------------------------------------------------------

    tf.keras.backend.clear_session()
    gc.collect()

    model_path = os.path.join(
        ROBUST_DIR,
        f"robust_ST_Load_seed_{seed}.keras"
    )

    assert os.path.isfile(
        model_path
    )

    model = load_model_safe(
        model_path
    )

    print("\nST Transformer — Load")

    for condition, X_cond in occluded_inputs.items():

        pred_scaled = model.predict(
            X_cond,
            batch_size=256,
            verbose=0
        ).reshape(-1)

        pred_load = inverse_target(
            pred_scaled,
            0
        )

        if condition == "Clean":
            day_back = 0
        else:
            day_back = int(
                condition.split("-")[1]
            )

        append_result(
            seed,
            "ST-Transformer",
            "Load",
            condition,
            day_back,
            actual_load,
            pred_load
        )

        print(
            f"  {condition:<10} done"
        )

    del model
    gc.collect()


    # --------------------------------------------------------
    # C. SINGLE-TASK SOLAR TRANSFORMER
    # --------------------------------------------------------

    tf.keras.backend.clear_session()
    gc.collect()

    model_path = os.path.join(
        ROBUST_DIR,
        f"robust_ST_Solar_seed_{seed}.keras"
    )

    assert os.path.isfile(
        model_path
    )

    model = load_model_safe(
        model_path
    )

    print("\nST Transformer — Solar")

    for condition, X_cond in occluded_inputs.items():

        pred_scaled = model.predict(
            X_cond,
            batch_size=256,
            verbose=0
        ).reshape(-1)

        pred_solar = inverse_target(
            pred_scaled,
            1
        )

        if condition == "Clean":
            day_back = 0
        else:
            day_back = int(
                condition.split("-")[1]
            )

        append_result(
            seed,
            "ST-Transformer",
            "Solar",
            condition,
            day_back,
            actual_solar,
            pred_solar
        )

        print(
            f"  {condition:<10} done"
        )

    del model
    gc.collect()


    # --------------------------------------------------------
    # D. WEIGHTED MTL TRANSFORMER
    # --------------------------------------------------------

    tf.keras.backend.clear_session()
    gc.collect()

    model_path = os.path.join(
        ROBUST_DIR,
        f"robust_MTL_2to1_seed_{seed}.keras"
    )

    assert os.path.isfile(
        model_path
    )

    model = load_model_safe(
        model_path
    )

    print("\nWeighted MTL Transformer 2:1")

    for condition, X_cond in occluded_inputs.items():

        pred = model.predict(
            X_cond,
            batch_size=256,
            verbose=0
        )

        pred_load = inverse_target(
            np.asarray(
                pred[0]
            ).reshape(-1),
            0
        )

        pred_solar = inverse_target(
            np.asarray(
                pred[1]
            ).reshape(-1),
            1
        )

        if condition == "Clean":
            day_back = 0
        else:
            day_back = int(
                condition.split("-")[1]
            )

        append_result(
            seed,
            "MTL-Transformer-2:1",
            "Load",
            condition,
            day_back,
            actual_load,
            pred_load
        )

        append_result(
            seed,
            "MTL-Transformer-2:1",
            "Solar",
            condition,
            day_back,
            actual_solar,
            pred_solar
        )

        print(
            f"  {condition:<10} done"
        )

    del model
    gc.collect()


# ============================================================
# 10. RUN-LEVEL RESULTS
#
# Expected:
#
# 3 seeds ×
# (
#   LSTM: 2 targets × 8 conditions = 16
#   ST:   2 targets × 8 conditions = 16
#   MTL:  2 targets × 8 conditions = 16
# )
#
# = 144 rows
# ============================================================

results = pd.DataFrame(
    result_rows
)

assert len(results) == 144

assert np.isfinite(
    results[
        [
            "MAE",
            "RMSE",
            "R2"
        ]
    ].to_numpy()
).all()


RUN_FILE = os.path.join(
    SAVE_DIR,
    "temporal_occlusion_runs.csv"
)

results.to_csv(
    RUN_FILE,
    index=False
)


print(
    f"\nRun-level rows: {len(results)}"
)


# ============================================================
# 11. SUMMARY ACROSS 3 SEEDS
# ============================================================

summary = (
    results
    .groupby(
        [
            "Model",
            "Target",
            "Condition",
            "Day_back"
        ],
        as_index=False
    )
    .agg(
        MAE_mean=(
            "MAE",
            "mean"
        ),

        MAE_std=(
            "MAE",
            "std"
        ),

        RMSE_mean=(
            "RMSE",
            "mean"
        ),

        RMSE_std=(
            "RMSE",
            "std"
        ),

        R2_mean=(
            "R2",
            "mean"
        ),

        R2_std=(
            "R2",
            "std"
        ),

        Seeds=(
            "Seed",
            "nunique"
        )
    )
)

assert len(summary) == 48

assert summary[
    "Seeds"
].eq(3).all()


SUMMARY_FILE = os.path.join(
    SAVE_DIR,
    "temporal_occlusion_summary.csv"
)

summary.to_csv(
    SUMMARY_FILE,
    index=False
)


# ============================================================
# 12. CALCULATE OCCLUSION IMPORTANCE
#
# Importance =
# percentage increase in RMSE relative to the model's
# own clean RMSE.
# ============================================================

clean = (
    summary[
        summary["Condition"]
        == "Clean"
    ][
        [
            "Model",
            "Target",
            "RMSE_mean"
        ]
    ]
    .rename(
        columns={
            "RMSE_mean":
                "Clean_RMSE"
        }
    )
)

assert len(clean) == 6


importance = summary.merge(
    clean,
    on=[
        "Model",
        "Target"
    ],
    how="left",
    validate="many_to_one"
)


importance[
    "RMSE_increase_pct"
] = (
    100.0
    *
    (
        importance["RMSE_mean"]
        -
        importance["Clean_RMSE"]
    )
    /
    importance["Clean_RMSE"]
)


importance[
    "RMSE_absolute_increase"
] = (
    importance["RMSE_mean"]
    -
    importance["Clean_RMSE"]
)


IMPORTANCE_FILE = os.path.join(
    SAVE_DIR,
    "temporal_occlusion_importance.csv"
)

importance.to_csv(
    IMPORTANCE_FILE,
    index=False
)


# ============================================================
# 13. DISPLAY OCCLUSION IMPORTANCE
# ============================================================

display_importance = (
    importance[
        importance["Condition"]
        != "Clean"
    ]
    .sort_values(
        [
            "Target",
            "Model",
            "Day_back"
        ]
    )
)


print("\n" + "=" * 100)
print("TEMPORAL OCCLUSION IMPORTANCE — RMSE CHANGE FROM CLEAN")
print("=" * 100)

print(
    display_importance[
        [
            "Target",
            "Model",
            "Condition",
            "RMSE_mean",
            "RMSE_std",
            "Clean_RMSE",
            "RMSE_absolute_increase",
            "RMSE_increase_pct"
        ]
    ].to_string(
        index=False,
        formatters={

            "RMSE_mean":
                lambda x: f"{x:.3f}",

            "RMSE_std":
                lambda x: f"{x:.3f}",

            "Clean_RMSE":
                lambda x: f"{x:.3f}",

            "RMSE_absolute_increase":
                lambda x: f"{x:+.3f}",

            "RMSE_increase_pct":
                lambda x: f"{x:+.2f}%"
        }
    )
)


# ============================================================
# 14. RANK TEMPORAL BLOCKS BY IMPORTANCE
#
# Rank separately for each Model × Target.
# Highest RMSE increase = rank 1.
# ============================================================

ranked = (
    importance[
        importance["Condition"]
        != "Clean"
    ]
    .copy()
)


ranked[
    "Importance_rank"
] = (
    ranked
    .groupby(
        [
            "Model",
            "Target"
        ]
    )[
        "RMSE_absolute_increase"
    ]
    .rank(
        method="min",
        ascending=False
    )
    .astype(int)
)


ranked = ranked.sort_values(
    [
        "Target",
        "Model",
        "Importance_rank"
    ]
)


RANK_FILE = os.path.join(
    SAVE_DIR,
    "temporal_occlusion_ranking.csv"
)

ranked.to_csv(
    RANK_FILE,
    index=False
)


print("\n" + "=" * 100)
print("TEMPORAL BLOCK IMPORTANCE RANKING")
print("=" * 100)

print(
    ranked[
        [
            "Target",
            "Model",
            "Importance_rank",
            "Condition",
            "RMSE_absolute_increase",
            "RMSE_increase_pct"
        ]
    ].to_string(
        index=False,
        formatters={

            "RMSE_absolute_increase":
                lambda x: f"{x:+.3f}",

            "RMSE_increase_pct":
                lambda x: f"{x:+.2f}%"
        }
    )
)


# ============================================================
# 15. MOST IMPORTANT BLOCK PER MODEL/TARGET
# ============================================================

top_blocks = (
    ranked[
        ranked["Importance_rank"]
        == 1
    ]
    [
        [
            "Model",
            "Target",
            "Condition",
            "Day_back",
            "RMSE_absolute_increase",
            "RMSE_increase_pct"
        ]
    ]
    .sort_values(
        [
            "Target",
            "Model"
        ]
    )
)


TOP_FILE = os.path.join(
    SAVE_DIR,
    "temporal_occlusion_top_blocks.csv"
)

top_blocks.to_csv(
    TOP_FILE,
    index=False
)


print("\n" + "=" * 100)
print("MOST INFLUENTIAL 24-HOUR BLOCK")
print("=" * 100)

print(
    top_blocks.to_string(
        index=False,
        formatters={

            "RMSE_absolute_increase":
                lambda x: f"{x:+.3f}",

            "RMSE_increase_pct":
                lambda x: f"{x:+.2f}%"
        }
    )
)


# ============================================================
# 16. TEMPORAL CONCENTRATION
#
# Compare recent 72h (Days -1,-2,-3) with older
# history (Days -4,-5,-6,-7).
#
# We use the SUM of positive absolute RMSE increases only.
# This is descriptive, not a probability distribution.
# ============================================================

concentration_rows = []

for (
    model_name,
    target_name
), group in ranked.groupby(
    [
        "Model",
        "Target"
    ]
):

    g = group.copy()

    g[
        "Positive_importance"
    ] = np.maximum(
        g[
            "RMSE_absolute_increase"
        ],
        0.0
    )

    total = g[
        "Positive_importance"
    ].sum()

    recent = g[
        g["Day_back"].isin(
            [1, 2, 3]
        )
    ][
        "Positive_importance"
    ].sum()

    older = g[
        g["Day_back"].isin(
            [4, 5, 6, 7]
        )
    ][
        "Positive_importance"
    ].sum()

    if total > 0:

        recent_pct = (
            100.0
            * recent
            / total
        )

        older_pct = (
            100.0
            * older
            / total
        )

    else:

        recent_pct = np.nan
        older_pct = np.nan

    concentration_rows.append({

        "Model":
            model_name,

        "Target":
            target_name,

        "Recent_72h_positive_importance_pct":
            recent_pct,

        "Older_96h_positive_importance_pct":
            older_pct,

        "Total_positive_RMSE_increase":
            total
    })


concentration = pd.DataFrame(
    concentration_rows
)


CONCENTRATION_FILE = os.path.join(
    SAVE_DIR,
    "temporal_occlusion_concentration.csv"
)

concentration.to_csv(
    CONCENTRATION_FILE,
    index=False
)


print("\n" + "=" * 100)
print("TEMPORAL IMPORTANCE CONCENTRATION")
print("=" * 100)

print(
    concentration.to_string(
        index=False,
        formatters={

            "Recent_72h_positive_importance_pct":
                lambda x: f"{x:.2f}%",

            "Older_96h_positive_importance_pct":
                lambda x: f"{x:.2f}%",

            "Total_positive_RMSE_increase":
                lambda x: f"{x:.3f}"
        }
    )
)


# ============================================================
# 17. FINAL INTEGRITY CHECKS
# ============================================================

assert (
    results
    .groupby(
        [
            "Model",
            "Target",
            "Condition"
        ]
    )["Seed"]
    .nunique()
    .eq(3)
    .all()
)


expected_conditions = {
    "Clean",
    "Day_-1",
    "Day_-2",
    "Day_-3",
    "Day_-4",
    "Day_-5",
    "Day_-6",
    "Day_-7"
}

assert set(
    results["Condition"].unique()
) == expected_conditions


expected_files = [

    "temporal_occlusion_blocks.csv",

    "temporal_occlusion_runs.csv",

    "temporal_occlusion_summary.csv",

    "temporal_occlusion_importance.csv",

    "temporal_occlusion_ranking.csv",

    "temporal_occlusion_top_blocks.csv",

    "temporal_occlusion_concentration.csv"
]


print("\n" + "=" * 100)
print("FILE CHECK")
print("=" * 100)

for filename in expected_files:

    path = os.path.join(
        SAVE_DIR,
        filename
    )

    exists = os.path.isfile(
        path
    )

    size = (
        os.path.getsize(path)
        if exists
        else 0
    )

    print(
        f"{filename:<48} "
        f"{'OK' if exists else 'MISSING':<8} "
        f"{size:,} bytes"
    )

    assert exists


# ============================================================
# 18. FREEZE ANALYSIS SPECIFICATION
# ============================================================

spec = {

    "analysis":
        "temporal occlusion sensitivity",

    "interpretation":
        "perturbation-based predictive sensitivity, not causal importance",

    "context_hours":
        168,

    "block_hours":
        24,

    "number_of_blocks":
        7,

    "most_recent_block":
        "Day_-1",

    "oldest_block":
        "Day_-7",

    "occluded_features": [
        "historical Load",
        "historical Solar"
    ],

    "replacement":
        "training mean (0 standardized units)",

    "calendar_features":
        "unchanged",

    "seeds":
        SEEDS,

    "primary_importance_metric":
        "absolute RMSE increase relative to clean",

    "supporting_metric":
        "percentage RMSE increase relative to clean",

    "model_selection":
        "none",

    "training":
        "none"
}


SPEC_FILE = os.path.join(
    SAVE_DIR,
    "temporal_occlusion_specification.json"
)

with open(
    SPEC_FILE,
    "w"
) as f:

    json.dump(
        spec,
        f,
        indent=2
    )


print("\n" + "=" * 100)
print("FINAL CHECKPOINT")
print("=" * 100)

print("\n✓ No training performed.")
print("✓ No model selection performed.")
print("✓ Same seven temporal blocks used for every model.")
print("✓ Every block contains exactly 24 hours.")
print("✓ Only historical Load/Solar channels were occluded.")
print("✓ Calendar variables remained unchanged.")
print("✓ Three saved seeds retained.")
print("✓ Absolute RMSE increase is primary importance measure.")
print("✓ Analysis is predictive sensitivity, NOT causal explanation.")

print("\nSaved:")
for filename in expected_files:
    print(f" ✓ {filename}")

print(" ✓ temporal_occlusion_specification.json")

print("\n" + "=" * 100)
print("STEP 5.1 COMPLETE")
print("=" * 100)


STEP 5.1 — SEVEN-DAY TEMPORAL OCCLUSION EXPLAINABILITY

Frozen test data:
X_test: (6576, 168, 8)
y_test: (6576, 2)

Target scaler:
Mean : [55821.81814021  4184.23949544]
Scale: [10034.61850806  6415.36095093]

TEMPORAL OCCLUSION BLOCKS
 Block  Day_back  Start_index  End_index_exclusive  Hours
Day_-1         1          144                  168     24
Day_-2         2          120                  144     24
Day_-3         3           96                  120     24
Day_-4         4           72                   96     24
Day_-5         5           48                   72     24
Day_-6         6           24                   48     24
Day_-7         7            0                   24     24

✓ Seven 24-hour occlusion conditions created.
✓ Only historical Load/Solar channels modified.
✓ Calendar features preserved exactly.

TEMPORAL OCCLUSION — SEED 1

LSTM
  Clean      done
  Day_-1     done
  Day_-2     done
  Day_-3     done
  Day_-4     done
  Day_-5     done
  Day_-6     done
  Day

Step 5.2 — Channel-Specific Day −1 Occlusion

In [19]:
# ============================================================
# STEP 5.2 — CHANNEL-SPECIFIC DAY -1 OCCLUSION
#
# PURPOSE
# ------------------------------------------------------------
# Determine whether Load and Solar forecasts depend on:
#
#   1. Historical Load only
#   2. Historical Solar only
#   3. Both historical channels
#
# within the MOST RECENT 24 HOURS (Day -1).
#
# WHY DAY -1?
# ------------------------------------------------------------
# Step 5.1 was completed BEFORE this analysis and showed that
# Day -1 is the most influential 24-hour block for every
# Model × Target combination.
#
# Therefore Day -1 is now FROZEN for channel analysis.
#
# NO TRAINING.
# NO MODEL SELECTION.
# NO HYPERPARAMETER TUNING.
#
# Cohort:
#   Seeds = [1, 42, 100]
#
# Conditions:
#   Clean
#   Occlude_Load
#   Occlude_Solar
#   Occlude_Both
#
# Occlusion:
#   standardized value = 0
#   (= training mean)
#
# Calendar features remain unchanged.
#
# Interpretation:
#   Perturbation-based predictive sensitivity.
#   NOT causal importance.
# ============================================================

import os
import gc
import json
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib

from tensorflow import keras
from tensorflow.keras import layers


print("=" * 104)
print("STEP 5.2 — CHANNEL-SPECIFIC DAY -1 OCCLUSION")
print("=" * 104)


# ============================================================
# 1. PATHS / FROZEN CONFIGURATION
# ============================================================

SAVE_DIR = (
    "/content/drive/MyDrive/SolarLoad_MTL_Study"
)

ROBUST_DIR = os.path.join(
    SAVE_DIR,
    "robustness_models"
)

NPZ_FILE = os.path.join(
    SAVE_DIR,
    "frozen_168h_sequences.npz"
)

TARGET_SCALER_FILE = os.path.join(
    SAVE_DIR,
    "target_scaler.joblib"
)

STEP51_FILE = os.path.join(
    SAVE_DIR,
    "temporal_occlusion_top_blocks.csv"
)

SEEDS = [1, 42, 100]

SEQ_LEN = 168

DAY1_START = 144
DAY1_END = 168

assert (
    DAY1_END - DAY1_START
) == 24


# ============================================================
# 2. VERIFY STEP 5.1 EVIDENCE
#
# Day -1 must already be the top block for all six
# Model × Target combinations.
# ============================================================

assert os.path.isfile(
    STEP51_FILE
), (
    "Step 5.1 top-block evidence is missing."
)


top_blocks = pd.read_csv(
    STEP51_FILE
)

print("\n" + "=" * 104)
print("STEP 5.1 DAY -1 FREEZE VERIFICATION")
print("=" * 104)

print(
    top_blocks.to_string(
        index=False
    )
)


assert len(top_blocks) == 6

assert (
    top_blocks["Condition"]
    .eq("Day_-1")
    .all()
)

assert (
    top_blocks["Day_back"]
    .eq(1)
    .all()
)

print(
    "\n✓ Day -1 was independently identified "
    "as the most influential block for all "
    "Model × Target combinations."
)

print(
    "✓ Day -1 is therefore frozen BEFORE "
    "channel-specific analysis."
)


# ============================================================
# 3. LOAD FROZEN TEST DATA
# ============================================================

data = np.load(
    NPZ_FILE
)

X_test = data[
    "X_test"
].astype(
    np.float32
)

y_test = data[
    "y_test"
].astype(
    np.float32
)


assert X_test.shape == (
    6576,
    168,
    8
)

assert y_test.shape == (
    6576,
    2
)


print("\nFrozen test arrays:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)


# ============================================================
# 4. TARGET SCALER
# ============================================================

target_scaler = joblib.load(
    TARGET_SCALER_FILE
)


print("\nTarget scaler:")
print("Mean :", target_scaler.mean_)
print("Scale:", target_scaler.scale_)


def inverse_target(
    values,
    target_index
):

    values = np.asarray(
        values
    ).reshape(-1)

    return (
        values
        * target_scaler.scale_[target_index]
        +
        target_scaler.mean_[target_index]
    )


actual_load = inverse_target(
    y_test[:, 0],
    0
)

actual_solar = inverse_target(
    y_test[:, 1],
    1
)


# ============================================================
# 5. POSITIONAL ENCODING FOR SAVED TRANSFORMERS
# ============================================================

@keras.utils.register_keras_serializable(
    package="SolarLoadStudy"
)
class PositionalEncoding(layers.Layer):

    def __init__(
        self,
        sequence_length,
        d_model,
        **kwargs
    ):

        super().__init__(**kwargs)

        self.sequence_length = (
            sequence_length
        )

        self.d_model = d_model

        position = np.arange(
            sequence_length
        )[:, np.newaxis]

        div_term = np.exp(
            np.arange(
                0,
                d_model,
                2
            )
            *
            (
                -np.log(10000.0)
                / d_model
            )
        )

        pe = np.zeros(
            (
                sequence_length,
                d_model
            ),
            dtype=np.float32
        )

        pe[:, 0::2] = np.sin(
            position * div_term
        )

        pe[:, 1::2] = np.cos(
            position * div_term
        )

        self.pe = tf.constant(
            pe[np.newaxis, ...],
            dtype=tf.float32
        )


    def call(
        self,
        inputs
    ):

        return (
            inputs
            +
            self.pe[
                :,
                :tf.shape(inputs)[1],
                :
            ]
        )


    def get_config(
        self
    ):

        config = (
            super().get_config()
        )

        config.update({

            "sequence_length":
                self.sequence_length,

            "d_model":
                self.d_model
        })

        return config


def load_model_safe(
    path
):

    return keras.models.load_model(
        path,
        custom_objects={
            "PositionalEncoding":
                PositionalEncoding
        },
        compile=False
    )


# ============================================================
# 6. METRICS
# ============================================================

def calculate_metrics(
    actual,
    predicted
):

    actual = np.asarray(
        actual
    ).reshape(-1)

    predicted = np.asarray(
        predicted
    ).reshape(-1)

    error = (
        predicted
        -
        actual
    )

    mae = np.mean(
        np.abs(error)
    )

    rmse = np.sqrt(
        np.mean(
            error ** 2
        )
    )

    ss_res = np.sum(
        error ** 2
    )

    ss_tot = np.sum(
        (
            actual
            -
            np.mean(actual)
        ) ** 2
    )

    r2 = (
        1.0
        -
        ss_res / ss_tot
    )

    return (
        mae,
        rmse,
        r2
    )


# ============================================================
# 7. CREATE FOUR FROZEN INPUT CONDITIONS
# ============================================================

conditions = {}


# ------------------------------------------------------------
# A. CLEAN
# ------------------------------------------------------------

conditions[
    "Clean"
] = X_test


# ------------------------------------------------------------
# B. OCCLUDE LOAD ONLY
# ------------------------------------------------------------

X_load = X_test.copy()

X_load[
    :,
    DAY1_START:DAY1_END,
    0
] = 0.0

conditions[
    "Occlude_Load"
] = X_load


# ------------------------------------------------------------
# C. OCCLUDE SOLAR ONLY
# ------------------------------------------------------------

X_solar = X_test.copy()

X_solar[
    :,
    DAY1_START:DAY1_END,
    1
] = 0.0

conditions[
    "Occlude_Solar"
] = X_solar


# ------------------------------------------------------------
# D. OCCLUDE BOTH
# ------------------------------------------------------------

X_both = X_test.copy()

X_both[
    :,
    DAY1_START:DAY1_END,
    0:2
] = 0.0

conditions[
    "Occlude_Both"
] = X_both


# ============================================================
# 8. CONDITION INTEGRITY AUDIT
# ============================================================

print("\n" + "=" * 104)
print("CHANNEL OCCLUSION INTEGRITY AUDIT")
print("=" * 104)


# Clean must be unchanged.
assert np.array_equal(
    conditions["Clean"],
    X_test
)


# Calendar variables must NEVER change.
for name, X_cond in (
    conditions.items()
):

    assert np.array_equal(
        X_cond[:, :, 2:],
        X_test[:, :, 2:]
    )


# Load-only:
assert np.all(
    conditions[
        "Occlude_Load"
    ][
        :,
        DAY1_START:DAY1_END,
        0
    ]
    == 0.0
)

assert np.array_equal(
    conditions[
        "Occlude_Load"
    ][
        :,
        DAY1_START:DAY1_END,
        1
    ],
    X_test[
        :,
        DAY1_START:DAY1_END,
        1
    ]
)


# Solar-only:
assert np.all(
    conditions[
        "Occlude_Solar"
    ][
        :,
        DAY1_START:DAY1_END,
        1
    ]
    == 0.0
)

assert np.array_equal(
    conditions[
        "Occlude_Solar"
    ][
        :,
        DAY1_START:DAY1_END,
        0
    ],
    X_test[
        :,
        DAY1_START:DAY1_END,
        0
    ]
)


# Both:
assert np.all(
    conditions[
        "Occlude_Both"
    ][
        :,
        DAY1_START:DAY1_END,
        0:2
    ]
    == 0.0
)


# Outside Day -1, dynamic features must be untouched.
for name in [
    "Occlude_Load",
    "Occlude_Solar",
    "Occlude_Both"
]:

    assert np.array_equal(
        conditions[name][
            :,
            :DAY1_START,
            0:2
        ],
        X_test[
            :,
            :DAY1_START,
            0:2
        ]
    )


print(
    "✓ Clean input unchanged."
)

print(
    "✓ Load-only condition modifies only "
    "historical Load in Day -1."
)

print(
    "✓ Solar-only condition modifies only "
    "historical Solar in Day -1."
)

print(
    "✓ Both condition modifies both dynamic "
    "channels in Day -1."
)

print(
    "✓ Calendar features unchanged."
)

print(
    "✓ Dynamic history before Day -1 unchanged."
)


# ============================================================
# 9. STORAGE
# ============================================================

result_rows = []


def append_result(
    seed,
    model_name,
    target_name,
    condition_name,
    actual,
    predicted
):

    mae, rmse, r2 = (
        calculate_metrics(
            actual,
            predicted
        )
    )

    result_rows.append({

        "Seed":
            seed,

        "Model":
            model_name,

        "Target":
            target_name,

        "Condition":
            condition_name,

        "MAE":
            mae,

        "RMSE":
            rmse,

        "R2":
            r2
    })


# ============================================================
# 10. EVALUATE ALL SAVED MODELS
# ============================================================

for seed in SEEDS:

    print("\n" + "=" * 104)
    print(
        f"CHANNEL OCCLUSION — SEED {seed}"
    )
    print("=" * 104)


    # --------------------------------------------------------
    # A. LSTM
    # --------------------------------------------------------

    tf.keras.backend.clear_session()
    gc.collect()

    path = os.path.join(
        ROBUST_DIR,
        f"robust_LSTM_seed_{seed}.keras"
    )

    assert os.path.isfile(
        path
    )

    model = load_model_safe(
        path
    )

    print("\nLSTM")

    for (
        condition_name,
        X_cond
    ) in conditions.items():

        pred = model.predict(
            X_cond,
            batch_size=256,
            verbose=0
        )

        pred_load = inverse_target(
            np.asarray(
                pred[0]
            ).reshape(-1),
            0
        )

        pred_solar = inverse_target(
            np.asarray(
                pred[1]
            ).reshape(-1),
            1
        )

        append_result(
            seed,
            "LSTM",
            "Load",
            condition_name,
            actual_load,
            pred_load
        )

        append_result(
            seed,
            "LSTM",
            "Solar",
            condition_name,
            actual_solar,
            pred_solar
        )

        print(
            f"  {condition_name:<16} done"
        )

    del model
    gc.collect()


    # --------------------------------------------------------
    # B. ST LOAD
    # --------------------------------------------------------

    tf.keras.backend.clear_session()
    gc.collect()

    path = os.path.join(
        ROBUST_DIR,
        f"robust_ST_Load_seed_{seed}.keras"
    )

    assert os.path.isfile(
        path
    )

    model = load_model_safe(
        path
    )

    print("\nST Transformer — Load")

    for (
        condition_name,
        X_cond
    ) in conditions.items():

        pred_scaled = (
            model.predict(
                X_cond,
                batch_size=256,
                verbose=0
            )
            .reshape(-1)
        )

        pred_load = inverse_target(
            pred_scaled,
            0
        )

        append_result(
            seed,
            "ST-Transformer",
            "Load",
            condition_name,
            actual_load,
            pred_load
        )

        print(
            f"  {condition_name:<16} done"
        )

    del model
    gc.collect()


    # --------------------------------------------------------
    # C. ST SOLAR
    # --------------------------------------------------------

    tf.keras.backend.clear_session()
    gc.collect()

    path = os.path.join(
        ROBUST_DIR,
        f"robust_ST_Solar_seed_{seed}.keras"
    )

    assert os.path.isfile(
        path
    )

    model = load_model_safe(
        path
    )

    print("\nST Transformer — Solar")

    for (
        condition_name,
        X_cond
    ) in conditions.items():

        pred_scaled = (
            model.predict(
                X_cond,
                batch_size=256,
                verbose=0
            )
            .reshape(-1)
        )

        pred_solar = inverse_target(
            pred_scaled,
            1
        )

        append_result(
            seed,
            "ST-Transformer",
            "Solar",
            condition_name,
            actual_solar,
            pred_solar
        )

        print(
            f"  {condition_name:<16} done"
        )

    del model
    gc.collect()


    # --------------------------------------------------------
    # D. WEIGHTED MTL TRANSFORMER
    # --------------------------------------------------------

    tf.keras.backend.clear_session()
    gc.collect()

    path = os.path.join(
        ROBUST_DIR,
        f"robust_MTL_2to1_seed_{seed}.keras"
    )

    assert os.path.isfile(
        path
    )

    model = load_model_safe(
        path
    )

    print("\nWeighted MTL Transformer 2:1")

    for (
        condition_name,
        X_cond
    ) in conditions.items():

        pred = model.predict(
            X_cond,
            batch_size=256,
            verbose=0
        )

        pred_load = inverse_target(
            np.asarray(
                pred[0]
            ).reshape(-1),
            0
        )

        pred_solar = inverse_target(
            np.asarray(
                pred[1]
            ).reshape(-1),
            1
        )

        append_result(
            seed,
            "MTL-Transformer-2:1",
            "Load",
            condition_name,
            actual_load,
            pred_load
        )

        append_result(
            seed,
            "MTL-Transformer-2:1",
            "Solar",
            condition_name,
            actual_solar,
            pred_solar
        )

        print(
            f"  {condition_name:<16} done"
        )

    del model
    gc.collect()


# ============================================================
# 11. RUN-LEVEL RESULTS
#
# Expected:
#
# 3 seeds ×
# 3 conceptual models ×
# 2 targets ×
# 4 conditions
#
# = 72 rows
# ============================================================

results = pd.DataFrame(
    result_rows
)

assert len(results) == 72

assert np.isfinite(
    results[
        [
            "MAE",
            "RMSE",
            "R2"
        ]
    ].to_numpy()
).all()


RUN_FILE = os.path.join(
    SAVE_DIR,
    "channel_occlusion_runs.csv"
)

results.to_csv(
    RUN_FILE,
    index=False
)

print(
    f"\nRun-level rows: {len(results)}"
)


# ============================================================
# 12. SUMMARY ACROSS THREE SEEDS
# ============================================================

summary = (
    results
    .groupby(
        [
            "Model",
            "Target",
            "Condition"
        ],
        as_index=False
    )
    .agg(

        MAE_mean=(
            "MAE",
            "mean"
        ),

        MAE_std=(
            "MAE",
            "std"
        ),

        RMSE_mean=(
            "RMSE",
            "mean"
        ),

        RMSE_std=(
            "RMSE",
            "std"
        ),

        R2_mean=(
            "R2",
            "mean"
        ),

        R2_std=(
            "R2",
            "std"
        ),

        Seeds=(
            "Seed",
            "nunique"
        )
    )
)


assert len(summary) == 24

assert summary[
    "Seeds"
].eq(3).all()


SUMMARY_FILE = os.path.join(
    SAVE_DIR,
    "channel_occlusion_summary.csv"
)

summary.to_csv(
    SUMMARY_FILE,
    index=False
)


# ============================================================
# 13. CLEAN-RELATIVE SENSITIVITY
# ============================================================

clean = (
    summary[
        summary["Condition"]
        == "Clean"
    ][
        [
            "Model",
            "Target",
            "RMSE_mean"
        ]
    ]
    .rename(
        columns={
            "RMSE_mean":
                "Clean_RMSE"
        }
    )
)


assert len(clean) == 6


sensitivity = summary.merge(
    clean,
    on=[
        "Model",
        "Target"
    ],
    how="left",
    validate="many_to_one"
)


sensitivity[
    "RMSE_absolute_increase"
] = (
    sensitivity[
        "RMSE_mean"
    ]
    -
    sensitivity[
        "Clean_RMSE"
    ]
)


sensitivity[
    "RMSE_increase_pct"
] = (
    100.0
    *
    sensitivity[
        "RMSE_absolute_increase"
    ]
    /
    sensitivity[
        "Clean_RMSE"
    ]
)


SENSITIVITY_FILE = os.path.join(
    SAVE_DIR,
    "channel_occlusion_sensitivity.csv"
)

sensitivity.to_csv(
    SENSITIVITY_FILE,
    index=False
)


print("\n" + "=" * 104)
print("CHANNEL OCCLUSION SENSITIVITY — RMSE CHANGE FROM CLEAN")
print("=" * 104)


display_sensitivity = (
    sensitivity[
        sensitivity["Condition"]
        != "Clean"
    ]
    .copy()
)


condition_order = {
    "Occlude_Load": 1,
    "Occlude_Solar": 2,
    "Occlude_Both": 3
}

display_sensitivity[
    "_order"
] = (
    display_sensitivity[
        "Condition"
    ]
    .map(
        condition_order
    )
)


display_sensitivity = (
    display_sensitivity
    .sort_values(
        [
            "Target",
            "Model",
            "_order"
        ]
    )
)


print(
    display_sensitivity[
        [
            "Target",
            "Model",
            "Condition",
            "RMSE_mean",
            "RMSE_std",
            "Clean_RMSE",
            "RMSE_absolute_increase",
            "RMSE_increase_pct"
        ]
    ].to_string(
        index=False,
        formatters={

            "RMSE_mean":
                lambda x: f"{x:.3f}",

            "RMSE_std":
                lambda x: f"{x:.3f}",

            "Clean_RMSE":
                lambda x: f"{x:.3f}",

            "RMSE_absolute_increase":
                lambda x: f"{x:+.3f}",

            "RMSE_increase_pct":
                lambda x: f"{x:+.2f}%"
        }
    )
)


# ============================================================
# 14. OWN-CHANNEL VS CROSS-CHANNEL SENSITIVITY
#
# For LOAD prediction:
#   Own-channel   = Occlude_Load
#   Cross-channel = Occlude_Solar
#
# For SOLAR prediction:
#   Own-channel   = Occlude_Solar
#   Cross-channel = Occlude_Load
# ============================================================

channel_rows = []


for model_name in [
    "LSTM",
    "ST-Transformer",
    "MTL-Transformer-2:1"
]:

    for target_name in [
        "Load",
        "Solar"
    ]:

        subset = sensitivity[
            (
                sensitivity["Model"]
                == model_name
            )
            &
            (
                sensitivity["Target"]
                == target_name
            )
        ]


        if target_name == "Load":

            own_condition = (
                "Occlude_Load"
            )

            cross_condition = (
                "Occlude_Solar"
            )

        else:

            own_condition = (
                "Occlude_Solar"
            )

            cross_condition = (
                "Occlude_Load"
            )


        own = subset[
            subset["Condition"]
            == own_condition
        ].iloc[0]


        cross = subset[
            subset["Condition"]
            == cross_condition
        ].iloc[0]


        both = subset[
            subset["Condition"]
            == "Occlude_Both"
        ].iloc[0]


        own_abs = float(
            own[
                "RMSE_absolute_increase"
            ]
        )

        cross_abs = float(
            cross[
                "RMSE_absolute_increase"
            ]
        )

        both_abs = float(
            both[
                "RMSE_absolute_increase"
            ]
        )


        # Positive-only sensitivity shares are descriptive.
        own_positive = max(
            own_abs,
            0.0
        )

        cross_positive = max(
            cross_abs,
            0.0
        )

        positive_total = (
            own_positive
            +
            cross_positive
        )


        if positive_total > 0:

            own_share = (
                100.0
                *
                own_positive
                /
                positive_total
            )

            cross_share = (
                100.0
                *
                cross_positive
                /
                positive_total
            )

        else:

            own_share = np.nan
            cross_share = np.nan


        channel_rows.append({

            "Model":
                model_name,

            "Target":
                target_name,

            "Own_channel_condition":
                own_condition,

            "Cross_channel_condition":
                cross_condition,

            "Own_channel_RMSE_increase":
                own_abs,

            "Cross_channel_RMSE_increase":
                cross_abs,

            "Both_channels_RMSE_increase":
                both_abs,

            "Own_channel_RMSE_increase_pct":
                float(
                    own[
                        "RMSE_increase_pct"
                    ]
                ),

            "Cross_channel_RMSE_increase_pct":
                float(
                    cross[
                        "RMSE_increase_pct"
                    ]
                ),

            "Both_channels_RMSE_increase_pct":
                float(
                    both[
                        "RMSE_increase_pct"
                    ]
                ),

            "Own_positive_sensitivity_share_pct":
                own_share,

            "Cross_positive_sensitivity_share_pct":
                cross_share
        })


channel_comparison = pd.DataFrame(
    channel_rows
)


CHANNEL_FILE = os.path.join(
    SAVE_DIR,
    "channel_occlusion_own_vs_cross.csv"
)

channel_comparison.to_csv(
    CHANNEL_FILE,
    index=False
)


print("\n" + "=" * 104)
print("OWN-CHANNEL VS CROSS-CHANNEL SENSITIVITY")
print("=" * 104)


print(
    channel_comparison.to_string(
        index=False,
        formatters={

            "Own_channel_RMSE_increase":
                lambda x: f"{x:+.3f}",

            "Cross_channel_RMSE_increase":
                lambda x: f"{x:+.3f}",

            "Both_channels_RMSE_increase":
                lambda x: f"{x:+.3f}",

            "Own_channel_RMSE_increase_pct":
                lambda x: f"{x:+.2f}%",

            "Cross_channel_RMSE_increase_pct":
                lambda x: f"{x:+.2f}%",

            "Both_channels_RMSE_increase_pct":
                lambda x: f"{x:+.2f}%",

            "Own_positive_sensitivity_share_pct":
                lambda x: (
                    "NA"
                    if pd.isna(x)
                    else f"{x:.2f}%"
                ),

            "Cross_positive_sensitivity_share_pct":
                lambda x: (
                    "NA"
                    if pd.isna(x)
                    else f"{x:.2f}%"
                )
        }
    )
)


# ============================================================
# 15. CROSS-CHANNEL MODEL COMPARISON
#
# This is especially important for MTL interpretation.
#
# Load forecast:
#   sensitivity to Solar history.
#
# Solar forecast:
#   sensitivity to Load history.
# ============================================================

cross_only = (
    channel_comparison[
        [
            "Model",
            "Target",
            "Cross_channel_condition",
            "Cross_channel_RMSE_increase",
            "Cross_channel_RMSE_increase_pct",
            "Cross_positive_sensitivity_share_pct"
        ]
    ]
    .sort_values(
        [
            "Target",
            "Model"
        ]
    )
)


CROSS_FILE = os.path.join(
    SAVE_DIR,
    "channel_occlusion_cross_channel.csv"
)

cross_only.to_csv(
    CROSS_FILE,
    index=False
)


print("\n" + "=" * 104)
print("CROSS-CHANNEL SENSITIVITY")
print("=" * 104)


print(
    cross_only.to_string(
        index=False,
        formatters={

            "Cross_channel_RMSE_increase":
                lambda x: f"{x:+.3f}",

            "Cross_channel_RMSE_increase_pct":
                lambda x: f"{x:+.2f}%",

            "Cross_positive_sensitivity_share_pct":
                lambda x: (
                    "NA"
                    if pd.isna(x)
                    else f"{x:.2f}%"
                )
        }
    )
)


# ============================================================
# 16. INTERACTION / NON-ADDITIVITY
#
# Both-channel effect does NOT have to equal:
#
# Load-only effect + Solar-only effect.
#
# Interaction residual:
#
#   Both increase
#   - (Load-only increase + Solar-only increase)
#
# This is descriptive perturbation non-additivity.
# It is NOT causal interaction.
# ============================================================

interaction_rows = []


for (
    model_name,
    target_name
), group in sensitivity.groupby(
    [
        "Model",
        "Target"
    ]
):

    load_only = float(
        group[
            group["Condition"]
            == "Occlude_Load"
        ][
            "RMSE_absolute_increase"
        ].iloc[0]
    )

    solar_only = float(
        group[
            group["Condition"]
            == "Occlude_Solar"
        ][
            "RMSE_absolute_increase"
        ].iloc[0]
    )

    both = float(
        group[
            group["Condition"]
            == "Occlude_Both"
        ][
            "RMSE_absolute_increase"
        ].iloc[0]
    )

    additive_expectation = (
        load_only
        +
        solar_only
    )

    residual = (
        both
        -
        additive_expectation
    )


    interaction_rows.append({

        "Model":
            model_name,

        "Target":
            target_name,

        "Load_only_RMSE_increase":
            load_only,

        "Solar_only_RMSE_increase":
            solar_only,

        "Additive_expectation":
            additive_expectation,

        "Both_RMSE_increase":
            both,

        "Nonadditivity_residual":
            residual
    })


interaction = pd.DataFrame(
    interaction_rows
)


INTERACTION_FILE = os.path.join(
    SAVE_DIR,
    "channel_occlusion_nonadditivity.csv"
)

interaction.to_csv(
    INTERACTION_FILE,
    index=False
)


print("\n" + "=" * 104)
print("CHANNEL OCCLUSION NON-ADDITIVITY")
print("=" * 104)


print(
    interaction.to_string(
        index=False,
        formatters={

            "Load_only_RMSE_increase":
                lambda x: f"{x:+.3f}",

            "Solar_only_RMSE_increase":
                lambda x: f"{x:+.3f}",

            "Additive_expectation":
                lambda x: f"{x:+.3f}",

            "Both_RMSE_increase":
                lambda x: f"{x:+.3f}",

            "Nonadditivity_residual":
                lambda x: f"{x:+.3f}"
        }
    )
)


# ============================================================
# 17. VERIFY BOTH-CHANNEL CONDITION AGAINST STEP 5.1
#
# This is a powerful reproducibility check:
#
# Step 5.2 Occlude_Both must reproduce Step 5.1 Day_-1
# because the perturbation is identical.
# ============================================================

STEP51_SUMMARY_FILE = os.path.join(
    SAVE_DIR,
    "temporal_occlusion_summary.csv"
)

assert os.path.isfile(
    STEP51_SUMMARY_FILE
)


step51 = pd.read_csv(
    STEP51_SUMMARY_FILE
)


step51_day1 = (
    step51[
        step51["Condition"]
        == "Day_-1"
    ][
        [
            "Model",
            "Target",
            "RMSE_mean"
        ]
    ]
    .rename(
        columns={
            "RMSE_mean":
                "Step51_Day1_RMSE"
        }
    )
)


step52_both = (
    summary[
        summary["Condition"]
        == "Occlude_Both"
    ][
        [
            "Model",
            "Target",
            "RMSE_mean"
        ]
    ]
    .rename(
        columns={
            "RMSE_mean":
                "Step52_Both_RMSE"
        }
    )
)


repro_check = (
    step51_day1.merge(
        step52_both,
        on=[
            "Model",
            "Target"
        ],
        validate="one_to_one"
    )
)


repro_check[
    "Absolute_difference"
] = np.abs(
    repro_check[
        "Step51_Day1_RMSE"
    ]
    -
    repro_check[
        "Step52_Both_RMSE"
    ]
)


print("\n" + "=" * 104)
print("STEP 5.1 ↔ STEP 5.2 REPRODUCIBILITY CHECK")
print("=" * 104)


print(
    repro_check.to_string(
        index=False,
        formatters={

            "Step51_Day1_RMSE":
                lambda x: f"{x:.6f}",

            "Step52_Both_RMSE":
                lambda x: f"{x:.6f}",

            "Absolute_difference":
                lambda x: f"{x:.10f}"
        }
    )
)


assert np.allclose(
    repro_check[
        "Step51_Day1_RMSE"
    ],
    repro_check[
        "Step52_Both_RMSE"
    ],
    rtol=1e-6,
    atol=1e-4
)


print(
    "\n✓ Step 5.2 Both-channel occlusion "
    "reproduces Step 5.1 Day -1."
)


REPRO_FILE = os.path.join(
    SAVE_DIR,
    "channel_occlusion_step51_reproducibility.csv"
)

repro_check.to_csv(
    REPRO_FILE,
    index=False
)


# ============================================================
# 18. SAVE FROZEN SPECIFICATION
# ============================================================

specification = {

    "analysis":
        "channel-specific temporal occlusion",

    "window":
        "Day -1",

    "window_hours":
        24,

    "indices":
        [
            DAY1_START,
            DAY1_END
        ],

    "selection_basis":
        (
            "Step 5.1 identified Day -1 as "
            "the most influential 24-hour block "
            "for every Model × Target combination"
        ),

    "conditions": [
        "Clean",
        "Occlude_Load",
        "Occlude_Solar",
        "Occlude_Both"
    ],

    "load_feature_index":
        0,

    "solar_feature_index":
        1,

    "replacement":
        "training mean = 0 standardized units",

    "calendar_features":
        "unchanged",

    "seeds":
        SEEDS,

    "training":
        "none",

    "model_selection":
        "none",

    "primary_measure":
        "absolute RMSE increase relative to clean",

    "supporting_measure":
        "percentage RMSE increase relative to clean",

    "cross_channel_definition": {

        "Load_target":
            "sensitivity to historical Solar",

        "Solar_target":
            "sensitivity to historical Load"
    },

    "interpretation":
        (
            "perturbation-based predictive sensitivity; "
            "not causal feature importance"
        )
}


SPEC_FILE = os.path.join(
    SAVE_DIR,
    "channel_occlusion_specification.json"
)

with open(
    SPEC_FILE,
    "w"
) as f:

    json.dump(
        specification,
        f,
        indent=2
    )


# ============================================================
# 19. FINAL FILE CHECK
# ============================================================

expected_files = [

    "channel_occlusion_runs.csv",

    "channel_occlusion_summary.csv",

    "channel_occlusion_sensitivity.csv",

    "channel_occlusion_own_vs_cross.csv",

    "channel_occlusion_cross_channel.csv",

    "channel_occlusion_nonadditivity.csv",

    "channel_occlusion_step51_reproducibility.csv",

    "channel_occlusion_specification.json"
]


print("\n" + "=" * 104)
print("FILE CHECK")
print("=" * 104)


for filename in expected_files:

    path = os.path.join(
        SAVE_DIR,
        filename
    )

    exists = os.path.isfile(
        path
    )

    size = (
        os.path.getsize(path)
        if exists
        else 0
    )

    print(
        f"{filename:<55} "
        f"{'OK' if exists else 'MISSING':<8} "
        f"{size:,} bytes"
    )

    assert exists


# ============================================================
# 20. FINAL INTEGRITY
# ============================================================

assert (
    results.groupby(
        [
            "Model",
            "Target",
            "Condition"
        ]
    )[
        "Seed"
    ]
    .nunique()
    .eq(3)
    .all()
)


assert set(
    results[
        "Condition"
    ].unique()
) == {
    "Clean",
    "Occlude_Load",
    "Occlude_Solar",
    "Occlude_Both"
}


print("\n" + "=" * 104)
print("FINAL CHECKPOINT")
print("=" * 104)

print("\n✓ No training performed.")
print("✓ No model selection performed.")
print("✓ Day -1 selection was frozen from Step 5.1.")
print("✓ Same channel perturbations used for every model.")
print("✓ Only Day -1 dynamic channels were modified.")
print("✓ Calendar variables remained unchanged.")
print("✓ Three saved seeds retained.")
print(
    "✓ Own-channel and cross-channel "
    "sensitivity quantified."
)
print(
    "✓ Both-channel result reproduced "
    "Step 5.1 Day -1."
)
print(
    "✓ Interpretation restricted to "
    "predictive sensitivity, not causality."
)

print("\nSaved:")

for filename in expected_files:
    print(f" ✓ {filename}")

print("\n" + "=" * 104)
print("STEP 5.2 COMPLETE")
print("=" * 104)

STEP 5.2 — CHANNEL-SPECIFIC DAY -1 OCCLUSION

STEP 5.1 DAY -1 FREEZE VERIFICATION
              Model Target Condition  Day_back  RMSE_absolute_increase  RMSE_increase_pct
               LSTM   Load    Day_-1         1             9291.386641        1167.829452
MTL-Transformer-2:1   Load    Day_-1         1             8865.833652         960.680930
     ST-Transformer   Load    Day_-1         1             9234.751048        1193.982172
               LSTM  Solar    Day_-1         1             7953.327596        1641.801252
MTL-Transformer-2:1  Solar    Day_-1         1             7572.105863        1079.072067
     ST-Transformer  Solar    Day_-1         1             7272.943901        1250.645050

✓ Day -1 was independently identified as the most influential block for all Model × Target combinations.
✓ Day -1 is therefore frozen BEFORE channel-specific analysis.

Frozen test arrays:
X_test: (6576, 168, 8)
y_test: (6576, 2)

Target scaler:
Mean : [55821.81814021  4184.23949544]
Sc

Final Colab step — Archive everything before shutdown

In [20]:
# ============================================================
# FINAL ARCHIVAL AUDIT — BEFORE CLOSING COLAB
# No training / no prediction / no model modification
# ============================================================

from pathlib import Path
import os
import json
import shutil
from datetime import datetime

PROJECT_DIR = Path("/content/drive/MyDrive/SolarLoad_MTL_Study")
ARCHIVE_DIR = PROJECT_DIR / "FINAL_FROZEN_ARCHIVE"
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 90)
print("FINAL FROZEN ARCHIVAL AUDIT")
print("=" * 90)
print("Project:", PROJECT_DIR)
print("Archive:", ARCHIVE_DIR)

# ------------------------------------------------------------
# 1. Inventory every persistent project artifact
# ------------------------------------------------------------
files = []

for p in PROJECT_DIR.rglob("*"):
    if p.is_file() and ARCHIVE_DIR not in p.parents:
        try:
            size = p.stat().st_size
        except:
            size = None

        files.append({
            "relative_path": str(p.relative_to(PROJECT_DIR)),
            "size_bytes": size,
            "suffix": p.suffix.lower()
        })

files = sorted(files, key=lambda x: x["relative_path"])

print(f"\nPersistent files found: {len(files)}")

# ------------------------------------------------------------
# 2. Save complete inventory
# ------------------------------------------------------------
inventory_path = ARCHIVE_DIR / "FINAL_PROJECT_FILE_INVENTORY.json"

with open(inventory_path, "w") as f:
    json.dump(
        {
            "created": datetime.now().isoformat(),
            "project_dir": str(PROJECT_DIR),
            "n_files": len(files),
            "files": files
        },
        f,
        indent=2
    )

# ------------------------------------------------------------
# 3. Categorize files relevant to final paper
# ------------------------------------------------------------
keywords = [
    "context",
    "weight",
    "validation",
    "test",
    "prediction",
    "metric",
    "stat",
    "shift",
    "distribution",
    "monthly",
    "degradation",
    "robust",
    "occlusion",
    "channel",
    "temporal",
    "efficiency",
    "seed",
    "manifest",
    "freeze",
    "config",
    "sequence"
]

paper_files = []

for item in files:
    name = item["relative_path"].lower()
    if any(k in name for k in keywords):
        paper_files.append(item)

paper_inventory_path = ARCHIVE_DIR / "FINAL_PAPER_ARTIFACT_INVENTORY.json"

with open(paper_inventory_path, "w") as f:
    json.dump(
        {
            "created": datetime.now().isoformat(),
            "n_candidate_paper_files": len(paper_files),
            "files": paper_files
        },
        f,
        indent=2
    )

# ------------------------------------------------------------
# 4. Explicitly verify the frozen sequence archive
# ------------------------------------------------------------
sequence_file = PROJECT_DIR / "frozen_168h_sequences.npz"

print("\n" + "-" * 90)
print("CRITICAL ARTIFACT CHECK")
print("-" * 90)

print(
    f"{'✓' if sequence_file.exists() else '✗'} "
    f"frozen_168h_sequences.npz"
)

# ------------------------------------------------------------
# 5. Look for the major evidence families
# ------------------------------------------------------------
families = {
    "final_test": ["test", "prediction"],
    "task_weighting": ["weight"],
    "statistics": ["stat"],
    "temporal_shift": ["shift", "distribution"],
    "degradation": ["degradation"],
    "temporal_occlusion": ["temporal", "occlusion"],
    "channel_occlusion": ["channel", "occlusion"],
    "freeze_metadata": ["freeze"],
}

all_names = [x["relative_path"].lower() for x in files]

family_status = {}

for family, terms in families.items():
    matches = [
        files[i]["relative_path"]
        for i, name in enumerate(all_names)
        if all(term in name for term in terms)
    ]

    family_status[family] = matches

    print(f"\n{family}: {len(matches)} candidate file(s)")
    for m in matches[:10]:
        print("   ", m)

# ------------------------------------------------------------
# 6. Save closure manifest
# ------------------------------------------------------------
closure_manifest = {
    "timestamp": datetime.now().isoformat(),
    "implementation_status": "FROZEN",
    "training_required_after_closure": False,
    "prediction_required_after_closure": False,
    "frozen_context_hours": 168,
    "forecast_horizon_hours": 1,
    "final_mtl_loss_weights": {
        "load": 2.0,
        "solar": 1.0
    },
    "primary_evaluation_seeds": [1, 7, 21, 42, 100],
    "robustness_explainability_seeds": [1, 42, 100],
    "feature_formula_doy": "(dayofyear - 1) / 365.25",
    "sequence_archive_exists": sequence_file.exists(),
    "evidence_families": family_status,
    "notes": [
        "No further model training is planned.",
        "Final test set has been exposed and is frozen.",
        "Subsequent work is limited to tables, figures, manuscript writing, and analysis of frozen outputs.",
        "Temporal and channel occlusion are predictive-sensitivity analyses, not causal explanations.",
        "Controlled degradation is distinct from natural temporal distribution change."
    ]
}

closure_path = ARCHIVE_DIR / "FINAL_EXPERIMENT_CLOSURE_MANIFEST.json"

with open(closure_path, "w") as f:
    json.dump(closure_manifest, f, indent=2)

print("\n" + "=" * 90)
print("ARCHIVAL AUDIT COMPLETE")
print("=" * 90)
print("Inventory :", inventory_path)
print("Paper files:", paper_inventory_path)
print("Closure    :", closure_path)
print("\nNO TRAINING OR PREDICTION WAS PERFORMED.")

FINAL FROZEN ARCHIVAL AUDIT
Project: /content/drive/MyDrive/SolarLoad_MTL_Study
Archive: /content/drive/MyDrive/SolarLoad_MTL_Study/FINAL_FROZEN_ARCHIVE

Persistent files found: 79

------------------------------------------------------------------------------------------
CRITICAL ARTIFACT CHECK
------------------------------------------------------------------------------------------
✓ frozen_168h_sequences.npz

final_test: 1 candidate file(s)
    final_test_predictions.csv

task_weighting: 5 candidate file(s)
    mtl_loss_weight_all_runs.csv
    mtl_loss_weight_balanced_runs.csv
    mtl_loss_weight_balanced_summary.csv
    mtl_loss_weight_summary.csv
    weighted_mtl_validation_runs.csv

statistics: 2 candidate file(s)
    final_statistical_comparisons.csv
    final_statistical_seed_comparisons.csv

temporal_shift: 2 candidate file(s)
    temporal_distribution_shift_audit.csv
    temporal_distribution_shift_monthly.csv

degradation: 4 candidate file(s)
    robustness_degradation_pred